<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Notebook 07 — Unified Schema and Lineage</h2>

<p>
هدف این Notebook تبدیل خروجی‌های پذیرفته‌شده سه خانواده منبع
<code>Website</code>،
<code>Book</code> و
<code>Transcript</code>
به یک مدل سند مشترک و قابل‌ردیابی است.
</p>

<p>
سه منبع ساختار متفاوتی دارند؛ اما لایه‌های بعدی RAG باید بتوانند همه آن‌ها را با یک قرارداد داده مشترک مصرف کنند، بدون اینکه تفاوت منبع، نویسنده، گوینده، نقش سردبیری، محل استناد یا منشأ محتوا از بین برود.
</p>

<h3>ورودی‌های اصلی</h3>

<ul>
<li>Artifact پذیرفته‌شده Website از Notebook 02</li>
<li>Artifact پذیرفته‌شده Book از Notebook 04</li>
<li>Artifact پذیرفته‌شده Transcript از Notebook 06</li>
</ul>

<h3>خروجی‌های مورد انتظار</h3>

<ul>
<li>Schema مشترک و نسخه‌دار برای Document</li>
<li>نگاشت مستقل هر سه Source به Common Document</li>
<li><code>document_id</code> پایدار و deterministic</li>
<li>Lineage قابل ردیابی از Document تا Source و Raw content</li>
<li>حفظ Attribution و تفکیک نویسنده، سردبیر و گوینده</li>
<li>Artifact یکپارچه <code>documents_unified_v1.jsonl</code></li>
<li>گزارش مستقل پوشش Lineage و Attribution</li>
</ul>

<h3>خارج از Scope</h3>

<p>
در این Notebook هیچ Persian normalization، Deduplication، Chunking، Embedding یا Indexing انجام نمی‌شود.
این مراحل در Notebookهای بعدی به‌صورت مستقل اجرا خواهند شد.
</p>

<h3>شرط نهایی PASS</h3>

<p>
Notebook فقط زمانی کامل است که هر Document معتبر به Source اصلی خود قابل ردیابی باشد،
Stable IDها معتبر باشند،
Lineage شکسته وجود نداشته باشد،
Attribution نامعتبر وجود نداشته باشد،
Artifact نهایی از روی Disk مستقلاً دوباره اعتبارسنجی شود
و اجرای
<code>Restart Kernel → Run All</code>
بدون Error پایان یابد.
</p>

</div>

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Start Gate — کشف Repository و ورودی‌های واقعی Notebook 07</h3>

<p>
قبل از تعریف Schema یا تبدیل داده، ابتدا وضعیت واقعی Repository و Artifactهای ورودی بررسی می‌شود.
این مرحله فقط Discovery انجام می‌دهد و هیچ فایل یا داده‌ای را تغییر نمی‌دهد.
</p>

<p>
هدف این است که بدون تکیه بر Variableهای Notebookهای قبلی و بدون حدس‌زدن Path،
مشخص شود سه Artifact اصلی Website، Book و Transcript واقعاً روی Disk وجود دارند و هر نام دقیقاً به یک فایل منتهی می‌شود.
</p>

<h4>بررسی‌های این مرحله</h4>

<ul>
<li>تشخیص خودکار Root پروژه بدون مسیر شخصی Hard-coded</li>
<li>بررسی Branch و HEAD فعلی</li>
<li>نمایش وضعیت Working Tree بدون ایجاد تغییر</li>
<li>جست‌وجوی Artifact اصلی Website</li>
<li>جست‌وجوی Artifact اصلی Book</li>
<li>جست‌وجوی Artifact اصلی Transcript</li>
</ul>

<h4>معیار PASS</h4>

<p>
Repository باید روی Branch مورد انتظار و HEAD شروع این Notebook باشد و برای هر یک از سه Artifact اصلی دقیقاً یک فایل واقعی پیدا شود.
اگر فایلی مفقود باشد یا بیش از یک Candidate هم‌نام وجود داشته باشد، مرحله FAIL می‌شود و قبل از ادامه علت آن بررسی خواهد شد.
</p>

</div>

In [1]:
# Code Cell 01

from pathlib import Path
import subprocess


def git_output(*args):
    return subprocess.check_output(
        ["git", *args],
        text=True,
        stderr=subprocess.STDOUT,
    ).strip()


project_root = Path(git_output("rev-parse", "--show-toplevel")).resolve()
current_branch = git_output("branch", "--show-current")
current_head = git_output("rev-parse", "--short", "HEAD")
git_status = git_output("status", "--short")

expected_branch = "rag-phase2"
expected_head = "45a796a"

search_roots = [
    project_root / "data" / "phase2",
    project_root / "reports" / "phase2",
    project_root / "artifacts" / "phase2",
]

source_keywords = {
    "website": ("website",),
    "book": ("book", "books"),
    "transcript": ("transcript", "transcripts"),
}

allowed_suffixes = {".json", ".jsonl", ".csv", ".md"}

print("PROJECT CONTEXT")
print("=" * 80)
print("Project root:", project_root)
print("Branch:", current_branch)
print("HEAD:", current_head)
print("Working tree:", "CLEAN" if not git_status else "HAS CHANGES")

if git_status:
    print("\nGit status:")
    print(git_status)

print("\nPHASE 2 INPUT DISCOVERY")
print("=" * 80)

discovered_candidates = {}

for source, keywords in source_keywords.items():
    candidates = []

    for root in search_roots:
        if not root.exists():
            continue

        for path in root.rglob("*"):
            if not path.is_file():
                continue

            relative_text = str(path.relative_to(project_root)).lower()

            if (
                path.suffix.lower() in allowed_suffixes
                and any(keyword in relative_text for keyword in keywords)
            ):
                candidates.append(path)

    candidates = sorted(set(candidates))
    discovered_candidates[source] = candidates

    print(f"\n{source.upper()} CANDIDATES: {len(candidates)}")

    if candidates:
        for path in candidates:
            print(" -", path.relative_to(project_root))
    else:
        print(" - No candidate files found")

branch_ok = current_branch == expected_branch
head_ok = current_head == expected_head

print("\nREPOSITORY START CHECK")
print("=" * 80)
print("Branch check:", "PASS" if branch_ok else "FAIL")
print("HEAD check:", "PASS" if head_ok else "FAIL")
print("Input artifact status: DISCOVERY")

assert branch_ok, f"Unexpected branch: {current_branch}"
assert head_ok, f"Unexpected HEAD: {current_head}"

PROJECT CONTEXT
Project root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Branch: rag-phase2
HEAD: 45a796a
Working tree: HAS CHANGES

Git status:
?? 07_unified_schema_and_lineage.ipynb
?? ../../schemas/

PHASE 2 INPUT DISCOVERY

WEBSITE CANDIDATES: 540
 - data/phase2/chunks/website_chunks_v2.json
 - data/phase2/chunks/website_chunks_v3.json
 - data/phase2/interim/website_extracted_v2.json
 - data/phase2/manifests/website_manifest_v1.json
 - data/phase2/processed/website_citation_locators_v1.json
 - data/phase2/processed/website_documents_v1.json
 - data/phase2/processed/website_documents_v2.json
 - data/phase2/raw/website/008E61CC/source.json
 - data/phase2/raw/website/00D061A8/source.json
 - data/phase2/raw/website/00D9A87F/source.json
 - data/phase2/raw/website/00EDA828/source.json
 - data/phase2/raw/website/0152EE71/source.json
 - data/phase2/raw/website/01AE96B5/source.json
 - data/phase2/raw/website/01D66D39/source.json
 - data/phase2/raw/website/021F

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>شناسایی Source of Truth از روی Acceptance Reportهای رسمی</h3>

<p>
Discovery اولیه نشان داد برای Website و Book چند Artifact و Version مختلف روی Disk وجود دارد.
بنابراین انتخاب ورودی Notebook 07 صرفاً بر اساس نام فایل، شماره Version یا جدیدتر به‌نظر رسیدن فایل مجاز نیست.
</p>

<p>
در این مرحله Acceptance Report رسمی هر سه Pipeline قبلی مستقیماً از Disk خوانده می‌شود تا مشخص شود کدام Artifact به‌عنوان خروجی پذیرفته‌شده ثبت شده است.
این مرحله فقط Inspection انجام می‌دهد و هیچ فایلی را تغییر نمی‌دهد.
</p>

<h4>گزارش‌های مورد بررسی</h4>

<ul>
<li><code>reports/phase2/website_acceptance_v1.json</code></li>
<li><code>reports/phase2/book_ingestion_acceptance_v1.json</code></li>
<li><code>reports/phase2/transcript_ingestion_acceptance_v1.json</code></li>
</ul>

<h4>هدف</h4>

<p>
Path، Version، Status، Count، Hash یا هر Metadata مرتبط با Artifact پذیرفته‌شده فقط از محتوای واقعی همین گزارش‌ها استخراج می‌شود.
در این مرحله هنوز هیچ Candidate به‌عنوان ورودی نهایی NB07 انتخاب نمی‌شود مگر اینکه گزارش رسمی آن را صریحاً مشخص کرده باشد.
</p>

<h4>معیار PASS</h4>

<p>
هر سه Acceptance Report باید روی Disk موجود، JSON معتبر و قابل خواندن باشند.
ساختار و فیلدهای مرتبط آن‌ها باید برای تعیین Source of Truth قابل مشاهده باشد.
</p>

</div>

In [2]:
# Code Cell 02

import json


acceptance_reports = {
    "website": project_root / "reports" / "phase2" / "website_acceptance_v1.json",
    "book": project_root / "reports" / "phase2" / "book_ingestion_acceptance_v1.json",
    "transcript": project_root / "reports" / "phase2" / "transcript_ingestion_acceptance_v1.json",
}

interesting_terms = (
    "artifact",
    "path",
    "file",
    "status",
    "accept",
    "version",
    "count",
    "record",
    "hash",
    "sha",
    "output",
)


def collect_interesting_fields(value, prefix=""):
    rows = []

    if isinstance(value, dict):
        for key, child in value.items():
            child_prefix = f"{prefix}.{key}" if prefix else str(key)

            if any(term in str(key).lower() for term in interesting_terms):
                if not isinstance(child, (dict, list)):
                    rows.append((child_prefix, child))

            rows.extend(collect_interesting_fields(child, child_prefix))

    elif isinstance(value, list):
        for index, child in enumerate(value):
            rows.extend(
                collect_interesting_fields(
                    child,
                    f"{prefix}[{index}]",
                )
            )

    return rows


loaded_acceptance_reports = {}

print("OFFICIAL ACCEPTANCE REPORT INSPECTION")
print("=" * 80)

for source, report_path in acceptance_reports.items():
    print(f"\n{source.upper()}")
    print("-" * 80)
    print("Report:", report_path.relative_to(project_root))
    print("Exists:", report_path.is_file())

    if not report_path.is_file():
        print("JSON status: NOT CHECKED")
        continue

    try:
        with report_path.open("r", encoding="utf-8") as handle:
            report = json.load(handle)

        loaded_acceptance_reports[source] = report

        print("JSON status: VALID")
        print("Top-level type:", type(report).__name__)

        if isinstance(report, dict):
            print("Top-level keys:", ", ".join(report.keys()))

        fields = collect_interesting_fields(report)

        print("\nRelevant fields:")
        if fields:
            for key, value in fields:
                print(f" - {key}: {value}")
        else:
            print(" - No matching scalar fields found")

    except Exception as exc:
        print("JSON status: INVALID")
        print("Error:", type(exc).__name__, str(exc))


reports_exist = all(path.is_file() for path in acceptance_reports.values())
reports_valid = len(loaded_acceptance_reports) == len(acceptance_reports)

print("\nACCEPTANCE REPORT GATE")
print("=" * 80)
print("All reports exist:", "PASS" if reports_exist else "FAIL")
print("All reports valid JSON:", "PASS" if reports_valid else "FAIL")

acceptance_report_gate = reports_exist and reports_valid
print(
    "ACCEPTANCE REPORT INSPECTION:",
    "PASS" if acceptance_report_gate else "FAIL",
)

assert reports_exist, "One or more official acceptance reports are missing."
assert reports_valid, "One or more official acceptance reports are not valid JSON."

OFFICIAL ACCEPTANCE REPORT INSPECTION

WEBSITE
--------------------------------------------------------------------------------
Report: reports/phase2/website_acceptance_v1.json
Exists: True
JSON status: VALID
Top-level type: dict
Top-level keys: schema_version, notebook, source, attribution, raw, interim, processed, chunks, citation, quality_flags, acceptance_status, limitations

Relevant fields:
 - schema_version: 1.0
 - source.manifest.path: data/phase2/manifests/website_manifest_v1.json
 - source.manifest.sha256: 35b41bfd6d2bbdc6e5799f55a864ce1f64cb798208bf0b55244c72a6cbda5be0
 - source.manifest.record_count: 529
 - source.manifest.status: accepted
 - raw.accepted_article_count: 529
 - raw.hash_mismatches: 0
 - interim.path: data/phase2/interim/website_extracted_v2.json
 - interim.record_count: 529
 - interim.status: accepted
 - processed.superseded.path: data/phase2/processed/website_documents_v1.json
 - processed.accepted.path: data/phase2/processed/website_documents_v2.json
 - p

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بازاعتبارسنجی مستقل Artifactهای پذیرفته‌شده سه منبع</h3>

<p>
Acceptance Reportها Source of Truth سه ورودی اصلی Notebook 07 را مشخص کردند.
در این مرحله خود Artifactهای پذیرفته‌شده مستقیماً از Disk دوباره خوانده می‌شوند تا قبل از طراحی Common Schema، وجود فایل، Hash، تعداد رکورد و ساختار واقعی داده مستقل از حافظه Notebookهای قبلی بررسی شود.
</p>

<p>
هیچ تبدیل، Normalization یا Modification در این مرحله انجام نمی‌شود.
هدف فقط Independent Re-validation و شناخت ساختار واقعی ورودی‌ها است.
</p>

<h4>Artifactهای پذیرفته‌شده</h4>

<ul>
<li>Website: <code>data/phase2/processed/website_documents_v2.json</code></li>
<li>Book: <code>data/phase2/interim/book_ingested_v2.json</code></li>
<li>Transcript: <code>data/phase2/interim/transcripts_extracted_v1.jsonl</code></li>
</ul>

<h4>بررسی‌های این مرحله</h4>

<ul>
<li>وجود واقعی هر سه Artifact روی Disk</li>
<li>محاسبه مستقل <code>SHA-256</code></li>
<li>خواندن مستقل JSON و JSONL</li>
<li>نمایش نوع ساختار Top-level</li>
<li>نمایش تعداد رکورد در ساختارهایی که Count آن بدون حدس قابل تعیین است</li>
<li>نمایش فیلدهای اولین رکورد برای شناخت قرارداد واقعی هر Source</li>
<li>مقایسه Hash فقط در مواردی که Acceptance Report مقدار مرجع صریح دارد</li>
</ul>

<h4>معیار PASS</h4>

<p>
هر سه Artifact باید موجود و قابل Parse باشند.
برای Transcript، Hash و تعداد رکورد باید با Acceptance Report رسمی منطبق باشد.
برای Website و Book در این مرحله هیچ Hash یا Count ناموجودی از روی حدس ساخته نمی‌شود و ساختار واقعی آن‌ها مبنای Validation مرحله بعد خواهد بود.
</p>

</div>

In [3]:
# Code Cell 03

import hashlib
import json


accepted_inputs = {
    "website": {
        "path": project_root / "data" / "phase2" / "processed" / "website_documents_v2.json",
        "expected_sha256": None,
        "expected_record_count": 529,
        "format": "json",
    },
    "book": {
        "path": project_root / "data" / "phase2" / "interim" / "book_ingested_v2.json",
        "expected_sha256": None,
        "expected_record_count": None,
        "format": "json",
    },
    "transcript": {
        "path": project_root / "data" / "phase2" / "interim" / "transcripts_extracted_v1.jsonl",
        "expected_sha256": "1e1d2ec379e9e8b5108efd1a5417e2a7d7394f78395fbf7164eae93ae0887848",
        "expected_record_count": 16226,
        "format": "jsonl",
    },
}


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


def load_jsonl(path):
    records = []

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSONL at line {line_number}: {exc}"
                ) from exc

    return records


validated_inputs = {}

print("INDEPENDENT ACCEPTED ARTIFACT REVALIDATION")
print("=" * 80)

for source, spec in accepted_inputs.items():
    path = spec["path"]

    print(f"\n{source.upper()}")
    print("-" * 80)
    print("Path:", path.relative_to(project_root))
    print("Exists:", path.is_file())

    if not path.is_file():
        print("Parse status: NOT CHECKED")
        continue

    actual_sha256 = sha256_file(path)
    print("SHA-256:", actual_sha256)

    expected_sha256 = spec["expected_sha256"]

    if expected_sha256 is None:
        print("Hash comparison: NOT AVAILABLE IN CURRENT ACCEPTANCE EVIDENCE")
    else:
        hash_ok = actual_sha256 == expected_sha256
        print("Hash comparison:", "PASS" if hash_ok else "FAIL")

    if spec["format"] == "json":
        with path.open("r", encoding="utf-8") as handle:
            data = json.load(handle)
    else:
        data = load_jsonl(path)

    validated_inputs[source] = data

    print("Parse status: PASS")
    print("Top-level type:", type(data).__name__)

    if isinstance(data, list):
        print("Top-level record count:", len(data))

        if data:
            first_record = data[0]
            print("First record type:", type(first_record).__name__)

            if isinstance(first_record, dict):
                print(
                    "First record keys:",
                    ", ".join(sorted(first_record.keys())),
                )

    elif isinstance(data, dict):
        print("Top-level keys:", ", ".join(sorted(data.keys())))

        for key, value in data.items():
            if isinstance(value, list):
                print(f"List field '{key}' count:", len(value))

    expected_count = spec["expected_record_count"]

    if expected_count is not None and isinstance(data, list):
        count_ok = len(data) == expected_count
        print(
            "Expected record count comparison:",
            "PASS" if count_ok else "FAIL",
        )


files_exist = all(
    spec["path"].is_file()
    for spec in accepted_inputs.values()
)

parse_ok = len(validated_inputs) == len(accepted_inputs)

transcript_hash_ok = (
    accepted_inputs["transcript"]["path"].is_file()
    and sha256_file(accepted_inputs["transcript"]["path"])
    == accepted_inputs["transcript"]["expected_sha256"]
)

transcript_count_ok = (
    isinstance(validated_inputs.get("transcript"), list)
    and len(validated_inputs["transcript"])
    == accepted_inputs["transcript"]["expected_record_count"]
)

print("\nINDEPENDENT REVALIDATION GATE")
print("=" * 80)
print("All accepted artifacts exist:", "PASS" if files_exist else "FAIL")
print("All accepted artifacts parse:", "PASS" if parse_ok else "FAIL")
print("Transcript SHA-256:", "PASS" if transcript_hash_ok else "FAIL")
print("Transcript record count:", "PASS" if transcript_count_ok else "FAIL")

revalidation_gate = (
    files_exist
    and parse_ok
    and transcript_hash_ok
    and transcript_count_ok
)

print(
    "ACCEPTED INPUT REVALIDATION:",
    "PASS" if revalidation_gate else "FAIL",
)

assert files_exist, "One or more accepted input artifacts are missing."
assert parse_ok, "One or more accepted input artifacts could not be parsed."
assert transcript_hash_ok, "Transcript artifact SHA-256 mismatch."
assert transcript_count_ok, "Transcript artifact record count mismatch."

INDEPENDENT ACCEPTED ARTIFACT REVALIDATION

WEBSITE
--------------------------------------------------------------------------------
Path: data/phase2/processed/website_documents_v2.json
Exists: True
SHA-256: 2779ef60dc94c4d47c00269a80e4f3c96b65823d9d664796418cc530e2385b2c
Hash comparison: NOT AVAILABLE IN CURRENT ACCEPTANCE EVIDENCE
Parse status: PASS
Top-level type: list
Top-level record count: 529
First record type: dict
First record keys: author, author_id, canonical_url, content, content_origin, document_id, knowledge_owner, metadata, raw_content_hash, source_id, source_type, title, url
Expected record count comparison: PASS

BOOK
--------------------------------------------------------------------------------
Path: data/phase2/interim/book_ingested_v2.json
Exists: True
SHA-256: 92afca02dfd2e0bb728bb8c0b7cf99e6c8a21dac627b995f5d7a06e987f1648a
Hash comparison: NOT AVAILABLE IN CURRENT ACCEPTANCE EVIDENCE
Parse status: PASS
Top-level type: dict
Top-level keys: artifact_type, artifac

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بازرسی ساختار داخلی سه Artifact و تکمیل Input Gate</h3>

<p>
سه Artifact پذیرفته‌شده با موفقیت از Disk بازخوانی شدند، اما ساختار داخلی آن‌ها یکسان نیست.
قبل از تعریف Common Document Schema باید مشخص شود اطلاعات اصلی هر Source دقیقاً در چه فیلدها و زیرساختارهایی قرار دارد.
</p>

<p>
این مرحله فقط Schema Discovery انجام می‌دهد.
هیچ فیلدی تغییر، Rename یا Normalize نمی‌شود و هیچ Common Document هنوز ساخته نمی‌شود.
</p>

<h4>بررسی‌های این مرحله</h4>

<ul>
<li>نمایش ساختار فیلدهای یک رکورد Website و زیرساخت‌های آن</li>
<li>نمایش ساختار Package کتاب و اولین Edition پذیرفته‌شده</li>
<li>نمایش ساختار یک Segment از Transcript و اجزای Attribution، Rights و Locator</li>
<li>تطبیق Countهای Book با Acceptance Report رسمی</li>
<li>تأیید Count پذیرفته‌شده Website</li>
</ul>

<h4>معیار PASS</h4>

<p>
Countهای قابل اثبات Website و Book باید با Acceptance Report منطبق باشند و ساختار داخلی هر سه Source باید بدون Error قابل بازرسی باشد.
پس از PASS شدن این مرحله، می‌توان Common Schema را بر اساس داده واقعی طراحی کرد، نه بر اساس حدس.
</p>

</div>

In [4]:
# Code Cell 04

def describe_structure(value, prefix="", depth=0, max_depth=3):
    rows = []

    if depth > max_depth:
        return rows

    if isinstance(value, dict):
        for key in sorted(value.keys()):
            child = value[key]
            field_path = f"{prefix}.{key}" if prefix else key

            if isinstance(child, dict):
                rows.append((field_path, "dict", len(child)))
                rows.extend(
                    describe_structure(
                        child,
                        field_path,
                        depth + 1,
                        max_depth,
                    )
                )

            elif isinstance(child, list):
                rows.append((field_path, "list", len(child)))

                if child and depth < max_depth:
                    rows.extend(
                        describe_structure(
                            child[0],
                            f"{field_path}[0]",
                            depth + 1,
                            max_depth,
                        )
                    )

            else:
                rows.append(
                    (
                        field_path,
                        type(child).__name__,
                        None,
                    )
                )

    return rows


website_data = validated_inputs["website"]
book_data = validated_inputs["book"]
transcript_data = validated_inputs["transcript"]

print("SOURCE STRUCTURE DISCOVERY")
print("=" * 80)

print("\nWEBSITE RECORD STRUCTURE")
print("-" * 80)

for field_path, field_type, size in describe_structure(
    website_data[0],
    max_depth=3,
):
    suffix = f" [{size}]" if size is not None else ""
    print(f"{field_path}: {field_type}{suffix}")


print("\nBOOK PACKAGE STRUCTURE")
print("-" * 80)

for field_path, field_type, size in describe_structure(
    book_data,
    max_depth=3,
):
    suffix = f" [{size}]" if size is not None else ""
    print(f"{field_path}: {field_type}{suffix}")


print("\nTRANSCRIPT RECORD STRUCTURE")
print("-" * 80)

for field_path, field_type, size in describe_structure(
    transcript_data[0],
    max_depth=3,
):
    suffix = f" [{size}]" if size is not None else ""
    print(f"{field_path}: {field_type}{suffix}")


website_report = loaded_acceptance_reports["website"]
book_report = loaded_acceptance_reports["book"]

expected_website_count = (
    website_report["processed"]["accepted"]["document_count"]
)

expected_book_editions = (
    book_report["accepted_scope"]["edition_count"]
)
expected_book_paragraphs = (
    book_report["accepted_scope"]["paragraph_count"]
)
expected_book_supplemental = (
    book_report["accepted_scope"]["supplemental_content_count"]
)
expected_book_assets = (
    book_report["accepted_scope"]["package_asset_count"]
)

website_count_ok = len(website_data) == expected_website_count

book_editions_ok = (
    book_data.get("edition_count") == expected_book_editions
    and len(book_data.get("edition_records", [])) == expected_book_editions
)

book_paragraphs_ok = (
    book_data.get("paragraph_count") == expected_book_paragraphs
)

book_supplemental_ok = (
    book_data.get("supplemental_content_count")
    == expected_book_supplemental
    and len(book_data.get("supplemental_content", []))
    == expected_book_supplemental
)

book_assets_ok = (
    book_data.get("package_asset_count") == expected_book_assets
    and len(book_data.get("package_assets", []))
    == expected_book_assets
)

print("\nINPUT STRUCTURE GATE")
print("=" * 80)
print(
    "Website accepted document count:",
    "PASS" if website_count_ok else "FAIL",
)
print(
    "Book edition count:",
    "PASS" if book_editions_ok else "FAIL",
)
print(
    "Book paragraph count:",
    "PASS" if book_paragraphs_ok else "FAIL",
)
print(
    "Book supplemental content count:",
    "PASS" if book_supplemental_ok else "FAIL",
)
print(
    "Book package asset count:",
    "PASS" if book_assets_ok else "FAIL",
)

input_structure_gate = all(
    [
        website_count_ok,
        book_editions_ok,
        book_paragraphs_ok,
        book_supplemental_ok,
        book_assets_ok,
    ]
)

print(
    "NB07 INPUT STRUCTURE GATE:",
    "PASS" if input_structure_gate else "FAIL",
)

assert website_count_ok, "Website accepted document count mismatch."
assert book_editions_ok, "Book edition count mismatch."
assert book_paragraphs_ok, "Book paragraph count mismatch."
assert book_supplemental_ok, "Book supplemental content count mismatch."
assert book_assets_ok, "Book package asset count mismatch."

SOURCE STRUCTURE DISCOVERY

WEBSITE RECORD STRUCTURE
--------------------------------------------------------------------------------
author: str
author_id: str
canonical_url: str
content: str
content_origin: str
document_id: str
knowledge_owner: str
metadata: dict [4]
metadata.article_id: str
metadata.study_time_minutes: int
metadata.tags: list [6]
metadata.tags[0].name: str
metadata.tags[0].tag_id: str
metadata.view_count: int
raw_content_hash: str
source_id: str
source_type: str
title: str
url: str

BOOK PACKAGE STRUCTURE
--------------------------------------------------------------------------------
artifact_type: str
artifact_version: str
coverage_state: dict [9]
coverage_state.body_paragraphs: str
coverage_state.charts_machine_readable_content: str
coverage_state.diagram_machine_readable_content: str
coverage_state.embedded_objects: str
coverage_state.endnotes: str
coverage_state.footers: str
coverage_state.footnotes: str
coverage_state.headers: str
coverage_state.media_files: s

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بررسی پوشش Metadata، Attribution و Lineage در سه منبع</h3>

<p>
ساختار داخلی سه Artifact مشخص شده است، اما برای طراحی Common Schema باید مشخص شود شناسه‌ها، Metadata، Attribution و Lineage در کل Dataset چگونه توزیع شده‌اند.
</p>

<p>
در بررسی اولیه مشخص شد <code>transcript_unit_id</code> برخلاف <code>utterance_id</code> و <code>locator_id</code> شناسه یکتای هر Segment نیست.
بنابراین یکتا بودن آن به‌عنوان Gate صحیح نیست.
در این نسخه، رابطه واقعی میان <code>transcript_unit_id</code> و <code>physical_file_id</code> بررسی می‌شود تا Contract بر اساس داده واقعی تعیین شود.
</p>

<p>
این مرحله فقط Profiling و Validation انجام می‌دهد و هیچ Metadata مفقود، Speaker یا Timestamp حدس زده یا Repair نمی‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>شناسه‌های Document در Website تکراری نباشند.</li>
<li>شناسه Edition و Paragraph در Book تکراری نباشند.</li>
<li><code>utterance_id</code> و <code>locator_id</code> در Transcript یکتا باشند.</li>
<li>رابطه <code>transcript_unit_id</code> و <code>physical_file_id</code> بدون ابهام باشد.</li>
<li>Hashهای لازم برای Lineage مفقود نباشند.</li>
</ul>

<p>
مواردی مانند Rights فاقد Evidence، Speaker حل‌نشده و Timestamp مفقود در این مرحله Failure محسوب نمی‌شوند؛
این وضعیت‌ها باید بدون جعل اطلاعات در Common Schema حفظ شوند.
</p>

</div>

In [5]:
# Code Cell 05

from collections import Counter, defaultdict


def missing_count(records, getter):
    count = 0

    for record in records:
        value = getter(record)

        if value is None:
            count += 1
        elif isinstance(value, str) and not value.strip():
            count += 1

    return count


def duplicate_count(values):
    clean_values = [
        value
        for value in values
        if value is not None and value != ""
    ]
    return len(clean_values) - len(set(clean_values))


print("SOURCE-WIDE METADATA AND LINEAGE PROFILING")
print("=" * 80)


print("\nWEBSITE")
print("-" * 80)

website_document_ids = [
    record.get("document_id")
    for record in website_data
]
website_source_ids = [
    record.get("source_id")
    for record in website_data
]

website_source_types = Counter(
    record.get("source_type")
    for record in website_data
)
website_content_origins = Counter(
    record.get("content_origin")
    for record in website_data
)

print("Documents:", len(website_data))
print("Duplicate document_id:", duplicate_count(website_document_ids))
print("Duplicate source_id:", duplicate_count(website_source_ids))
print("Missing raw_content_hash:", missing_count(
    website_data,
    lambda record: record.get("raw_content_hash"),
))
print("Missing author:", missing_count(
    website_data,
    lambda record: record.get("author"),
))
print("Missing knowledge_owner:", missing_count(
    website_data,
    lambda record: record.get("knowledge_owner"),
))
print("Missing canonical_url:", missing_count(
    website_data,
    lambda record: record.get("canonical_url"),
))
print("source_type values:", dict(website_source_types))
print("content_origin values:", dict(website_content_origins))


print("\nBOOK")
print("-" * 80)

edition_records = book_data.get("edition_records", [])
book_paragraphs = [
    paragraph
    for edition in edition_records
    for paragraph in edition.get("paragraphs", [])
]

book_work_ids = [
    edition.get("work_id")
    for edition in edition_records
]
book_edition_ids = [
    edition.get("edition_id")
    for edition in edition_records
]
book_paragraph_ids = [
    paragraph.get("paragraph_id")
    for paragraph in book_paragraphs
]

book_rights_statuses = Counter(
    edition.get("rights_status")
    for edition in edition_records
)

print("Editions:", len(edition_records))
print("Paragraphs:", len(book_paragraphs))
print("Unique work_id:", len(set(book_work_ids)))
print("Duplicate edition_id:", duplicate_count(book_edition_ids))
print("Duplicate paragraph_id:", duplicate_count(book_paragraph_ids))
print("Missing paragraph text_sha256:", missing_count(
    book_paragraphs,
    lambda paragraph: paragraph.get("text_sha256"),
))
print("Missing edition author:", missing_count(
    edition_records,
    lambda edition: edition.get("author"),
))
print("Missing edition knowledge_owner:", missing_count(
    edition_records,
    lambda edition: edition.get("knowledge_owner"),
))
print("Missing rights_holder:", missing_count(
    edition_records,
    lambda edition: edition.get("rights_holder"),
))
print("Missing rights_evidence_id:", missing_count(
    edition_records,
    lambda edition: edition.get("rights_evidence_id"),
))
print("Missing allowed_uses:", missing_count(
    edition_records,
    lambda edition: edition.get("allowed_uses"),
))
print("rights_status values:", dict(book_rights_statuses))


print("\nTRANSCRIPT")
print("-" * 80)

transcript_unit_ids = [
    record.get("transcript_unit_id")
    for record in transcript_data
]
utterance_ids = [
    record.get("utterance_id")
    for record in transcript_data
]
locator_ids = [
    record.get("source_locator", {}).get("locator_id")
    for record in transcript_data
]
physical_file_ids = [
    record.get("source_identity", {}).get("physical_file_id")
    for record in transcript_data
]

unit_to_files = defaultdict(set)
file_to_units = defaultdict(set)

for record in transcript_data:
    unit_id = record.get("transcript_unit_id")
    physical_file_id = record.get(
        "source_identity", {}
    ).get("physical_file_id")

    if unit_id and physical_file_id:
        unit_to_files[unit_id].add(physical_file_id)
        file_to_units[physical_file_id].add(unit_id)

ambiguous_units = {
    unit_id: values
    for unit_id, values in unit_to_files.items()
    if len(values) != 1
}

ambiguous_files = {
    file_id: values
    for file_id, values in file_to_units.items()
    if len(values) != 1
}

transcript_rights_statuses = Counter(
    record.get("rights", {}).get("status")
    for record in transcript_data
)

speaker_statuses = Counter(
    record.get("speaker", {})
    .get("base", {})
    .get("status")
    for record in transcript_data
)

effective_speakers = [
    record.get("speaker", {})
    .get("effective", {})
    .get("canonical")
    for record in transcript_data
]

records_with_source_annotations = sum(
    bool(record.get("timestamps", {}).get("source_annotations"))
    for record in transcript_data
)

expected_physical_file_count = (
    loaded_acceptance_reports["transcript"]
    ["accepted_scope"]["physical_file_count"]
)

expected_unique_utterance_count = (
    loaded_acceptance_reports["transcript"]
    ["accepted_scope"]["unique_utterance_id_count"]
)

expected_unique_locator_count = (
    loaded_acceptance_reports["transcript"]
    ["accepted_scope"]["unique_locator_id_count"]
)

print("Segments:", len(transcript_data))
print("Unique transcript_unit_id:", len(set(transcript_unit_ids)))
print("Unique physical_file_id:", len(set(physical_file_ids)))
print("Duplicate utterance_id:", duplicate_count(utterance_ids))
print("Duplicate locator_id:", duplicate_count(locator_ids))
print("Ambiguous transcript_unit_id mappings:", len(ambiguous_units))
print("Ambiguous physical_file_id mappings:", len(ambiguous_files))
print("Missing source SHA-256:", missing_count(
    transcript_data,
    lambda record: record.get("source_identity", {}).get("source_sha256"),
))
print("Missing content text SHA-256:", missing_count(
    transcript_data,
    lambda record: record.get("content", {}).get("text_sha256"),
))
print("Missing effective canonical speaker:", sum(
    value is None or value == ""
    for value in effective_speakers
))
print(
    "Records with source timestamp annotations:",
    records_with_source_annotations,
)
print("rights.status values:", dict(transcript_rights_statuses))
print("speaker.base.status values:", dict(speaker_statuses))


website_identity_ok = (
    duplicate_count(website_document_ids) == 0
    and duplicate_count(website_source_ids) == 0
    and missing_count(
        website_data,
        lambda record: record.get("raw_content_hash"),
    ) == 0
)

book_identity_ok = (
    duplicate_count(book_edition_ids) == 0
    and duplicate_count(book_paragraph_ids) == 0
    and missing_count(
        book_paragraphs,
        lambda paragraph: paragraph.get("text_sha256"),
    ) == 0
)

transcript_identity_ok = (
    duplicate_count(utterance_ids) == 0
    and duplicate_count(locator_ids) == 0
    and len(set(utterance_ids)) == expected_unique_utterance_count
    and len(set(locator_ids)) == expected_unique_locator_count
    and len(set(transcript_unit_ids)) == expected_physical_file_count
    and len(set(physical_file_ids)) == expected_physical_file_count
    and len(ambiguous_units) == 0
    and len(ambiguous_files) == 0
    and missing_count(
        transcript_data,
        lambda record: record.get("source_identity", {}).get("source_sha256"),
    ) == 0
    and missing_count(
        transcript_data,
        lambda record: record.get("content", {}).get("text_sha256"),
    ) == 0
)

print("\nSOURCE IDENTITY AND LINEAGE GATE")
print("=" * 80)
print(
    "Website identity/lineage:",
    "PASS" if website_identity_ok else "FAIL",
)
print(
    "Book identity/lineage:",
    "PASS" if book_identity_ok else "FAIL",
)
print(
    "Transcript identity/lineage:",
    "PASS" if transcript_identity_ok else "FAIL",
)

source_profile_gate = (
    website_identity_ok
    and book_identity_ok
    and transcript_identity_ok
)

print(
    "SOURCE PROFILING GATE:",
    "PASS" if source_profile_gate else "FAIL",
)

assert website_identity_ok, "Website identity or lineage validation failed."
assert book_identity_ok, "Book identity or lineage validation failed."
assert transcript_identity_ok, "Transcript identity or lineage validation failed."

SOURCE-WIDE METADATA AND LINEAGE PROFILING

WEBSITE
--------------------------------------------------------------------------------
Documents: 529
Duplicate document_id: 0
Duplicate source_id: 0
Missing raw_content_hash: 0
Missing author: 0
Missing knowledge_owner: 0
Missing canonical_url: 0
source_type values: {'website': 529}
content_origin values: {'supervised': 453, 'primary_author': 76}

BOOK
--------------------------------------------------------------------------------
Editions: 3
Paragraphs: 2689
Unique work_id: 3
Duplicate edition_id: 0
Duplicate paragraph_id: 0
Missing paragraph text_sha256: 0
Missing edition author: 0
Missing edition knowledge_owner: 0
Missing rights_holder: 3
Missing rights_evidence_id: 3
Missing allowed_uses: 3
rights_status values: {'EVIDENCE_REQUIRED': 3}

TRANSCRIPT
--------------------------------------------------------------------------------
Segments: 16226
Unique transcript_unit_id: 183
Unique physical_file_id: 183
Duplicate utterance_id: 0
Dupli

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بررسی شناسه‌های واقعی برای طراحی Stable Document ID</h3>

<p>
Common Schema به یک <code>document_id</code> پایدار نیاز دارد.
سه Source فعلی از شناسه‌های متفاوت استفاده می‌کنند:
Website دارای <code>document_id</code> و <code>source_id</code> است،
Book دارای <code>work_id</code>، <code>edition_id</code> و <code>paragraph_id</code> است،
و Transcript دارای <code>transcript_unit_id</code> و <code>utterance_id</code> است.
</p>

<p>
قبل از تعیین سیاست شناسه مشترک، این مرحله فرمت واقعی این IDها، یکتایی آن‌ها و احتمال Collision میان سه Source را بررسی می‌کند.
هیچ ID جدیدی در این مرحله ساخته یا ذخیره نمی‌شود.
</p>

<h4>هدف</h4>

<ul>
<li>مشاهده نمونه واقعی IDهای هر Source</li>
<li>تأیید یکتایی واحد محتوایی منتخب در هر Source</li>
<li>بررسی Collision میان IDهای Website، Paragraphهای Book و Utteranceهای Transcript</li>
<li>بررسی اینکه آیا IDهای فعلی از قبل Source prefix دارند یا خیر</li>
</ul>

<h4>معیار PASS</h4>

<p>
واحد محتوایی منتخب هر Source باید دارای ID غیرخالی و یکتا باشد.
نتیجه این مرحله مبنای تصمیم بعدی برای Stable <code>document_id</code> مشترک خواهد بود.
</p>

</div>

In [6]:
# Code Cell 06

def nonempty_unique(values):
    return (
        all(isinstance(value, str) and value.strip() for value in values)
        and len(values) == len(set(values))
    )


website_ids = [
    record.get("document_id")
    for record in website_data
]

book_ids = [
    paragraph.get("paragraph_id")
    for paragraph in book_paragraphs
]

transcript_ids = [
    record.get("utterance_id")
    for record in transcript_data
]

website_source_ids = [
    record.get("source_id")
    for record in website_data
]

book_edition_ids = [
    edition.get("edition_id")
    for edition in edition_records
]

transcript_unit_ids_unique = sorted(
    set(
        record.get("transcript_unit_id")
        for record in transcript_data
        if record.get("transcript_unit_id")
    )
)


def sample_values(values, limit=5):
    return values[:limit]


def prefix_summary(values):
    prefixes = Counter()

    for value in values:
        if not isinstance(value, str) or not value:
            prefixes["<missing>"] += 1
            continue

        if ":" in value:
            prefixes[value.split(":", 1)[0]] += 1
        else:
            prefixes["<no_colon_prefix>"] += 1

    return dict(prefixes)


website_book_collision = set(website_ids) & set(book_ids)
website_transcript_collision = set(website_ids) & set(transcript_ids)
book_transcript_collision = set(book_ids) & set(transcript_ids)

print("STABLE ID DISCOVERY")
print("=" * 80)

print("\nWEBSITE")
print("-" * 80)
print("Document IDs:", len(website_ids))
print("Unique/non-empty:", nonempty_unique(website_ids))
print("document_id samples:")
for value in sample_values(website_ids):
    print(" -", value)
print("document_id prefix summary:", prefix_summary(website_ids))

print("\nsource_id samples:")
for value in sample_values(website_source_ids):
    print(" -", value)


print("\nBOOK")
print("-" * 80)
print("Paragraph IDs:", len(book_ids))
print("Unique/non-empty:", nonempty_unique(book_ids))
print("paragraph_id samples:")
for value in sample_values(book_ids):
    print(" -", value)
print("paragraph_id prefix summary:", prefix_summary(book_ids))

print("\nedition_id values:")
for value in book_edition_ids:
    print(" -", value)


print("\nTRANSCRIPT")
print("-" * 80)
print("Utterance IDs:", len(transcript_ids))
print("Unique/non-empty:", nonempty_unique(transcript_ids))
print("utterance_id samples:")
for value in sample_values(transcript_ids):
    print(" -", value)
print("utterance_id prefix summary:", prefix_summary(transcript_ids))

print("\ntranscript_unit_id samples:")
for value in sample_values(transcript_unit_ids_unique):
    print(" -", value)


print("\nCROSS-SOURCE RAW ID COLLISIONS")
print("-" * 80)
print("Website ↔ Book:", len(website_book_collision))
print("Website ↔ Transcript:", len(website_transcript_collision))
print("Book ↔ Transcript:", len(book_transcript_collision))


website_id_ok = nonempty_unique(website_ids)
book_id_ok = nonempty_unique(book_ids)
transcript_id_ok = nonempty_unique(transcript_ids)

id_discovery_gate = (
    website_id_ok
    and book_id_ok
    and transcript_id_ok
)

print("\nSTABLE ID DISCOVERY GATE")
print("=" * 80)
print("Website candidate IDs:", "PASS" if website_id_ok else "FAIL")
print("Book candidate IDs:", "PASS" if book_id_ok else "FAIL")
print("Transcript candidate IDs:", "PASS" if transcript_id_ok else "FAIL")
print(
    "STABLE ID DISCOVERY:",
    "PASS" if id_discovery_gate else "FAIL",
)

assert website_id_ok, "Website candidate document IDs are invalid."
assert book_id_ok, "Book candidate paragraph IDs are invalid."
assert transcript_id_ok, "Transcript candidate utterance IDs are invalid."

STABLE ID DISCOVERY

WEBSITE
--------------------------------------------------------------------------------
Document IDs: 529
Unique/non-empty: True
document_id samples:
 - website:008E61CC
 - website:00D061A8
 - website:00D9A87F
 - website:00EDA828
 - website:0152EE71
document_id prefix summary: {'website': 529}

source_id samples:
 - website:008E61CC
 - website:00D061A8
 - website:00D9A87F
 - website:00EDA828
 - website:0152EE71

BOOK
--------------------------------------------------------------------------------
Paragraph IDs: 2689
Unique/non-empty: True
paragraph_id samples:
 - book:adult-financial-literacy-course:fa-source:p000001
 - book:adult-financial-literacy-course:fa-source:p000002
 - book:adult-financial-literacy-course:fa-source:p000003
 - book:adult-financial-literacy-course:fa-source:p000004
 - book:adult-financial-literacy-course:fa-source:p000005
paragraph_id prefix summary: {'book': 2689}

edition_id values:
 - book:adult-financial-literacy-course:fa-source
 - book

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>کشف Schemaها و Contractهای موجود پیش از تعریف Common Document Schema</h3>

<p>
شناسه‌های پایدار هر سه Source تأیید شدند و برای ساخت شناسه جدید نیازی اثبات نشده است.
اکنون قبل از ایجاد یا تغییر هر Schema باید بررسی شود آیا Repository از قبل Contract، JSON Schema یا تعریف ساختاری مرتبط با Document، Lineage، Rights، Attribution یا Source دارد یا خیر.
</p>

<p>
این مرحله فقط Discovery انجام می‌دهد.
هیچ Schema یا فایل جدیدی ساخته یا تغییر داده نمی‌شود.
</p>

<h4>هدف</h4>

<ul>
<li>بررسی مسیرهای موجود <code>schemas/</code> و <code>schemas/phase2/</code></li>
<li>کشف فایل‌های Schema و Contract مرتبط با Document و Source</li>
<li>جلوگیری از ایجاد Contract موازی یا ناسازگار</li>
<li>مشخص‌کردن اینکه Common Document Schema باید از صفر ساخته شود یا بر پایه فایل موجود توسعه یابد</li>
</ul>

<h4>معیار PASS</h4>

<p>
وضعیت واقعی Schemaهای موجود باید بدون حدس مشخص شود و تمام Candidateهای مرتبط برای بررسی مرحله بعد فهرست شوند.
نبود Schema موجود Failure نیست؛ اما وضعیت آن باید صریح و قابل اثبات باشد.
</p>

</div>

In [7]:
# Code Cell 07

schema_search_roots = [
    project_root / "schemas",
    project_root / "src" / "phase2",
]

schema_suffixes = {
    ".json",
    ".jsonl",
    ".yaml",
    ".yml",
    ".py",
}

schema_terms = (
    "schema",
    "contract",
    "document",
    "source",
    "lineage",
    "rights",
    "attribution",
    "manifest",
)

print("EXISTING SCHEMA AND CONTRACT DISCOVERY")
print("=" * 80)

existing_schema_roots = []

for root in schema_search_roots:
    exists = root.exists()
    print(
        f"{root.relative_to(project_root)}:",
        "EXISTS" if exists else "MISSING",
    )

    if exists:
        existing_schema_roots.append(root)

candidates = []

for root in existing_schema_roots:
    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in schema_suffixes:
            continue

        relative_text = str(path.relative_to(project_root)).lower()

        if any(term in relative_text for term in schema_terms):
            candidates.append(path)

candidates = sorted(set(candidates))

print("\nRELATED CANDIDATES")
print("=" * 80)
print("Candidate files:", len(candidates))

if not candidates:
    print("No related schema or contract candidates found.")
else:
    for path in candidates:
        print(" -", path.relative_to(project_root))

print("\nJSON SCHEMA TOP-LEVEL INSPECTION")
print("=" * 80)

json_candidates = [
    path
    for path in candidates
    if path.suffix.lower() == ".json"
]

valid_json_files = 0

if not json_candidates:
    print("No JSON candidates found.")
else:
    for path in json_candidates:
        print(f"\n{path.relative_to(project_root)}")
        print("-" * 80)

        try:
            with path.open("r", encoding="utf-8") as handle:
                data = json.load(handle)

            valid_json_files += 1
            print("JSON status: VALID")
            print("Top-level type:", type(data).__name__)

            if isinstance(data, dict):
                print(
                    "Top-level keys:",
                    ", ".join(sorted(data.keys())),
                )

                if "$schema" in data:
                    print("$schema:", data["$schema"])

                if "title" in data:
                    print("title:", data["title"])

                if "type" in data:
                    print("type:", data["type"])

        except Exception as exc:
            print("JSON status: INVALID")
            print("Error:", type(exc).__name__, str(exc))

print("\nSCHEMA DISCOVERY GATE")
print("=" * 80)
print("Existing search roots:", len(existing_schema_roots))
print("Related candidates:", len(candidates))
print("Valid JSON candidates:", valid_json_files)
print("SCHEMA DISCOVERY: PASS")

EXISTING SCHEMA AND CONTRACT DISCOVERY
schemas: EXISTS
src/phase2: MISSING

RELATED CANDIDATES
Candidate files: 1
 - schemas/phase2/document.schema.json

JSON SCHEMA TOP-LEVEL INSPECTION

schemas/phase2/document.schema.json
--------------------------------------------------------------------------------
JSON status: VALID
Top-level type: dict
Top-level keys: $id, $schema, additionalProperties, allOf, properties, required, title, type
$schema: https://json-schema.org/draft/2020-12/schema
title: RAG Finance Phase 2 Common Document
type: object

SCHEMA DISCOVERY GATE
Existing search roots: 1
Related candidates: 1
Valid JSON candidates: 1
SCHEMA DISCOVERY: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>تعیین واحد محتوایی پایه برای Common Document</h3>

<p>
برای ساخت Schema مشترک باید ابتدا مشخص شود هر رکورد Unified Document نماینده چه واحدی از محتوای اصلی است.
این تصمیم نباید صرفاً بر اساس نام فیلدها یا طراحی ذهنی انجام شود.
</p>

<p>
در این مرحله برای هر Source رابطه میان شناسه پایدار، متن، Parent و Locator واقعی بررسی می‌شود تا واحد پایه‌ای که بدون از دست‌دادن Lineage می‌تواند وارد Common Document شود مشخص گردد.
</p>

<h4>Candidateهای فعلی</h4>

<ul>
<li>Website: یک Article پذیرفته‌شده</li>
<li>Book: یک Paragraph داخل Edition</li>
<li>Transcript: یک Utterance داخل Transcript Unit</li>
</ul>

<p>
این Candidateها در این مرحله فقط فرضیه طراحی هستند و با داده واقعی Validation می‌شوند.
هیچ Unified Document یا Schema هنوز ساخته نمی‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>هر Candidate باید ID یکتا و متن غیرخالی داشته باشد.</li>
<li>Book Paragraph باید به Work و Edition خود قابل اتصال باشد.</li>
<li>Transcript Utterance باید به Transcript Unit، Physical File و Locator خود قابل اتصال باشد.</li>
<li>Website Article باید به Source ID و Raw Hash خود قابل اتصال باشد.</li>
</ul>

<p>
اگر این روابط برای تمام رکوردها برقرار باشد، این سه سطح به‌عنوان واحد پایه Common Document پذیرفته می‌شوند.
</p>

</div>

In [8]:
# Code Cell 08

website_units = website_data
book_units = book_paragraphs
transcript_units = transcript_data


def nonempty_text(value):
    return isinstance(value, str) and bool(value.strip())


website_unit_ok = all(
    nonempty_text(record.get("document_id"))
    and nonempty_text(record.get("content"))
    and nonempty_text(record.get("source_id"))
    and nonempty_text(record.get("raw_content_hash"))
    for record in website_units
)


edition_lookup = {
    edition.get("edition_id"): edition
    for edition in edition_records
}

book_parent_failures = []

for paragraph in book_units:
    edition_id = paragraph.get("edition_id")
    work_id = paragraph.get("work_id")
    edition = edition_lookup.get(edition_id)

    if (
        not nonempty_text(paragraph.get("paragraph_id"))
        or not nonempty_text(paragraph.get("text"))
        or not nonempty_text(paragraph.get("text_sha256"))
        or edition is None
        or edition.get("work_id") != work_id
    ):
        book_parent_failures.append(paragraph.get("paragraph_id"))

book_unit_ok = len(book_parent_failures) == 0


transcript_parent_failures = []

for record in transcript_units:
    source_identity = record.get("source_identity", {})
    source_locator = record.get("source_locator", {})
    content = record.get("content", {})

    if (
        not nonempty_text(record.get("utterance_id"))
        or not nonempty_text(record.get("transcript_unit_id"))
        or not nonempty_text(content.get("text"))
        or not nonempty_text(content.get("text_sha256"))
        or not nonempty_text(source_identity.get("physical_file_id"))
        or not nonempty_text(source_identity.get("source_sha256"))
        or not nonempty_text(source_locator.get("locator_id"))
    ):
        transcript_parent_failures.append(record.get("utterance_id"))

transcript_unit_ok = len(transcript_parent_failures) == 0


print("COMMON DOCUMENT UNIT VALIDATION")
print("=" * 80)

print("\nWEBSITE")
print("-" * 80)
print("Candidate unit: ARTICLE")
print("Unit count:", len(website_units))
print("Identity/text/lineage completeness:", "PASS" if website_unit_ok else "FAIL")

print("\nBOOK")
print("-" * 80)
print("Candidate unit: PARAGRAPH")
print("Unit count:", len(book_units))
print("Parent/linkage failures:", len(book_parent_failures))
print("Identity/text/lineage completeness:", "PASS" if book_unit_ok else "FAIL")

print("\nTRANSCRIPT")
print("-" * 80)
print("Candidate unit: UTTERANCE")
print("Unit count:", len(transcript_units))
print("Parent/linkage failures:", len(transcript_parent_failures))
print(
    "Identity/text/lineage completeness:",
    "PASS" if transcript_unit_ok else "FAIL",
)

candidate_total = (
    len(website_units)
    + len(book_units)
    + len(transcript_units)
)

print("\nCANDIDATE UNIFIED DOCUMENT POPULATION")
print("-" * 80)
print("Website article units:", len(website_units))
print("Book paragraph units:", len(book_units))
print("Transcript utterance units:", len(transcript_units))
print("Total candidate units:", candidate_total)

unit_gate = (
    website_unit_ok
    and book_unit_ok
    and transcript_unit_ok
)

print("\nCOMMON DOCUMENT UNIT GATE")
print("=" * 80)
print(
    "Website article unit:",
    "PASS" if website_unit_ok else "FAIL",
)
print(
    "Book paragraph unit:",
    "PASS" if book_unit_ok else "FAIL",
)
print(
    "Transcript utterance unit:",
    "PASS" if transcript_unit_ok else "FAIL",
)
print(
    "COMMON DOCUMENT UNIT DECISION:",
    "PASS" if unit_gate else "FAIL",
)

assert website_unit_ok, "Website article unit validation failed."
assert book_unit_ok, "Book paragraph unit validation failed."
assert transcript_unit_ok, "Transcript utterance unit validation failed."

COMMON DOCUMENT UNIT VALIDATION

WEBSITE
--------------------------------------------------------------------------------
Candidate unit: ARTICLE
Unit count: 529
Identity/text/lineage completeness: PASS

BOOK
--------------------------------------------------------------------------------
Candidate unit: PARAGRAPH
Unit count: 2689
Parent/linkage failures: 0
Identity/text/lineage completeness: PASS

TRANSCRIPT
--------------------------------------------------------------------------------
Candidate unit: UTTERANCE
Unit count: 16226
Parent/linkage failures: 0
Identity/text/lineage completeness: PASS

CANDIDATE UNIFIED DOCUMENT POPULATION
--------------------------------------------------------------------------------
Website article units: 529
Book paragraph units: 2689
Transcript utterance units: 16226
Total candidate units: 19444

COMMON DOCUMENT UNIT GATE
Website article unit: PASS
Book paragraph unit: PASS
Transcript utterance unit: PASS
COMMON DOCUMENT UNIT DECISION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>کشف قرارداد Attribution، Rights و Locator برای Common Schema</h3>

<p>
واحد پایه هر سه Source مشخص شد.
اکنون باید تعیین شود کدام اطلاعات Attribution، Rights و Location واقعاً از Artifactهای پذیرفته‌شده قابل انتقال به Common Document هستند.
</p>

<p>
در این مرحله هیچ Metadata جدیدی ساخته نمی‌شود و هیچ مقدار مفقودی حدس زده نمی‌شود.
هدف فقط ثبت دقیق قابلیت واقعی هر Source پیش از تعریف Schema مشترک است.
</p>

<h4>بررسی‌های این مرحله</h4>

<ul>
<li>مقادیر واقعی نویسنده، مالک دانش و منشأ محتوای Website</li>
<li>شواهد Attribution ثبت‌شده در Acceptance Report سایت</li>
<li>وضعیت Rights و Locator کتاب در سطح Paragraph</li>
<li>وضعیت Speaker، Rights و Locator Transcript</li>
<li>تفکیک فیلدهای Required از Nullable و Source-specific</li>
</ul>

<h4>معیار PASS</h4>

<p>
باید بدون حدس مشخص شود هر Source چه اطلاعاتی را می‌تواند به Common Schema منتقل کند و چه اطلاعاتی باید به‌صورت <code>null</code> یا وضعیت صریح unresolved/evidence-required حفظ شود.
</p>

</div>

In [9]:
# Code Cell 09

website_acceptance = loaded_acceptance_reports["website"]

print("ATTRIBUTION, RIGHTS, AND LOCATOR CONTRACT DISCOVERY")
print("=" * 80)


print("\nWEBSITE ATTRIBUTION")
print("-" * 80)

website_author_values = Counter(
    record.get("author")
    for record in website_data
)

website_knowledge_owner_values = Counter(
    record.get("knowledge_owner")
    for record in website_data
)

print("Unique author values:", len(website_author_values))
print("Unique knowledge_owner values:", len(website_knowledge_owner_values))
print("content_origin values:", dict(website_content_origins))

print("\nAcceptance attribution section:")
print(
    json.dumps(
        website_acceptance.get("attribution", {}),
        ensure_ascii=False,
        indent=2,
    )
)

print("\nWebsite locator-capable fields:")
print(
    [
        key
        for key in (
            "document_id",
            "source_id",
            "canonical_url",
            "url",
            "raw_content_hash",
        )
        if key in website_data[0]
    ]
)


print("\nBOOK ATTRIBUTION / RIGHTS / LOCATOR")
print("-" * 80)

book_rights_contract = Counter(
    (
        edition.get("rights_status"),
        edition.get("rights_holder"),
        edition.get("rights_evidence_id"),
        str(edition.get("allowed_uses")),
    )
    for edition in edition_records
)

print("Rights combinations:", len(book_rights_contract))

for combination, count in book_rights_contract.items():
    print(" -", combination, "=>", count)

sample_book_paragraph = book_paragraphs[0]

print(
    "Paragraph locator-capable fields:",
    [
        key
        for key in (
            "paragraph_id",
            "paragraph_number",
            "work_id",
            "edition_id",
            "heading_path",
        )
        if key in sample_book_paragraph
    ],
)

print(
    "Edition attribution fields:",
    [
        key
        for key in (
            "author",
            "knowledge_owner",
            "rights_status",
            "rights_holder",
            "rights_evidence_id",
            "allowed_uses",
        )
        if key in edition_records[0]
    ],
)


print("\nTRANSCRIPT ATTRIBUTION / RIGHTS / LOCATOR")
print("-" * 80)

transcript_rights_contract = Counter(
    (
        record.get("rights", {}).get("status"),
        record.get("rights", {}).get("basis"),
    )
    for record in transcript_data
)

print("Rights combinations:", len(transcript_rights_contract))

for combination, count in transcript_rights_contract.items():
    print(" -", combination, "=>", count)

effective_speaker_basis = Counter(
    record.get("speaker", {})
    .get("effective", {})
    .get("basis")
    for record in transcript_data
)

print("Effective speaker basis values:", dict(effective_speaker_basis))

sample_transcript = transcript_data[0]

print(
    "Transcript locator fields:",
    sorted(sample_transcript.get("source_locator", {}).keys()),
)

print(
    "Transcript source identity fields:",
    sorted(sample_transcript.get("source_identity", {}).keys()),
)

print(
    "Transcript speaker fields:",
    sorted(sample_transcript.get("speaker", {}).keys()),
)


print("\nCOMMON CONTRACT OBSERVATIONS")
print("=" * 80)

print("Website attribution available:", bool(website_author_values))
print(
    "Book unresolved rights preserved:",
    all(
        edition.get("rights_status") == "EVIDENCE_REQUIRED"
        for edition in edition_records
    ),
)
print(
    "Transcript unresolved rights preserved:",
    all(
        record.get("rights", {}).get("status") == "EVIDENCE_REQUIRED"
        for record in transcript_data
    ),
)
print(
    "Transcript unresolved speakers present:",
    any(
        record.get("speaker", {})
        .get("effective", {})
        .get("canonical") is None
        for record in transcript_data
    ),
)

contract_discovery_gate = (
    bool(website_author_values)
    and all(
        "paragraph_id" in paragraph
        and "edition_id" in paragraph
        and "work_id" in paragraph
        for paragraph in book_paragraphs
    )
    and all(
        record.get("source_locator", {}).get("locator_id")
        for record in transcript_data
    )
)

print(
    "\nATTRIBUTION / RIGHTS / LOCATOR DISCOVERY:",
    "PASS" if contract_discovery_gate else "FAIL",
)

assert contract_discovery_gate, (
    "Required attribution or locator evidence is incomplete."
)

ATTRIBUTION, RIGHTS, AND LOCATOR CONTRACT DISCOVERY

WEBSITE ATTRIBUTION
--------------------------------------------------------------------------------
Unique author values: 19
Unique knowledge_owner values: 1
content_origin values: {'supervised': 453, 'primary_author': 76}

Acceptance attribution section:
{
  "primary_author": 76,
  "supervised": 453,
  "anomalies": 0
}

Website locator-capable fields:
['document_id', 'source_id', 'canonical_url', 'url', 'raw_content_hash']

BOOK ATTRIBUTION / RIGHTS / LOCATOR
--------------------------------------------------------------------------------
Rights combinations: 1
 - ('EVIDENCE_REQUIRED', None, None, 'None') => 3
Paragraph locator-capable fields: ['paragraph_id', 'paragraph_number', 'work_id', 'edition_id', 'heading_path']
Edition attribution fields: ['author', 'knowledge_owner', 'rights_status', 'rights_holder', 'rights_evidence_id', 'allowed_uses']

TRANSCRIPT ATTRIBUTION / RIGHTS / LOCATOR
------------------------------------------

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>تعریف نسخه اولیه Common Document Contract در حافظه</h3>

<p>
پس از Discovery ساختار، شناسه‌ها، Attribution، Rights و Locator سه Source، اکنون نسخه اولیه Common Document Contract تعریف می‌شود.
این Contract هنوز روی Disk ذخیره نمی‌شود و ابتدا باید روی داده واقعی هر سه Source آزمایش شود.
</p>

<p>
اصل طراحی این است که اطلاعات مشترک در سطح بالا قرار گیرند و تفاوت‌های Source در ساختارهای <code>locator</code>، <code>attribution</code>، <code>rights</code> و <code>lineage</code> حفظ شوند.
هیچ Metadata مفقودی حدس زده نمی‌شود.
</p>

<h4>تصمیم‌های اصلی Contract</h4>

<ul>
<li><code>document_id</code> از Stable ID موجود هر Source حفظ می‌شود.</li>
<li><code>source_id</code> به واحد والد منبع اشاره می‌کند.</li>
<li><code>text</code> و <code>text_sha256</code> محتوای همان Common Document را ثبت می‌کنند.</li>
<li><code>raw_content_hash</code> Hash منبع بالادستی را برای Lineage نگه می‌دارد.</li>
<li>Attribution نویسنده، مالک دانش و Speaker را بدون تبدیل نقش‌ها به یکدیگر حفظ می‌کند.</li>
<li>Rights می‌تواند دارای وضعیت <code>EVIDENCE_REQUIRED</code> و مقادیر Nullable باشد.</li>
<li>Locator برای Website، Book و Transcript Source-specific باقی می‌ماند.</li>
</ul>

<h4>معیار PASS</h4>

<p>
Contract باید بتواند یک نمونه واقعی از هر سه Source را بدون جعل Metadata نمایش دهد و تمام فیلدهای Required آن از داده واقعی قابل تولید باشند.
در این مرحله فقط ساختار در حافظه بررسی می‌شود و هیچ Artifact جدیدی نوشته نمی‌شود.
</p>

</div>

In [10]:
# Code Cell 10

common_document_contract_v1 = {
    "schema_version": "common_document_v1",
    "required_fields": [
        "schema_version",
        "document_id",
        "source_type",
        "source_id",
        "text",
        "text_sha256",
        "raw_content_hash",
        "title",
        "language",
        "content_origin",
        "attribution",
        "rights",
        "locator",
        "lineage",
    ],
    "nullable_fields": [
        "title",
        "language",
        "attribution.author",
        "attribution.knowledge_owner",
        "attribution.speaker",
        "rights.rights_holder",
        "rights.rights_evidence_id",
        "rights.allowed_uses",
    ],
    "source_types": [
        "website",
        "book",
        "transcript",
    ],
}


def build_website_common_sample(record):
    return {
        "schema_version": "common_document_v1",
        "document_id": record["document_id"],
        "source_type": "website",
        "source_id": record["source_id"],
        "text": record["content"],
        "text_sha256": hashlib.sha256(
            record["content"].encode("utf-8")
        ).hexdigest(),
        "raw_content_hash": record["raw_content_hash"],
        "title": record.get("title"),
        "language": None,
        "content_origin": record.get("content_origin"),
        "attribution": {
            "author": record.get("author"),
            "knowledge_owner": record.get("knowledge_owner"),
            "speaker": None,
        },
        "rights": {
            "status": None,
            "basis": None,
            "rights_holder": None,
            "rights_evidence_id": None,
            "allowed_uses": None,
        },
        "locator": {
            "canonical_url": record.get("canonical_url"),
            "url": record.get("url"),
        },
        "lineage": {
            "parent_id": None,
            "raw_source_path": None,
            "raw_source_hash": record["raw_content_hash"],
        },
    }


def build_book_common_sample(paragraph):
    edition = edition_lookup[paragraph["edition_id"]]
    source_file = edition["source_file"]

    return {
        "schema_version": "common_document_v1",
        "document_id": paragraph["paragraph_id"],
        "source_type": "book",
        "source_id": paragraph["edition_id"],
        "text": paragraph["text"],
        "text_sha256": paragraph["text_sha256"],
        "raw_content_hash": source_file["file_sha256"],
        "title": edition.get("title"),
        "language": edition.get("language"),
        "content_origin": "authored_work",
        "attribution": {
            "author": edition.get("author"),
            "knowledge_owner": edition.get("knowledge_owner"),
            "speaker": None,
        },
        "rights": {
            "status": edition.get("rights_status"),
            "basis": None,
            "rights_holder": edition.get("rights_holder"),
            "rights_evidence_id": edition.get("rights_evidence_id"),
            "allowed_uses": edition.get("allowed_uses"),
        },
        "locator": {
            "work_id": paragraph.get("work_id"),
            "edition_id": paragraph.get("edition_id"),
            "paragraph_id": paragraph.get("paragraph_id"),
            "paragraph_number": paragraph.get("paragraph_number"),
            "heading_path": paragraph.get("heading_path"),
        },
        "lineage": {
            "parent_id": paragraph.get("edition_id"),
            "raw_source_path": source_file.get("relative_path"),
            "raw_source_hash": source_file.get("file_sha256"),
        },
    }


def build_transcript_common_sample(record):
    source_identity = record["source_identity"]
    source_locator = record["source_locator"]
    speaker = record["speaker"]
    rights = record["rights"]

    return {
        "schema_version": "common_document_v1",
        "document_id": record["utterance_id"],
        "source_type": "transcript",
        "source_id": record["transcript_unit_id"],
        "text": record["content"]["text"],
        "text_sha256": record["content"]["text_sha256"],
        "raw_content_hash": source_identity["source_sha256"],
        "title": record.get("program"),
        "language": None,
        "content_origin": "transcript",
        "attribution": {
            "author": None,
            "knowledge_owner": None,
            "speaker": speaker.get("effective", {}).get("canonical"),
        },
        "rights": {
            "status": rights.get("status"),
            "basis": rights.get("basis"),
            "rights_holder": None,
            "rights_evidence_id": None,
            "allowed_uses": None,
        },
        "locator": {
            "locator_id": source_locator.get("locator_id"),
            "source_line_start": source_locator.get("source_line_start"),
            "source_line_end": source_locator.get("source_line_end"),
            "program_id": record.get("program_id"),
            "episode_number": source_identity.get("episode_number"),
            "season_number": source_identity.get("season_number"),
            "physical_file_id": source_identity.get("physical_file_id"),
        },
        "lineage": {
            "parent_id": record.get("transcript_unit_id"),
            "raw_source_path": source_identity.get("relative_path"),
            "raw_source_hash": source_identity.get("source_sha256"),
        },
    }


common_samples = {
    "website": build_website_common_sample(website_data[0]),
    "book": build_book_common_sample(book_paragraphs[0]),
    "transcript": build_transcript_common_sample(transcript_data[0]),
}


def validate_common_sample(sample):
    missing_required = [
        field
        for field in common_document_contract_v1["required_fields"]
        if field not in sample
    ]

    source_type_ok = (
        sample.get("source_type")
        in common_document_contract_v1["source_types"]
    )

    identity_ok = (
        nonempty_text(sample.get("document_id"))
        and nonempty_text(sample.get("source_id"))
        and nonempty_text(sample.get("text"))
        and nonempty_text(sample.get("text_sha256"))
        and nonempty_text(sample.get("raw_content_hash"))
    )

    nested_ok = all(
        isinstance(sample.get(field), dict)
        for field in (
            "attribution",
            "rights",
            "locator",
            "lineage",
        )
    )

    return {
        "missing_required": missing_required,
        "source_type_ok": source_type_ok,
        "identity_ok": identity_ok,
        "nested_ok": nested_ok,
        "pass": (
            not missing_required
            and source_type_ok
            and identity_ok
            and nested_ok
        ),
    }


print("COMMON DOCUMENT CONTRACT V1 — IN-MEMORY VALIDATION")
print("=" * 80)

sample_results = {}

for source, sample in common_samples.items():
    result = validate_common_sample(sample)
    sample_results[source] = result

    print(f"\n{source.upper()}")
    print("-" * 80)
    print("document_id:", sample["document_id"])
    print("source_id:", sample["source_id"])
    print("source_type:", sample["source_type"])
    print("content_origin:", sample["content_origin"])
    print("attribution:", sample["attribution"])
    print("rights:", sample["rights"])
    print("locator keys:", sorted(sample["locator"].keys()))
    print("lineage:", sample["lineage"])
    print("Missing required fields:", result["missing_required"])
    print("Validation:", "PASS" if result["pass"] else "FAIL")


contract_gate = all(
    result["pass"]
    for result in sample_results.values()
)

print("\nCOMMON DOCUMENT CONTRACT GATE")
print("=" * 80)
print(
    "In-memory sample validation:",
    "PASS" if contract_gate else "FAIL",
)

assert contract_gate, "Common Document Contract sample validation failed."

COMMON DOCUMENT CONTRACT V1 — IN-MEMORY VALIDATION

WEBSITE
--------------------------------------------------------------------------------
document_id: website:008E61CC
source_id: website:008E61CC
source_type: website
content_origin: supervised
attribution: {'author': 'سودابه فرخنده', 'knowledge_owner': 'Dr. Komeil Roudi', 'speaker': None}
rights: {'status': None, 'basis': None, 'rights_holder': None, 'rights_evidence_id': None, 'allowed_uses': None}
locator keys: ['canonical_url', 'url']
lineage: {'parent_id': None, 'raw_source_path': None, 'raw_source_hash': '2510dbf81e13f55647cb9e89d4e1f2ecb6dc6276696f8e7871e1132ea5c940bd'}
Missing required fields: []
Validation: PASS

BOOK
--------------------------------------------------------------------------------
document_id: book:adult-financial-literacy-course:fa-source:p000001
source_id: book:adult-financial-literacy-course:fa-source
source_type: book
content_origin: authored_work
attribution: {'author': 'Dr. Komeil Roudi', 'knowledge_ow

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>اعتبارسنجی Common Document Contract روی کل Corpus</h3>

<p>
نسخه اولیه Common Document Contract روی یک نمونه واقعی از هر سه Source با موفقیت Validation شد.
اکنون همان Mapping روی تمام واحدهای پذیرفته‌شده Website، Book و Transcript اجرا می‌شود تا مشخص شود Contract در کل Corpus نیز معتبر است.
</p>

<p>
این مرحله هنوز Artifact نهایی یا Schema روی Disk نمی‌نویسد.
هدف کشف Edge Caseها، Missingهای واقعی و ناسازگاری‌های احتمالی پیش از Freeze کردن Schema است.
</p>

<h4>بررسی‌های این مرحله</h4>

<ul>
<li>ساخت Common Document موقت برای تمام ۱۹۴۴۴ واحد</li>
<li>کنترل Required fieldها</li>
<li>کنترل یکتایی <code>document_id</code> در کل سه Source</li>
<li>کنترل تطابق <code>text_sha256</code> با متن واقعی</li>
<li>کنترل وجود Lineage hash</li>
<li>گزارش Nullableهای واقعی Attribution، Rights، Language و Title</li>
<li>گزارش توزیع Source و Content Origin</li>
</ul>

<h4>معیار PASS</h4>

<p>
تمام Common Documentها باید Required fieldهای ساختاری را داشته باشند،
<code>document_id</code> در کل Dataset یکتا باشد،
Hash متن معتبر باشد
و هیچ Lineage hash مفقودی وجود نداشته باشد.
مقادیر Nullable واقعی فقط گزارش می‌شوند و باعث ساخت Metadata حدسی نخواهند شد.
</p>

</div>

In [11]:
# Code Cell 11

def build_website_common(record):
    return build_website_common_sample(record)


def build_book_common(paragraph):
    return build_book_common_sample(paragraph)


def build_transcript_common(record):
    return build_transcript_common_sample(record)


website_common_documents = [
    build_website_common(record)
    for record in website_data
]

book_common_documents = [
    build_book_common(paragraph)
    for paragraph in book_paragraphs
]

transcript_common_documents = [
    build_transcript_common(record)
    for record in transcript_data
]

common_documents_candidate = (
    website_common_documents
    + book_common_documents
    + transcript_common_documents
)


def nested_get(record, path):
    value = record

    for part in path.split("."):
        if not isinstance(value, dict):
            return None
        value = value.get(part)

    return value


required_failures = []
text_hash_failures = []
lineage_hash_failures = []

for document in common_documents_candidate:
    validation = validate_common_sample(document)

    if not validation["pass"]:
        required_failures.append(
            {
                "document_id": document.get("document_id"),
                "missing_required": validation["missing_required"],
                "source_type_ok": validation["source_type_ok"],
                "identity_ok": validation["identity_ok"],
                "nested_ok": validation["nested_ok"],
            }
        )

    computed_text_hash = hashlib.sha256(
        document["text"].encode("utf-8")
    ).hexdigest()

    if computed_text_hash != document["text_sha256"]:
        text_hash_failures.append(document["document_id"])

    if not nonempty_text(
        document.get("lineage", {}).get("raw_source_hash")
    ):
        lineage_hash_failures.append(document["document_id"])


all_document_ids = [
    document["document_id"]
    for document in common_documents_candidate
]

duplicate_document_ids = (
    len(all_document_ids)
    - len(set(all_document_ids))
)


nullable_paths = [
    "title",
    "language",
    "attribution.author",
    "attribution.knowledge_owner",
    "attribution.speaker",
    "rights.status",
    "rights.basis",
    "rights.rights_holder",
    "rights.rights_evidence_id",
    "rights.allowed_uses",
]

nullable_profile = {}

for path in nullable_paths:
    missing = sum(
        nested_get(document, path) is None
        or nested_get(document, path) == ""
        for document in common_documents_candidate
    )
    nullable_profile[path] = missing


source_distribution = Counter(
    document["source_type"]
    for document in common_documents_candidate
)

content_origin_distribution = Counter(
    (
        document["source_type"],
        document["content_origin"],
    )
    for document in common_documents_candidate
)


print("FULL CORPUS COMMON DOCUMENT CONTRACT VALIDATION")
print("=" * 80)

print("\nDOCUMENT POPULATION")
print("-" * 80)
print("Website:", len(website_common_documents))
print("Book:", len(book_common_documents))
print("Transcript:", len(transcript_common_documents))
print("Total:", len(common_documents_candidate))

print("\nIDENTITY AND STRUCTURE")
print("-" * 80)
print("Duplicate document_id:", duplicate_document_ids)
print("Required/structure failures:", len(required_failures))
print("Text SHA-256 failures:", len(text_hash_failures))
print("Missing lineage raw_source_hash:", len(lineage_hash_failures))

print("\nSOURCE DISTRIBUTION")
print("-" * 80)
print(dict(source_distribution))

print("\nCONTENT ORIGIN DISTRIBUTION")
print("-" * 80)

for key, count in sorted(content_origin_distribution.items()):
    print(f"{key}: {count}")

print("\nNULLABLE FIELD PROFILE")
print("-" * 80)

for path, count in nullable_profile.items():
    print(f"{path}: {count}")

if required_failures:
    print("\nFirst required-field failures:")
    for failure in required_failures[:5]:
        print(failure)

if text_hash_failures:
    print("\nFirst text-hash failures:")
    for document_id in text_hash_failures[:5]:
        print(" -", document_id)

if lineage_hash_failures:
    print("\nFirst lineage-hash failures:")
    for document_id in lineage_hash_failures[:5]:
        print(" -", document_id)


full_contract_gate = (
    len(common_documents_candidate) == 19444
    and duplicate_document_ids == 0
    and len(required_failures) == 0
    and len(text_hash_failures) == 0
    and len(lineage_hash_failures) == 0
)

print("\nFULL CORPUS CONTRACT GATE")
print("=" * 80)
print(
    "Common Document population:",
    "PASS"
    if len(common_documents_candidate) == 19444
    else "FAIL",
)
print(
    "Global document_id uniqueness:",
    "PASS" if duplicate_document_ids == 0 else "FAIL",
)
print(
    "Required structure:",
    "PASS" if len(required_failures) == 0 else "FAIL",
)
print(
    "Text hash integrity:",
    "PASS" if len(text_hash_failures) == 0 else "FAIL",
)
print(
    "Lineage hash coverage:",
    "PASS" if len(lineage_hash_failures) == 0 else "FAIL",
)
print(
    "FULL CORPUS CONTRACT:",
    "PASS" if full_contract_gate else "FAIL",
)

assert len(common_documents_candidate) == 19444, (
    "Unexpected Common Document population."
)
assert duplicate_document_ids == 0, (
    "Duplicate document_id detected across sources."
)
assert len(required_failures) == 0, (
    "Common Document required-field validation failed."
)
assert len(text_hash_failures) == 0, (
    "Common Document text SHA-256 validation failed."
)
assert len(lineage_hash_failures) == 0, (
    "Common Document lineage hash coverage failed."
)

FULL CORPUS COMMON DOCUMENT CONTRACT VALIDATION

DOCUMENT POPULATION
--------------------------------------------------------------------------------
Website: 529
Book: 2689
Transcript: 16226
Total: 19444

IDENTITY AND STRUCTURE
--------------------------------------------------------------------------------
Duplicate document_id: 0
Required/structure failures: 0
Text SHA-256 failures: 0
Missing lineage raw_source_hash: 0

SOURCE DISTRIBUTION
--------------------------------------------------------------------------------
{'website': 529, 'book': 2689, 'transcript': 16226}

CONTENT ORIGIN DISTRIBUTION
--------------------------------------------------------------------------------
('book', 'authored_work'): 2689
('transcript', 'transcript'): 16226
('website', 'primary_author'): 76
('website', 'supervised'): 453

NULLABLE FIELD PROFILE
--------------------------------------------------------------------------------
title: 0
language: 16755
attribution.author: 16226
attribution.knowledge

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ممیزی حفظ اطلاعات Source-specific پیش از Freeze کردن Schema</h3>

<p>
Common Document Contract پایه روی کل Corpus معتبر است، اما Schema مشترک نباید هنگام تبدیل سه Source، Metadata مهمی را که برای Chunking، Citation، Attribution و Lineage لازم است حذف کند.
</p>

<p>
در بررسی اولیه Transcript مشخص شد بعضی مقادیر Speaker عمداً <code>null</code> هستند.
این وضعیت با سیاست Notebook 06 سازگار است، زیرا Speaker ناشناخته نباید حدس زده شود.
بنابراین در این مرحله «وجود ساختار و کلید» از «موجود بودن مقدار» جدا می‌شود.
</p>

<h4>اصل Validation</h4>

<ul>
<li>Metadata ساختاری Website باید برای همه رکوردها وجود داشته باشد.</li>
<li>فیلدهای ساختاری Paragraph کتاب باید برای همه Paragraphها وجود داشته باشند؛ مقدار Nullable مانند <code>heading_level</code> یا <code>style</code> مجاز است.</li>
<li>ساختارهای <code>speaker</code>، <code>timestamps</code>، <code>merge</code> و <code>processing_state</code> Transcript باید برای همه Segmentها وجود داشته باشند.</li>
<li>کلیدهای Speaker باید حفظ شوند، اما مقدار واقعی آن‌ها می‌تواند در موارد unresolved برابر <code>null</code> باشد.</li>
</ul>

<h4>معیار PASS</h4>

<p>
Schema ورودی هر سه Source باید Metadata لازم را از نظر ساختاری حفظ کرده باشد.
Nullable بودن مقدار واقعی Failure نیست؛ حذف کامل کلید یا ساختار مورد انتظار Failure محسوب می‌شود.
</p>

</div>

In [12]:
# Code Cell 12

print("SOURCE-SPECIFIC INFORMATION PRESERVATION AUDIT")
print("=" * 80)


def missing_key_count(records, key_path):
    parts = key_path.split(".")
    missing = 0

    for record in records:
        value = record
        exists = True

        for part in parts:
            if not isinstance(value, dict) or part not in value:
                exists = False
                break
            value = value[part]

        if not exists:
            missing += 1

    return missing


print("\nWEBSITE")
print("-" * 80)

website_author_id_key_missing = missing_key_count(
    website_data,
    "author_id",
)

website_metadata_key_missing = missing_key_count(
    website_data,
    "metadata",
)

website_metadata_keys = Counter()

for record in website_data:
    metadata = record.get("metadata", {})

    if isinstance(metadata, dict):
        for key in metadata:
            website_metadata_keys[key] += 1

print("Missing author_id key:", website_author_id_key_missing)
print("Missing metadata key:", website_metadata_key_missing)
print("Metadata key coverage:", dict(website_metadata_keys))


print("\nBOOK")
print("-" * 80)

book_structural_fields = (
    "heading_path",
    "heading_level",
    "is_heading",
    "style",
    "in_table",
    "paragraph_number",
)

book_structure_profile = {}

for field in book_structural_fields:
    key_missing = sum(
        field not in paragraph
        for paragraph in book_paragraphs
    )

    null_values = sum(
        paragraph.get(field) is None
        for paragraph in book_paragraphs
    )

    book_structure_profile[field] = {
        "key_missing": key_missing,
        "null_values": null_values,
    }

for field, profile in book_structure_profile.items():
    print(
        f"{field}: "
        f"key_missing={profile['key_missing']}, "
        f"null_values={profile['null_values']}"
    )


print("\nTRANSCRIPT")
print("-" * 80)

transcript_required_key_paths = (
    "speaker",
    "speaker.base",
    "speaker.base.status",
    "speaker.base.raw",
    "speaker.effective",
    "speaker.effective.label",
    "speaker.effective.canonical",
    "speaker.effective.basis",
    "speaker.human_attribution_overlay",
    "timestamps",
    "timestamps.policy",
    "timestamps.segment_start_time",
    "timestamps.segment_end_time",
    "timestamps.source_annotations",
    "merge",
    "merge.status",
    "merge.merged_from_utterance_ids",
    "processing_state",
)

transcript_key_profile = {}

for key_path in transcript_required_key_paths:
    transcript_key_profile[key_path] = missing_key_count(
        transcript_data,
        key_path,
    )

for key_path, count in transcript_key_profile.items():
    print(f"{key_path}: key_missing={count}")


effective_label_null = sum(
    record.get("speaker", {})
    .get("effective", {})
    .get("label") is None
    for record in transcript_data
)

effective_canonical_null = sum(
    record.get("speaker", {})
    .get("effective", {})
    .get("canonical") is None
    for record in transcript_data
)

base_raw_null = sum(
    record.get("speaker", {})
    .get("base", {})
    .get("raw") is None
    for record in transcript_data
)

print("\nNullable speaker values observed:")
print("effective.label null:", effective_label_null)
print("effective.canonical null:", effective_canonical_null)
print("base.raw null:", base_raw_null)


website_preservation_ok = (
    website_author_id_key_missing == 0
    and website_metadata_key_missing == 0
    and all(
        website_metadata_keys.get(key, 0) == len(website_data)
        for key in (
            "article_id",
            "view_count",
            "study_time_minutes",
            "tags",
        )
    )
)

book_preservation_ok = all(
    profile["key_missing"] == 0
    for profile in book_structure_profile.values()
)

transcript_preservation_ok = all(
    count == 0
    for count in transcript_key_profile.values()
)

print("\nPRESERVATION DESIGN GATE")
print("=" * 80)
print(
    "Website source metadata structure:",
    "PASS" if website_preservation_ok else "FAIL",
)
print(
    "Book structural metadata:",
    "PASS" if book_preservation_ok else "FAIL",
)
print(
    "Transcript attribution/state structure:",
    "PASS" if transcript_preservation_ok else "FAIL",
)

preservation_gate = (
    website_preservation_ok
    and book_preservation_ok
    and transcript_preservation_ok
)

print(
    "SOURCE-SPECIFIC PRESERVATION:",
    "PASS" if preservation_gate else "FAIL",
)

assert website_preservation_ok, (
    "Website source-specific metadata preservation check failed."
)
assert book_preservation_ok, (
    "Book structural metadata preservation check failed."
)
assert transcript_preservation_ok, (
    "Transcript source-specific metadata preservation check failed."
)

SOURCE-SPECIFIC INFORMATION PRESERVATION AUDIT

WEBSITE
--------------------------------------------------------------------------------
Missing author_id key: 0
Missing metadata key: 0
Metadata key coverage: {'article_id': 529, 'view_count': 529, 'study_time_minutes': 529, 'tags': 529}

BOOK
--------------------------------------------------------------------------------
heading_path: key_missing=0, null_values=0
heading_level: key_missing=0, null_values=2509
is_heading: key_missing=0, null_values=0
style: key_missing=0, null_values=1574
in_table: key_missing=0, null_values=0
paragraph_number: key_missing=0, null_values=0

TRANSCRIPT
--------------------------------------------------------------------------------
speaker: key_missing=0
speaker.base: key_missing=0
speaker.base.status: key_missing=0
speaker.base.raw: key_missing=0
speaker.effective: key_missing=0
speaker.effective.label: key_missing=0
speaker.effective.canonical: key_missing=0
speaker.effective.basis: key_missing=0
spea

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>تکمیل Common Document Contract و حفظ Metadata اختصاصی منبع</h3>

<p>
ممیزی Source-specific نشان داد هر سه منبع Metadata مهمی دارند که نباید هنگام یکپارچه‌سازی حذف شوند.
بنابراین نسخه نهایی Candidate Contract در حافظه با بخش <code>source_metadata</code> تکمیل می‌شود.
</p>

<p>
همچنین هیچ <code>content_origin</code> جدیدی برای Book یا Transcript ساخته نمی‌شود.
فقط Website دارای مقدار واقعی ثبت‌شده برای این فیلد است؛ در دو Source دیگر مقدار تا زمان وجود Evidence برابر <code>null</code> باقی می‌ماند.
</p>

<h4>اطلاعات حفظ‌شده</h4>

<ul>
<li>Website: شناسه نویسنده، Article metadata، Tagها و اطلاعات مشاهده/زمان مطالعه</li>
<li>Book: Work/Edition، Heading، Style، Table state و شماره Paragraph</li>
<li>Transcript: Speaker state، Timestamp policy، Merge state، Processing state و Source identity</li>
<li>Transcript: متن Source و Hash آن برای حفظ Lineage پیش از Normalizationهای بعدی</li>
</ul>

<h4>معیار PASS</h4>

<p>
نسخه تکمیل‌شده Contract باید برای تمام ۱۹۴۴۴ Common Document معتبر باشد،
شناسه تکراری نداشته باشد،
Hash متن و Raw lineage سالم بماند
و Metadata اختصاصی مورد انتظار هر Source بدون حذف ساختاری حفظ شود.
</p>

</div>

In [13]:
# Code Cell 13

from copy import deepcopy


common_document_contract_v1 = {
    "schema_version": "common_document_v1",
    "required_fields": [
        "schema_version",
        "document_id",
        "source_type",
        "source_id",
        "text",
        "text_sha256",
        "raw_content_hash",
        "title",
        "language",
        "content_origin",
        "attribution",
        "rights",
        "locator",
        "lineage",
        "source_metadata",
    ],
    "source_types": [
        "website",
        "book",
        "transcript",
    ],
}


def build_website_common_v1(record):
    return {
        "schema_version": "common_document_v1",
        "document_id": record["document_id"],
        "source_type": "website",
        "source_id": record["source_id"],
        "text": record["content"],
        "text_sha256": hashlib.sha256(
            record["content"].encode("utf-8")
        ).hexdigest(),
        "raw_content_hash": record["raw_content_hash"],
        "title": record.get("title"),
        "language": None,
        "content_origin": record.get("content_origin"),
        "attribution": {
            "author": record.get("author"),
            "author_id": record.get("author_id"),
            "knowledge_owner": record.get("knowledge_owner"),
            "speaker": None,
        },
        "rights": {
            "status": None,
            "basis": None,
            "rights_holder": None,
            "rights_evidence_id": None,
            "allowed_uses": None,
        },
        "locator": {
            "canonical_url": record.get("canonical_url"),
            "url": record.get("url"),
        },
        "lineage": {
            "parent_id": None,
            "raw_source_path": None,
            "raw_source_hash": record["raw_content_hash"],
        },
        "source_metadata": deepcopy(record.get("metadata", {})),
    }


def build_book_common_v1(paragraph):
    edition = edition_lookup[paragraph["edition_id"]]
    source_file = edition["source_file"]

    return {
        "schema_version": "common_document_v1",
        "document_id": paragraph["paragraph_id"],
        "source_type": "book",
        "source_id": paragraph["edition_id"],
        "text": paragraph["text"],
        "text_sha256": paragraph["text_sha256"],
        "raw_content_hash": source_file["file_sha256"],
        "title": edition.get("title"),
        "language": edition.get("language"),
        "content_origin": None,
        "attribution": {
            "author": edition.get("author"),
            "author_id": None,
            "knowledge_owner": edition.get("knowledge_owner"),
            "speaker": None,
        },
        "rights": {
            "status": edition.get("rights_status"),
            "basis": None,
            "rights_holder": edition.get("rights_holder"),
            "rights_evidence_id": edition.get("rights_evidence_id"),
            "allowed_uses": edition.get("allowed_uses"),
        },
        "locator": {
            "work_id": paragraph.get("work_id"),
            "edition_id": paragraph.get("edition_id"),
            "paragraph_id": paragraph.get("paragraph_id"),
            "paragraph_number": paragraph.get("paragraph_number"),
            "heading_path": deepcopy(paragraph.get("heading_path")),
        },
        "lineage": {
            "parent_id": paragraph.get("edition_id"),
            "raw_source_path": source_file.get("relative_path"),
            "raw_source_hash": source_file.get("file_sha256"),
        },
        "source_metadata": {
            "work_id": paragraph.get("work_id"),
            "edition_id": paragraph.get("edition_id"),
            "heading_level": paragraph.get("heading_level"),
            "heading_path": deepcopy(paragraph.get("heading_path")),
            "is_heading": paragraph.get("is_heading"),
            "style": paragraph.get("style"),
            "in_table": paragraph.get("in_table"),
            "paragraph_number": paragraph.get("paragraph_number"),
        },
    }


def build_transcript_common_v1(record):
    source_identity = record["source_identity"]
    source_locator = record["source_locator"]
    content = record["content"]
    rights = record["rights"]
    speaker = record["speaker"]

    return {
        "schema_version": "common_document_v1",
        "document_id": record["utterance_id"],
        "source_type": "transcript",
        "source_id": record["transcript_unit_id"],
        "text": content["text"],
        "text_sha256": content["text_sha256"],
        "raw_content_hash": source_identity["source_sha256"],
        "title": record.get("program"),
        "language": None,
        "content_origin": None,
        "attribution": {
            "author": None,
            "author_id": None,
            "knowledge_owner": None,
            "speaker": speaker.get("effective", {}).get("canonical"),
        },
        "rights": {
            "status": rights.get("status"),
            "basis": rights.get("basis"),
            "rights_holder": None,
            "rights_evidence_id": None,
            "allowed_uses": None,
        },
        "locator": {
            "locator_id": source_locator.get("locator_id"),
            "source_line_start": source_locator.get("source_line_start"),
            "source_line_end": source_locator.get("source_line_end"),
            "program_id": record.get("program_id"),
            "episode_number": source_identity.get("episode_number"),
            "season_number": source_identity.get("season_number"),
            "physical_file_id": source_identity.get("physical_file_id"),
        },
        "lineage": {
            "parent_id": record.get("transcript_unit_id"),
            "raw_source_path": source_identity.get("relative_path"),
            "raw_source_hash": source_identity.get("source_sha256"),
        },
        "source_metadata": {
            "program": record.get("program"),
            "program_id": record.get("program_id"),
            "record_type": record.get("record_type"),
            "record_version": record.get("record_version"),
            "transcript_unit_id": record.get("transcript_unit_id"),
            "utterance_index": record.get("utterance_index"),
            "source_identity": deepcopy(source_identity),
            "speaker": deepcopy(speaker),
            "timestamps": deepcopy(record.get("timestamps")),
            "merge": deepcopy(record.get("merge")),
            "processing_state": deepcopy(record.get("processing_state")),
            "source_text": content.get("source_text"),
            "source_text_sha256": content.get("source_text_sha256"),
            "text_state": content.get("text_state"),
        },
    }


website_common_documents_v1 = [
    build_website_common_v1(record)
    for record in website_data
]

book_common_documents_v1 = [
    build_book_common_v1(paragraph)
    for paragraph in book_paragraphs
]

transcript_common_documents_v1 = [
    build_transcript_common_v1(record)
    for record in transcript_data
]

common_documents_v1_candidate = (
    website_common_documents_v1
    + book_common_documents_v1
    + transcript_common_documents_v1
)


def validate_common_document_v1(document):
    missing_required = [
        field
        for field in common_document_contract_v1["required_fields"]
        if field not in document
    ]

    source_type_ok = (
        document.get("source_type")
        in common_document_contract_v1["source_types"]
    )

    identity_ok = all(
        nonempty_text(document.get(field))
        for field in (
            "document_id",
            "source_id",
            "text",
            "text_sha256",
            "raw_content_hash",
        )
    )

    nested_ok = all(
        isinstance(document.get(field), dict)
        for field in (
            "attribution",
            "rights",
            "locator",
            "lineage",
            "source_metadata",
        )
    )

    return (
        not missing_required
        and source_type_ok
        and identity_ok
        and nested_ok
    )


validation_failures = []
text_hash_failures = []
source_metadata_failures = []

for document in common_documents_v1_candidate:
    if not validate_common_document_v1(document):
        validation_failures.append(document.get("document_id"))

    calculated_hash = hashlib.sha256(
        document["text"].encode("utf-8")
    ).hexdigest()

    if calculated_hash != document["text_sha256"]:
        text_hash_failures.append(document["document_id"])

    source_type = document["source_type"]
    metadata = document["source_metadata"]

    if source_type == "website":
        required_metadata = {
            "article_id",
            "view_count",
            "study_time_minutes",
            "tags",
        }

    elif source_type == "book":
        required_metadata = {
            "work_id",
            "edition_id",
            "heading_level",
            "heading_path",
            "is_heading",
            "style",
            "in_table",
            "paragraph_number",
        }

    else:
        required_metadata = {
            "program",
            "program_id",
            "record_type",
            "record_version",
            "transcript_unit_id",
            "utterance_index",
            "source_identity",
            "speaker",
            "timestamps",
            "merge",
            "processing_state",
            "source_text",
            "source_text_sha256",
            "text_state",
        }

    if not required_metadata.issubset(metadata.keys()):
        source_metadata_failures.append(document["document_id"])


document_ids_v1 = [
    document["document_id"]
    for document in common_documents_v1_candidate
]

duplicate_document_ids_v1 = (
    len(document_ids_v1)
    - len(set(document_ids_v1))
)

content_origin_profile_v1 = Counter(
    (
        document["source_type"],
        document["content_origin"],
    )
    for document in common_documents_v1_candidate
)


print("ENRICHED COMMON DOCUMENT CONTRACT VALIDATION")
print("=" * 80)

print("Total documents:", len(common_documents_v1_candidate))
print("Duplicate document_id:", duplicate_document_ids_v1)
print("Contract validation failures:", len(validation_failures))
print("Text SHA-256 failures:", len(text_hash_failures))
print(
    "Source metadata preservation failures:",
    len(source_metadata_failures),
)

print("\nCONTENT ORIGIN — EVIDENCE-PRESERVING PROFILE")
print("-" * 80)

for key, count in sorted(
    content_origin_profile_v1.items(),
    key=lambda item: str(item[0]),
):
    print(f"{key}: {count}")


enriched_contract_gate = (
    len(common_documents_v1_candidate) == 19444
    and duplicate_document_ids_v1 == 0
    and len(validation_failures) == 0
    and len(text_hash_failures) == 0
    and len(source_metadata_failures) == 0
)

print("\nENRICHED CONTRACT GATE")
print("=" * 80)
print(
    "Population:",
    "PASS"
    if len(common_documents_v1_candidate) == 19444
    else "FAIL",
)
print(
    "Global ID uniqueness:",
    "PASS" if duplicate_document_ids_v1 == 0 else "FAIL",
)
print(
    "Contract validation:",
    "PASS" if len(validation_failures) == 0 else "FAIL",
)
print(
    "Text hash integrity:",
    "PASS" if len(text_hash_failures) == 0 else "FAIL",
)
print(
    "Source metadata preservation:",
    "PASS"
    if len(source_metadata_failures) == 0
    else "FAIL",
)
print(
    "ENRICHED COMMON DOCUMENT CONTRACT:",
    "PASS" if enriched_contract_gate else "FAIL",
)

assert len(common_documents_v1_candidate) == 19444
assert duplicate_document_ids_v1 == 0
assert len(validation_failures) == 0
assert len(text_hash_failures) == 0
assert len(source_metadata_failures) == 0

ENRICHED COMMON DOCUMENT CONTRACT VALIDATION
Total documents: 19444
Duplicate document_id: 0
Contract validation failures: 0
Text SHA-256 failures: 0
Source metadata preservation failures: 0

CONTENT ORIGIN — EVIDENCE-PRESERVING PROFILE
--------------------------------------------------------------------------------
('book', None): 2689
('transcript', None): 16226
('website', 'primary_author'): 76
('website', 'supervised'): 453

ENRICHED CONTRACT GATE
Population: PASS
Global ID uniqueness: PASS
Contract validation: PASS
Text hash integrity: PASS
Source metadata preservation: PASS
ENRICHED COMMON DOCUMENT CONTRACT: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>تعریف و اعتبارسنجی JSON Schema رسمی Common Document</h3>

<p>
Common Document Contract روی کل Corpus با موفقیت آزمایش شد.
اکنون همان Contract به یک JSON Schema رسمی و ماشین‌خوان تبدیل می‌شود تا Validation مراحل بعدی فقط به توابع داخل Notebook وابسته نباشد.
</p>

<p>
در این مرحله Schema فقط در حافظه ساخته می‌شود و هنوز هیچ فایل جدیدی روی Disk نوشته نمی‌شود.
Schema با استاندارد <code>JSON Schema Draft 2020-12</code> تعریف و روی تمام ۱۹۴۴۴ Common Document اجرا خواهد شد.
</p>

<h4>اصول Schema</h4>

<ul>
<li>فیلدهای اصلی هویتی، متن، Hash، Attribution، Rights، Locator، Lineage و Source Metadata اجباری هستند.</li>
<li>Nullable بودن واقعی Metadata حفظ می‌شود.</li>
<li><code>source_type</code> فقط یکی از <code>website</code>، <code>book</code> یا <code>transcript</code> است.</li>
<li><code>content_origin</code> فقط مقادیر مشاهده‌شده و مستند Website را می‌پذیرد و برای Book/Transcript می‌تواند <code>null</code> باشد.</li>
<li>برای هر Source، Locator و <code>source_metadata</code> مخصوص همان Source با شرط مستقل Validation می‌شود.</li>
<li>هیچ Rights، Speaker، Language یا Content Origin مفقودی ساخته نمی‌شود.</li>
</ul>

<h4>معیار PASS</h4>

<p>
خود JSON Schema باید از نظر استاندارد معتبر باشد و تمام ۱۹۴۴۴ Common Document فعلی باید بدون Validation Error آن را پاس کنند.
اگر حتی یک رکورد ناسازگار باشد، Schema هنوز Freeze نمی‌شود و همان مرحله اصلاح خواهد شد.
</p>

</div>

In [14]:
# Code Cell 14

import importlib.util


jsonschema_available = importlib.util.find_spec("jsonschema") is not None

print("JSON SCHEMA RUNTIME CHECK")
print("=" * 80)
print("jsonschema package available:", jsonschema_available)

assert jsonschema_available, (
    "The jsonschema package is required for formal schema validation."
)

from jsonschema import Draft202012Validator


document_schema_v1 = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$id": "https://rag-finance.local/schemas/phase2/document.schema.json",
    "title": "RAG Finance Phase 2 Common Document",
    "type": "object",
    "required": [
        "schema_version",
        "document_id",
        "source_type",
        "source_id",
        "text",
        "text_sha256",
        "raw_content_hash",
        "title",
        "language",
        "content_origin",
        "attribution",
        "rights",
        "locator",
        "lineage",
        "source_metadata",
    ],
    "properties": {
        "schema_version": {
            "const": "common_document_v1",
        },
        "document_id": {
            "type": "string",
            "minLength": 1,
        },
        "source_type": {
            "enum": [
                "website",
                "book",
                "transcript",
            ],
        },
        "source_id": {
            "type": "string",
            "minLength": 1,
        },
        "text": {
            "type": "string",
            "minLength": 1,
        },
        "text_sha256": {
            "type": "string",
            "pattern": "^[0-9a-f]{64}$",
        },
        "raw_content_hash": {
            "type": "string",
            "pattern": "^[0-9a-f]{64}$",
        },
        "title": {
            "type": ["string", "null"],
        },
        "language": {
            "type": ["string", "null"],
        },
        "content_origin": {
            "enum": [
                "primary_author",
                "supervised",
                None,
            ],
        },
        "attribution": {
            "type": "object",
            "required": [
                "author",
                "author_id",
                "knowledge_owner",
                "speaker",
            ],
            "properties": {
                "author": {
                    "type": ["string", "null"],
                },
                "author_id": {
                    "type": ["string", "null"],
                },
                "knowledge_owner": {
                    "type": ["string", "null"],
                },
                "speaker": {
                    "type": ["string", "null"],
                },
            },
            "additionalProperties": False,
        },
        "rights": {
            "type": "object",
            "required": [
                "status",
                "basis",
                "rights_holder",
                "rights_evidence_id",
                "allowed_uses",
            ],
            "properties": {
                "status": {
                    "type": ["string", "null"],
                },
                "basis": {
                    "type": ["string", "null"],
                },
                "rights_holder": {
                    "type": ["string", "null"],
                },
                "rights_evidence_id": {
                    "type": ["string", "null"],
                },
                "allowed_uses": {
                    "type": [
                        "string",
                        "array",
                        "object",
                        "null",
                    ],
                },
            },
            "additionalProperties": False,
        },
        "locator": {
            "type": "object",
        },
        "lineage": {
            "type": "object",
            "required": [
                "parent_id",
                "raw_source_path",
                "raw_source_hash",
            ],
            "properties": {
                "parent_id": {
                    "type": ["string", "null"],
                },
                "raw_source_path": {
                    "type": ["string", "null"],
                },
                "raw_source_hash": {
                    "type": "string",
                    "pattern": "^[0-9a-f]{64}$",
                },
            },
            "additionalProperties": False,
        },
        "source_metadata": {
            "type": "object",
        },
    },
    "allOf": [
        {
            "if": {
                "properties": {
                    "source_type": {
                        "const": "website",
                    }
                },
                "required": ["source_type"],
            },
            "then": {
                "properties": {
                    "locator": {
                        "type": "object",
                        "required": [
                            "canonical_url",
                            "url",
                        ],
                        "properties": {
                            "canonical_url": {
                                "type": ["string", "null"],
                            },
                            "url": {
                                "type": ["string", "null"],
                            },
                        },
                        "additionalProperties": False,
                    },
                    "source_metadata": {
                        "type": "object",
                        "required": [
                            "article_id",
                            "view_count",
                            "study_time_minutes",
                            "tags",
                        ],
                    },
                },
            },
        },
        {
            "if": {
                "properties": {
                    "source_type": {
                        "const": "book",
                    }
                },
                "required": ["source_type"],
            },
            "then": {
                "properties": {
                    "locator": {
                        "type": "object",
                        "required": [
                            "work_id",
                            "edition_id",
                            "paragraph_id",
                            "paragraph_number",
                            "heading_path",
                        ],
                        "properties": {
                            "work_id": {
                                "type": "string",
                                "minLength": 1,
                            },
                            "edition_id": {
                                "type": "string",
                                "minLength": 1,
                            },
                            "paragraph_id": {
                                "type": "string",
                                "minLength": 1,
                            },
                            "paragraph_number": {
                                "type": "integer",
                            },
                            "heading_path": {
                                "type": "array",
                            },
                        },
                        "additionalProperties": False,
                    },
                    "source_metadata": {
                        "type": "object",
                        "required": [
                            "work_id",
                            "edition_id",
                            "heading_level",
                            "heading_path",
                            "is_heading",
                            "style",
                            "in_table",
                            "paragraph_number",
                        ],
                    },
                },
            },
        },
        {
            "if": {
                "properties": {
                    "source_type": {
                        "const": "transcript",
                    }
                },
                "required": ["source_type"],
            },
            "then": {
                "properties": {
                    "locator": {
                        "type": "object",
                        "required": [
                            "locator_id",
                            "source_line_start",
                            "source_line_end",
                            "program_id",
                            "episode_number",
                            "season_number",
                            "physical_file_id",
                        ],
                        "properties": {
                            "locator_id": {
                                "type": "string",
                                "minLength": 1,
                            },
                            "source_line_start": {
                                "type": "integer",
                            },
                            "source_line_end": {
                                "type": "integer",
                            },
                            "program_id": {
                                "type": "string",
                                "minLength": 1,
                            },
                            "episode_number": {
                                "type": ["integer", "null"],
                            },
                            "season_number": {
                                "type": ["integer", "null"],
                            },
                            "physical_file_id": {
                                "type": "string",
                                "minLength": 1,
                            },
                        },
                        "additionalProperties": False,
                    },
                    "source_metadata": {
                        "type": "object",
                        "required": [
                            "program",
                            "program_id",
                            "record_type",
                            "record_version",
                            "transcript_unit_id",
                            "utterance_index",
                            "source_identity",
                            "speaker",
                            "timestamps",
                            "merge",
                            "processing_state",
                            "source_text",
                            "source_text_sha256",
                            "text_state",
                        ],
                    },
                },
            },
        },
    ],
    "additionalProperties": False,
}


Draft202012Validator.check_schema(document_schema_v1)

validator = Draft202012Validator(document_schema_v1)

schema_failures = []

for document in common_documents_v1_candidate:
    errors = sorted(
        validator.iter_errors(document),
        key=lambda error: list(error.absolute_path),
    )

    if errors:
        schema_failures.append(
            {
                "document_id": document.get("document_id"),
                "source_type": document.get("source_type"),
                "errors": [
                    {
                        "path": ".".join(
                            str(part)
                            for part in error.absolute_path
                        ),
                        "message": error.message,
                    }
                    for error in errors[:5]
                ],
            }
        )


print("\nFORMAL JSON SCHEMA VALIDATION")
print("=" * 80)
print("Schema draft: 2020-12")
print("Schema definition: PASS")
print("Documents validated:", len(common_documents_v1_candidate))
print("Documents with schema errors:", len(schema_failures))

if schema_failures:
    print("\nFirst schema failures:")
    for failure in schema_failures[:5]:
        print(json.dumps(
            failure,
            ensure_ascii=False,
            indent=2,
        ))


formal_schema_gate = (
    len(common_documents_v1_candidate) == 19444
    and len(schema_failures) == 0
)

print("\nFORMAL SCHEMA GATE")
print("=" * 80)
print(
    "JSON Schema definition:",
    "PASS",
)
print(
    "Full-corpus validation:",
    "PASS" if len(schema_failures) == 0 else "FAIL",
)
print(
    "DOCUMENT SCHEMA CANDIDATE:",
    "PASS" if formal_schema_gate else "FAIL",
)

assert formal_schema_gate, (
    "Formal Common Document JSON Schema validation failed."
)

JSON SCHEMA RUNTIME CHECK
jsonschema package available: True

FORMAL JSON SCHEMA VALIDATION
Schema draft: 2020-12
Schema definition: PASS
Documents validated: 19444
Documents with schema errors: 0

FORMAL SCHEMA GATE
JSON Schema definition: PASS
Full-corpus validation: PASS
DOCUMENT SCHEMA CANDIDATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ثبت Schema رسمی Common Document و بازاعتبارسنجی مستقل</h3>

<p>
JSON Schema رسمی Common Document در حافظه روی تمام ۱۹۴۴۴ رکورد با موفقیت Validation شد.
اکنون همان Schema به‌عنوان Artifact نسخه‌دار Notebook 07 روی Disk ثبت می‌شود.
</p>

<p>
برای جلوگیری از پذیرش فایلی که فقط در Memory معتبر بوده است، پس از نوشتن Schema فایل دوباره از Disk خوانده می‌شود، Hash آن محاسبه می‌شود و Validation کل Corpus با نسخه بازخوانی‌شده تکرار خواهد شد.
</p>

<h4>خروجی این مرحله</h4>

<p><code>schemas/phase2/document.schema.json</code></p>

<h4>محدوده تغییر</h4>

<ul>
<li>در صورت نیاز فقط پوشه <code>schemas/phase2/</code> ساخته می‌شود.</li>
<li>فقط فایل <code>document.schema.json</code> نوشته می‌شود.</li>
<li>هیچ Dataset، Report، Notebook دیگر یا فایل Git تغییر داده نمی‌شود.</li>
<li>هیچ Commit یا Push انجام نمی‌شود.</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>فایل Schema روی Disk وجود داشته باشد.</li>
<li>Schema بازخوانی‌شده از نظر Draft 2020-12 معتبر باشد.</li>
<li>تمام ۱۹۴۴۴ Common Document نسخه فعلی را بدون Error Validate کند.</li>
<li>SHA-256 فایل ثبت و گزارش شود.</li>
</ul>

</div>

In [15]:
# Code Cell 15

schema_output_path = (
    project_root
    / "schemas"
    / "phase2"
    / "document.schema.json"
)

schema_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

schema_json_text = json.dumps(
    document_schema_v1,
    ensure_ascii=False,
    indent=2,
    sort_keys=True,
) + "\n"

temporary_schema_path = schema_output_path.with_suffix(
    ".schema.json.tmp"
)

temporary_schema_path.write_text(
    schema_json_text,
    encoding="utf-8",
)

with temporary_schema_path.open(
    "r",
    encoding="utf-8",
) as handle:
    temporary_schema = json.load(handle)

Draft202012Validator.check_schema(
    temporary_schema
)

temporary_validator = Draft202012Validator(
    temporary_schema
)

temporary_failures = []

for document in common_documents_v1_candidate:
    errors = list(
        temporary_validator.iter_errors(document)
    )

    if errors:
        temporary_failures.append(
            document["document_id"]
        )

assert len(temporary_failures) == 0, (
    "Temporary schema failed full-corpus validation."
)

temporary_schema_path.replace(
    schema_output_path
)


with schema_output_path.open(
    "r",
    encoding="utf-8",
) as handle:
    persisted_schema = json.load(handle)

Draft202012Validator.check_schema(
    persisted_schema
)

persisted_validator = Draft202012Validator(
    persisted_schema
)

persisted_failures = []

for document in common_documents_v1_candidate:
    errors = list(
        persisted_validator.iter_errors(document)
    )

    if errors:
        persisted_failures.append(
            document["document_id"]
        )

persisted_schema_sha256 = sha256_file(
    schema_output_path
)

print("PERSISTED COMMON DOCUMENT SCHEMA")
print("=" * 80)
print(
    "Path:",
    schema_output_path.relative_to(project_root),
)
print(
    "Size bytes:",
    schema_output_path.stat().st_size,
)
print(
    "SHA-256:",
    persisted_schema_sha256,
)
print(
    "Schema reload:",
    "PASS",
)
print(
    "Draft 2020-12 validation:",
    "PASS",
)
print(
    "Documents independently validated:",
    len(common_documents_v1_candidate),
)
print(
    "Validation failures:",
    len(persisted_failures),
)

persisted_schema_gate = (
    schema_output_path.is_file()
    and len(persisted_failures) == 0
)

print("\nPERSISTED SCHEMA GATE")
print("=" * 80)
print(
    "document.schema.json:",
    "PASS" if persisted_schema_gate else "FAIL",
)

assert schema_output_path.is_file(), (
    "Persisted document schema is missing."
)

assert len(persisted_failures) == 0, (
    "Persisted document schema failed full-corpus validation."
)

PERSISTED COMMON DOCUMENT SCHEMA
Path: schemas/phase2/document.schema.json
Size bytes: 8004
SHA-256: a42f57aa53e3ee8c955232664ef49e58bc7abe65f816a51db6836ce62dba3d11
Schema reload: PASS
Draft 2020-12 validation: PASS
Documents independently validated: 19444
Validation failures: 0

PERSISTED SCHEMA GATE
document.schema.json: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بررسی Evidence انتساب Website پیش از ساخت Dataset یکپارچه</h3>

<p>
Schema رسمی Common Document ثبت شده است، اما پیش از تولید Dataset نهایی باید بررسی شود آیا اطلاعات انتساب Website فقط به فیلدهای موجود در <code>website_documents_v2.json</code> محدود است یا Evidence دقیق‌تری مانند Byline خام، نقش سردبیری یا منبع تصمیم Attribution در Artifactهای پذیرفته‌شده قبلی وجود دارد.
</p>

<p>
هدف این مرحله جلوگیری از از دست‌رفتن Attribution evidence و همچنین جلوگیری از ساخت Metadata حدسی است.
هیچ فایل موجود تغییر نمی‌کند.
</p>

<h4>منابع بررسی</h4>

<ul>
<li><code>data/phase2/manifests/website_manifest_v1.json</code></li>
<li>نمونه‌های واقعی <code>data/phase2/raw/website/&lt;article_id&gt;/source.json</code></li>
<li>رکوردهای پذیرفته‌شده <code>website_documents_v2.json</code></li>
</ul>

<h4>بررسی‌ها</h4>

<ul>
<li>فیلدهای Attribution موجود در Manifest</li>
<li>فیلدهای Attribution موجود در Raw source metadata</li>
<li>تطابق <code>article_id</code> میان Processed، Manifest و Raw</li>
<li>کشف فیلدهای احتمالی Byline، Author، Editor، Supervisor یا Attribution evidence بدون فرض نام آن‌ها</li>
</ul>

<h4>معیار PASS</h4>

<p>
برای نمونه‌های بررسی‌شده باید ارتباط Processed → Manifest → Raw بدون ابهام برقرار باشد و وضعیت واقعی Evidence انتساب مشخص شود.
نبود Evidence اضافی Failure نیست؛ اما در آن صورت Common Dataset نباید چنین Metadataای را جعل کند.
</p>

</div>

In [16]:
# Code Cell 16

website_manifest_path = (
    project_root
    / "data"
    / "phase2"
    / "manifests"
    / "website_manifest_v1.json"
)

with website_manifest_path.open(
    "r",
    encoding="utf-8",
) as handle:
    website_manifest_data = json.load(handle)


if isinstance(website_manifest_data, list):
    website_manifest_records = website_manifest_data
elif isinstance(website_manifest_data, dict):
    list_values = [
        value
        for value in website_manifest_data.values()
        if isinstance(value, list)
    ]

    assert len(list_values) == 1, (
        "Website manifest record collection could not be identified uniquely."
    )

    website_manifest_records = list_values[0]
else:
    raise TypeError(
        "Unexpected website manifest top-level structure."
    )


def flatten_key_paths(value, prefix="", max_depth=4):
    paths = set()

    if max_depth < 0:
        return paths

    if isinstance(value, dict):
        for key, child in value.items():
            current = f"{prefix}.{key}" if prefix else str(key)
            paths.add(current)

            paths.update(
                flatten_key_paths(
                    child,
                    current,
                    max_depth - 1,
                )
            )

    elif isinstance(value, list) and value:
        paths.update(
            flatten_key_paths(
                value[0],
                f"{prefix}[]",
                max_depth - 1,
            )
        )

    return paths


attribution_terms = (
    "author",
    "byline",
    "editor",
    "supervisor",
    "attribution",
    "credit",
    "owner",
    "origin",
)


def attribution_related_paths(record):
    return sorted(
        path
        for path in flatten_key_paths(record)
        if any(
            term in path.lower()
            for term in attribution_terms
        )
    )


manifest_by_article_id = {}

for record in website_manifest_records:
    article_id = (
        record.get("article_id")
        or record.get("metadata", {}).get("article_id")
    )

    if article_id is not None:
        manifest_by_article_id[str(article_id)] = record


sample_website_records = website_data[:5]

print("WEBSITE ATTRIBUTION EVIDENCE DISCOVERY")
print("=" * 80)

print("Manifest records:", len(website_manifest_records))
print("Processed records:", len(website_data))
print(
    "Manifest attribution-related paths:",
    attribution_related_paths(
        website_manifest_records[0]
    )
    if website_manifest_records
    else [],
)

sample_link_failures = []
raw_parse_failures = []

for index, processed_record in enumerate(
    sample_website_records,
    start=1,
):
    article_id = str(
        processed_record
        .get("metadata", {})
        .get("article_id")
    )

    manifest_record = manifest_by_article_id.get(
        article_id
    )

    raw_source_path = (
        project_root
        / "data"
        / "phase2"
        / "raw"
        / "website"
        / article_id
        / "source.json"
    )

    print(f"\nSAMPLE {index}")
    print("-" * 80)
    print("article_id:", article_id)
    print(
        "processed document_id:",
        processed_record.get("document_id"),
    )
    print(
        "processed author:",
        processed_record.get("author"),
    )
    print(
        "processed content_origin:",
        processed_record.get("content_origin"),
    )

    manifest_ok = manifest_record is not None
    raw_exists = raw_source_path.is_file()

    print(
        "Manifest link:",
        "PASS" if manifest_ok else "FAIL",
    )
    print(
        "Raw source.json exists:",
        "PASS" if raw_exists else "FAIL",
    )

    if not manifest_ok or not raw_exists:
        sample_link_failures.append(article_id)
        continue

    print(
        "Manifest attribution-related paths:",
        attribution_related_paths(
            manifest_record
        ),
    )

    try:
        with raw_source_path.open(
            "r",
            encoding="utf-8",
        ) as handle:
            raw_source_record = json.load(handle)

        print(
            "Raw attribution-related paths:",
            attribution_related_paths(
                raw_source_record
            ),
        )

    except Exception as exc:
        raw_parse_failures.append(article_id)
        print(
            "Raw source parse:",
            "FAIL",
            type(exc).__name__,
            str(exc),
        )


print("\nATTRIBUTION EVIDENCE DISCOVERY GATE")
print("=" * 80)
print(
    "Sample lineage links:",
    "PASS" if not sample_link_failures else "FAIL",
)
print(
    "Raw source parsing:",
    "PASS" if not raw_parse_failures else "FAIL",
)

website_attribution_discovery_gate = (
    len(sample_link_failures) == 0
    and len(raw_parse_failures) == 0
)

print(
    "WEBSITE ATTRIBUTION EVIDENCE DISCOVERY:",
    "PASS"
    if website_attribution_discovery_gate
    else "FAIL",
)

assert len(sample_link_failures) == 0, (
    "Processed → Manifest → Raw website lineage failed."
)

assert len(raw_parse_failures) == 0, (
    "One or more sampled website source.json files could not be parsed."
)

WEBSITE ATTRIBUTION EVIDENCE DISCOVERY
Manifest records: 529
Processed records: 529
Manifest attribution-related paths: ['author', 'content_origin', 'knowledge_owner']

SAMPLE 1
--------------------------------------------------------------------------------
article_id: 008E61CC
processed document_id: website:008E61CC
processed author: سودابه فرخنده
processed content_origin: supervised
Manifest link: PASS
Raw source.json exists: PASS
Manifest attribution-related paths: ['author', 'content_origin', 'knowledge_owner']
Raw attribution-related paths: []

SAMPLE 2
--------------------------------------------------------------------------------
article_id: 00D061A8
processed document_id: website:00D061A8
processed author: عاطفه جلادتی
processed content_origin: supervised
Manifest link: PASS
Raw source.json exists: PASS
Manifest attribution-related paths: ['author', 'content_origin', 'knowledge_owner']
Raw attribution-related paths: []

SAMPLE 3
-----------------------------------------------

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بازاعتبارسنجی کامل Attribution و Lineage وب‌سایت</h3>

<p>
بررسی اولیه نشان داد <code>website_manifest_v1.json</code> و Artifact پردازش‌شده پذیرفته‌شده از نظر فیلد <code>author</code> همگام نیستند.
Manifest مربوط به مرحله Discovery است و نباید به‌عنوان Source of Truth نهایی Attribution استفاده شود.
</p>

<p>
در این مرحله نقش هر Artifact جدا می‌شود:
Manifest برای شناسه و پوشش Source،
Raw برای Lineage،
و <code>website_documents_v2.json</code> به‌عنوان Artifact پذیرفته‌شده برای Attribution استفاده می‌شود.
</p>

<p>
نبود Byline، Editor یا Supervisor مستقل در Raw <code>source.json</code> به‌عنوان محدودیت داده حفظ می‌شود و هیچ Evidence جدیدی ساخته یا حدس زده نمی‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام ۵۲۹ Processed record به Manifest و Raw source متصل باشند.</li>
<li><code>document_id</code> و <code>article_id</code> بدون ابهام باقی بمانند.</li>
<li>Author در تمام Processed recordهای پذیرفته‌شده غیرخالی باشد.</li>
<li>توزیع <code>content_origin</code> دقیقاً با Acceptance Report رسمی منطبق باشد.</li>
<li>نبود Attribution evidence اضافی در Raw فقط گزارش شود و Failure محسوب نشود.</li>
</ul>

</div>

In [17]:
# Code Cell 17

processed_by_article_id = {}

for record in website_data:
    article_id = str(
        record.get("metadata", {}).get("article_id")
    )

    if article_id in processed_by_article_id:
        raise ValueError(
            f"Duplicate processed article_id: {article_id}"
        )

    processed_by_article_id[article_id] = record


processed_ids = set(processed_by_article_id)
manifest_ids = set(manifest_by_article_id)

missing_in_manifest = sorted(
    processed_ids - manifest_ids
)

missing_in_processed = sorted(
    manifest_ids - processed_ids
)

missing_raw_sources = []
raw_parse_failures = []
raw_attribution_paths = Counter()

for article_id in sorted(processed_ids):
    raw_source_path = (
        project_root
        / "data"
        / "phase2"
        / "raw"
        / "website"
        / article_id
        / "source.json"
    )

    if not raw_source_path.is_file():
        missing_raw_sources.append(article_id)
        continue

    try:
        with raw_source_path.open(
            "r",
            encoding="utf-8",
        ) as handle:
            raw_record = json.load(handle)

        for path in attribution_related_paths(raw_record):
            raw_attribution_paths[path] += 1

    except Exception as exc:
        raw_parse_failures.append(
            {
                "article_id": article_id,
                "error": f"{type(exc).__name__}: {exc}",
            }
        )


processed_missing_author = missing_count(
    website_data,
    lambda record: record.get("author"),
)

processed_missing_content_origin = missing_count(
    website_data,
    lambda record: record.get("content_origin"),
)

processed_missing_knowledge_owner = missing_count(
    website_data,
    lambda record: record.get("knowledge_owner"),
)

processed_origin_distribution = Counter(
    record.get("content_origin")
    for record in website_data
)

expected_attribution = (
    loaded_acceptance_reports["website"]
    .get("attribution", {})
)

expected_origin_distribution = {
    key: value
    for key, value in expected_attribution.items()
    if key in {
        "primary_author",
        "supervised",
    }
}

origin_distribution_ok = (
    dict(processed_origin_distribution)
    == expected_origin_distribution
)


manifest_author_profile = Counter(
    record.get("author")
    for record in website_manifest_records
)

manifest_processed_author_mismatch_count = sum(
    manifest_by_article_id[article_id].get("author")
    != processed_by_article_id[article_id].get("author")
    for article_id in sorted(processed_ids & manifest_ids)
)


print("FULL WEBSITE ATTRIBUTION AND LINEAGE REVALIDATION")
print("=" * 80)

print("\nPOPULATION AND LINEAGE")
print("-" * 80)
print("Processed records:", len(processed_by_article_id))
print("Manifest records:", len(manifest_by_article_id))
print("Missing in Manifest:", len(missing_in_manifest))
print("Missing in Processed:", len(missing_in_processed))
print("Missing source.json:", len(missing_raw_sources))
print("Raw parse failures:", len(raw_parse_failures))

print("\nACCEPTED PROCESSED ATTRIBUTION")
print("-" * 80)
print("Missing author:", processed_missing_author)
print("Missing content_origin:", processed_missing_content_origin)
print("Missing knowledge_owner:", processed_missing_knowledge_owner)
print(
    "Processed content_origin distribution:",
    dict(processed_origin_distribution),
)
print(
    "Acceptance expected distribution:",
    expected_origin_distribution,
)
print(
    "Content-origin comparison:",
    "PASS" if origin_distribution_ok else "FAIL",
)

print("\nLEGACY MANIFEST ATTRIBUTION STATE")
print("-" * 80)
print(
    "Manifest author null count:",
    manifest_author_profile.get(None, 0),
)
print(
    "Manifest ↔ Processed author mismatches:",
    manifest_processed_author_mismatch_count,
)
print(
    "Manifest author equality used as Gate:",
    "NO",
)

print("\nRAW ATTRIBUTION EVIDENCE")
print("-" * 80)

if raw_attribution_paths:
    for path, count in sorted(raw_attribution_paths.items()):
        print(f"{path}: {count}")
else:
    print(
        "No additional attribution-related paths found "
        "in raw source.json files."
    )

print(
    "Raw attribution evidence limitation:",
    "RECORDED"
    if not raw_attribution_paths
    else "ADDITIONAL_FIELDS_PRESENT",
)


population_ok = (
    len(processed_by_article_id)
    == len(manifest_by_article_id)
    and not missing_in_manifest
    and not missing_in_processed
)

raw_lineage_ok = (
    len(missing_raw_sources) == 0
    and len(raw_parse_failures) == 0
)

processed_attribution_ok = (
    processed_missing_author == 0
    and processed_missing_content_origin == 0
    and processed_missing_knowledge_owner == 0
    and origin_distribution_ok
)

website_full_attribution_gate = (
    population_ok
    and raw_lineage_ok
    and processed_attribution_ok
)

print("\nFULL WEBSITE ATTRIBUTION GATE")
print("=" * 80)
print(
    "Processed ↔ Manifest population:",
    "PASS" if population_ok else "FAIL",
)
print(
    "Raw lineage coverage:",
    "PASS" if raw_lineage_ok else "FAIL",
)
print(
    "Accepted processed attribution:",
    "PASS" if processed_attribution_ok else "FAIL",
)
print(
    "FULL WEBSITE ATTRIBUTION REVALIDATION:",
    "PASS"
    if website_full_attribution_gate
    else "FAIL",
)

assert population_ok, (
    "Website Processed ↔ Manifest population mismatch."
)

assert raw_lineage_ok, (
    "Website Raw source coverage or parsing failed."
)

assert processed_attribution_ok, (
    "Accepted Website processed attribution validation failed."
)

FULL WEBSITE ATTRIBUTION AND LINEAGE REVALIDATION

POPULATION AND LINEAGE
--------------------------------------------------------------------------------
Processed records: 529
Manifest records: 529
Missing in Manifest: 0
Missing in Processed: 0
Missing source.json: 0
Raw parse failures: 0

ACCEPTED PROCESSED ATTRIBUTION
--------------------------------------------------------------------------------
Missing author: 0
Missing content_origin: 0
Missing knowledge_owner: 0
Processed content_origin distribution: {'supervised': 453, 'primary_author': 76}
Acceptance expected distribution: {'primary_author': 76, 'supervised': 453}
Content-origin comparison: PASS

LEGACY MANIFEST ATTRIBUTION STATE
--------------------------------------------------------------------------------
Manifest author null count: 453
Manifest ↔ Processed author mismatches: 529
Manifest author equality used as Gate: NO

RAW ATTRIBUTION EVIDENCE
---------------------------------------------------------------------------

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بازاعتبارسنجی مستقل Raw Lineage برای سه منبع</h3>

<p>
Common Document باید فقط به یک Hash ذخیره‌شده اشاره نکند؛ مسیر Raw source نیز باید واقعاً روی Disk وجود داشته باشد و Hash فایل واقعی با Lineage ثبت‌شده منطبق باشد.
</p>

<p>
در این مرحله مسیرهای Raw برای Website، Book و Transcript از داده واقعی موجود استخراج می‌شوند و فایل‌های منبع دوباره از Disk Hash می‌شوند.
هیچ فایل یا Metadata تغییر نمی‌کند.
</p>

<h4>بررسی‌های این مرحله</h4>

<ul>
<li>کشف فایل Raw اصلی Website از داخل هر پوشه Article بدون حدس نام فایل</li>
<li>تطبیق SHA-256 فایل محتوایی Website با <code>raw_content_hash</code></li>
<li>تطبیق فایل DOCX هر Edition کتاب با <code>source_file.file_sha256</code></li>
<li>تطبیق فایل Markdown هر Transcript Unit با <code>source_identity.source_sha256</code></li>
<li>کنترل یکتایی مسیر Raw برای هر Source unit</li>
</ul>

<h4>معیار PASS</h4>

<p>
تمام واحدهای پذیرفته‌شده باید به یک فایل Raw موجود روی Disk متصل شوند و Broken hash برابر صفر باشد.
فقط بعد از این Gate مسیر Raw می‌تواند در Unified Lineage ثبت شود.
</p>

</div>

In [18]:
# Code Cell 18

website_raw_resolution = {}
website_raw_missing = []
website_raw_ambiguous = []
website_raw_hash_mismatches = []

for record in website_data:
    article_id = str(
        record.get("metadata", {}).get("article_id")
    )

    raw_dir = (
        project_root
        / "data"
        / "phase2"
        / "raw"
        / "website"
        / article_id
    )

    content_candidates = sorted(
        path
        for path in raw_dir.iterdir()
        if path.is_file()
        and path.name != "source.json"
    ) if raw_dir.is_dir() else []

    matching_candidates = [
        path
        for path in content_candidates
        if sha256_file(path) == record["raw_content_hash"]
    ]

    if not content_candidates:
        website_raw_missing.append(article_id)
        continue

    if len(matching_candidates) != 1:
        website_raw_ambiguous.append(
            {
                "article_id": article_id,
                "candidate_count": len(content_candidates),
                "hash_match_count": len(matching_candidates),
                "candidates": [
                    str(path.relative_to(project_root))
                    for path in content_candidates
                ],
            }
        )
        continue

    raw_path = matching_candidates[0]

    website_raw_resolution[
        record["document_id"]
    ] = raw_path.relative_to(project_root).as_posix()

    if sha256_file(raw_path) != record["raw_content_hash"]:
        website_raw_hash_mismatches.append(
            record["document_id"]
        )


book_raw_resolution = {}
book_raw_missing = []
book_raw_hash_mismatches = []

for edition in edition_records:
    source_file = edition["source_file"]

    raw_path = (
        project_root
        / source_file["relative_path"]
    )

    if not raw_path.is_file():
        book_raw_missing.append(
            edition["edition_id"]
        )
        continue

    actual_hash = sha256_file(raw_path)

    if actual_hash != source_file["file_sha256"]:
        book_raw_hash_mismatches.append(
            edition["edition_id"]
        )
        continue

    book_raw_resolution[
        edition["edition_id"]
    ] = raw_path.relative_to(project_root).as_posix()


transcript_raw_resolution = {}
transcript_raw_missing = []
transcript_raw_hash_mismatches = []
transcript_unit_source_conflicts = []

for record in transcript_data:
    unit_id = record["transcript_unit_id"]
    source_identity = record["source_identity"]

    raw_path = (
        project_root
        / source_identity["relative_path"]
    )

    expected_hash = source_identity["source_sha256"]

    if not raw_path.is_file():
        transcript_raw_missing.append(unit_id)
        continue

    actual_hash = sha256_file(raw_path)

    if actual_hash != expected_hash:
        transcript_raw_hash_mismatches.append(
            unit_id
        )
        continue

    resolved_path = (
        raw_path.relative_to(project_root).as_posix()
    )

    previous = transcript_raw_resolution.get(unit_id)

    if previous is not None and previous != resolved_path:
        transcript_unit_source_conflicts.append(
            {
                "transcript_unit_id": unit_id,
                "first_path": previous,
                "new_path": resolved_path,
            }
        )
    else:
        transcript_raw_resolution[
            unit_id
        ] = resolved_path


website_lineage_ok = (
    len(website_raw_resolution) == len(website_data)
    and len(website_raw_missing) == 0
    and len(website_raw_ambiguous) == 0
    and len(website_raw_hash_mismatches) == 0
)

book_lineage_ok = (
    len(book_raw_resolution) == len(edition_records)
    and len(book_raw_missing) == 0
    and len(book_raw_hash_mismatches) == 0
)

transcript_lineage_ok = (
    len(transcript_raw_resolution)
    == len(set(transcript_unit_ids))
    and len(transcript_raw_missing) == 0
    and len(transcript_raw_hash_mismatches) == 0
    and len(transcript_unit_source_conflicts) == 0
)


print("INDEPENDENT RAW LINEAGE REVALIDATION")
print("=" * 80)

print("\nWEBSITE")
print("-" * 80)
print(
    "Resolved raw sources:",
    len(website_raw_resolution),
)
print(
    "Missing raw content files:",
    len(website_raw_missing),
)
print(
    "Ambiguous raw resolutions:",
    len(website_raw_ambiguous),
)
print(
    "Raw hash mismatches:",
    len(website_raw_hash_mismatches),
)

if website_raw_resolution:
    first_document_id = next(
        iter(website_raw_resolution)
    )
    print(
        "Example:",
        first_document_id,
        "→",
        website_raw_resolution[first_document_id],
    )

if website_raw_ambiguous:
    print("\nFirst Website ambiguities:")
    for item in website_raw_ambiguous[:3]:
        print(item)


print("\nBOOK")
print("-" * 80)
print(
    "Resolved Edition raw sources:",
    len(book_raw_resolution),
)
print(
    "Missing raw files:",
    len(book_raw_missing),
)
print(
    "Raw hash mismatches:",
    len(book_raw_hash_mismatches),
)

for edition_id, path in book_raw_resolution.items():
    print(" -", edition_id, "→", path)


print("\nTRANSCRIPT")
print("-" * 80)
print(
    "Resolved Transcript Unit raw sources:",
    len(transcript_raw_resolution),
)
print(
    "Missing raw files:",
    len(set(transcript_raw_missing)),
)
print(
    "Raw hash mismatches:",
    len(set(transcript_raw_hash_mismatches)),
)
print(
    "Unit/source path conflicts:",
    len(transcript_unit_source_conflicts),
)

for unit_id in sorted(
    transcript_raw_resolution
)[:5]:
    print(
        " -",
        unit_id,
        "→",
        transcript_raw_resolution[unit_id],
    )


raw_lineage_gate = (
    website_lineage_ok
    and book_lineage_ok
    and transcript_lineage_ok
)

print("\nRAW LINEAGE GATE")
print("=" * 80)
print(
    "Website raw lineage:",
    "PASS" if website_lineage_ok else "FAIL",
)
print(
    "Book raw lineage:",
    "PASS" if book_lineage_ok else "FAIL",
)
print(
    "Transcript raw lineage:",
    "PASS" if transcript_lineage_ok else "FAIL",
)
print(
    "INDEPENDENT RAW LINEAGE:",
    "PASS" if raw_lineage_gate else "FAIL",
)

assert website_lineage_ok, (
    "Website raw lineage validation failed."
)
assert book_lineage_ok, (
    "Book raw lineage validation failed."
)
assert transcript_lineage_ok, (
    "Transcript raw lineage validation failed."
)

INDEPENDENT RAW LINEAGE REVALIDATION

WEBSITE
--------------------------------------------------------------------------------
Resolved raw sources: 529
Missing raw content files: 0
Ambiguous raw resolutions: 0
Raw hash mismatches: 0
Example: website:008E61CC → data/phase2/raw/website/008E61CC/article.html

BOOK
--------------------------------------------------------------------------------
Resolved Edition raw sources: 3
Missing raw files: 0
Raw hash mismatches: 0
 - book:adult-financial-literacy-course:fa-source → data/books/درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx
 - book:financial-literacy-for-all:fa-source → data/books/سواد مالی برای همه.docx
 - book:trainer-training-financial-literacy:fa-source → data/books/نوشتار آموزشی تربیت مربی سواد مالی.docx

TRANSCRIPT
--------------------------------------------------------------------------------
Resolved Transcript Unit raw sources: 183
Missing raw files: 0
Raw hash mismatches: 0
Unit/source path conflicts: 0
 - program:asre-

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بررسی نقش Supplemental Content کتاب در Unified Dataset</h3>

<p>
Artifact پذیرفته‌شده Book شامل ۲۸ رکورد Supplemental Content است، اما بررسی اولیه نشان داد الزام داشتن متن و Hash معتبر برای تمام انواع Supplemental Content فرض درستی نیست.
</p>

<p>
در این مرحله Footnote، Chart و Diagram از یکدیگر تفکیک می‌شوند و وضعیت واقعی Text و Hash آن‌ها بررسی می‌شود.
هدف این است که مشخص شود کدام رکوردها محتوای متنی قابل تبدیل به Common Document هستند و کدام رکوردها فقط Asset یا Structural Evidence هستند.
</p>

<p>
هیچ Supplemental Content در این مرحله حذف، تبدیل یا وارد Unified Dataset نمی‌شود و هیچ مقدار مفقودی ساخته نمی‌شود.
</p>

<h4>بررسی‌ها</h4>

<ul>
<li>توزیع نوع Supplemental Content</li>
<li>یکتایی Content ID</li>
<li>صحت Parent / Edition / Work Lineage</li>
<li>وضعیت Text و Text SHA-256 به تفکیک Content Type</li>
<li>بررسی Fieldهای کامل رکوردهای بدون Text معتبر</li>
<li>بررسی Locator و Asset reference آن‌ها</li>
</ul>

<h4>معیار PASS</h4>

<p>
تمام ۲۸ رکورد باید Lineage معتبر داشته باشند.
رکوردهای دارای Text باید Hash صحیح داشته باشند.
رکورد فاقد Text معتبر فقط گزارش می‌شود و Failure محسوب نمی‌شود؛ درباره ورود آن به Common Document پس از مشاهده ساختار واقعی تصمیم گرفته خواهد شد.
</p>

</div>

In [19]:
# Code Cell 19

supplemental_content = book_data.get(
    "supplemental_content",
    [],
)

supplemental_ids = [
    record.get("content_id")
    for record in supplemental_content
]

supplemental_types = Counter(
    record.get("content_type")
    for record in supplemental_content
)

paragraph_lookup = {
    paragraph["paragraph_id"]: paragraph
    for paragraph in book_paragraphs
}

edition_id_set = {
    edition["edition_id"]
    for edition in edition_records
}

work_id_set = {
    edition["work_id"]
    for edition in edition_records
}

supplemental_lineage_failures = []

text_state_by_type = Counter()
text_hash_failures = []
records_without_usable_text = []

for record in supplemental_content:
    content_id = record.get("content_id")
    content_type = record.get("content_type")
    parent_paragraph_id = record.get(
        "parent_paragraph_id"
    )
    edition_id = record.get("edition_id")
    work_id = record.get("work_id")

    parent = paragraph_lookup.get(
        parent_paragraph_id
    )

    lineage_ok = (
        parent is not None
        and parent.get("edition_id") == edition_id
        and parent.get("work_id") == work_id
        and edition_id in edition_id_set
        and work_id in work_id_set
    )

    if not lineage_ok:
        supplemental_lineage_failures.append(
            content_id
        )

    text = record.get("text")
    text_sha256 = record.get("text_sha256")

    has_text = (
        isinstance(text, str)
        and len(text) > 0
    )

    has_hash = (
        isinstance(text_sha256, str)
        and len(text_sha256) > 0
    )

    if has_text:
        calculated_hash = hashlib.sha256(
            text.encode("utf-8")
        ).hexdigest()

        hash_matches = (
            has_hash
            and calculated_hash == text_sha256
        )

        text_state_by_type[
            (
                content_type,
                "TEXT_HASH_PASS"
                if hash_matches
                else "TEXT_HASH_FAIL",
            )
        ] += 1

        if not hash_matches:
            text_hash_failures.append(
                {
                    "content_id": content_id,
                    "content_type": content_type,
                    "text_sha256": text_sha256,
                    "calculated_sha256": calculated_hash,
                }
            )

    else:
        text_state_by_type[
            (
                content_type,
                "NO_USABLE_TEXT",
            )
        ] += 1

        records_without_usable_text.append(
            record
        )


duplicate_supplemental_ids = duplicate_count(
    supplemental_ids
)

locator_key_profile = Counter()

for record in supplemental_content:
    locator = record.get(
        "source_locator",
        {},
    )

    if isinstance(locator, dict):
        for key in locator:
            locator_key_profile[
                (
                    record.get("content_type"),
                    key,
                )
            ] += 1


print("BOOK SUPPLEMENTAL CONTENT DISCOVERY")
print("=" * 80)

print("\nPOPULATION")
print("-" * 80)
print(
    "Supplemental records:",
    len(supplemental_content),
)
print(
    "Duplicate content_id:",
    duplicate_supplemental_ids,
)
print(
    "content_type distribution:",
    dict(supplemental_types),
)

print("\nLINEAGE")
print("-" * 80)
print(
    "Parent/Edition/Work lineage failures:",
    len(supplemental_lineage_failures),
)

print("\nTEXT STATE BY CONTENT TYPE")
print("-" * 80)

for key, count in sorted(
    text_state_by_type.items()
):
    print(f"{key}: {count}")

print(
    "Text-bearing records with hash failures:",
    len(text_hash_failures),
)
print(
    "Records without usable text:",
    len(records_without_usable_text),
)

if text_hash_failures:
    print("\nTEXT HASH FAILURE DETAILS")
    print("-" * 80)

    for item in text_hash_failures[:10]:
        print(
            json.dumps(
                item,
                ensure_ascii=False,
                indent=2,
            )
        )


print("\nLOCATOR COVERAGE BY TYPE")
print("-" * 80)

for key, count in sorted(
    locator_key_profile.items()
):
    print(f"{key}: {count}")


print("\nNON-TEXT RECORD STRUCTURE")
print("-" * 80)

non_text_key_profiles = Counter()

for record in records_without_usable_text:
    for key in record.keys():
        non_text_key_profiles[
            (
                record.get("content_type"),
                key,
            )
        ] += 1

for key, count in sorted(
    non_text_key_profiles.items()
):
    print(f"{key}: {count}")


print("\nNON-TEXT SAMPLE RECORDS")
print("-" * 80)

for record in records_without_usable_text[:5]:
    print(
        json.dumps(
            record,
            ensure_ascii=False,
            indent=2,
        )
    )


supplemental_gate = (
    len(supplemental_content)
    == book_data.get(
        "supplemental_content_count"
    )
    and duplicate_supplemental_ids == 0
    and len(supplemental_lineage_failures) == 0
    and len(text_hash_failures) == 0
)

print("\nSUPPLEMENTAL CONTENT DISCOVERY GATE")
print("=" * 80)
print(
    "Accepted population:",
    "PASS"
    if len(supplemental_content)
    == book_data.get("supplemental_content_count")
    else "FAIL",
)
print(
    "ID uniqueness:",
    "PASS"
    if duplicate_supplemental_ids == 0
    else "FAIL",
)
print(
    "Parent lineage:",
    "PASS"
    if len(supplemental_lineage_failures) == 0
    else "FAIL",
)
print(
    "Hash integrity for text-bearing records:",
    "PASS"
    if len(text_hash_failures) == 0
    else "FAIL",
)
print(
    "BOOK SUPPLEMENTAL DISCOVERY:",
    "PASS" if supplemental_gate else "FAIL",
)

assert len(supplemental_content) == book_data.get(
    "supplemental_content_count"
)

assert duplicate_supplemental_ids == 0

assert len(supplemental_lineage_failures) == 0

assert len(text_hash_failures) == 0

BOOK SUPPLEMENTAL CONTENT DISCOVERY

POPULATION
--------------------------------------------------------------------------------
Supplemental records: 28
Duplicate content_id: 0
content_type distribution: {'FOOTNOTE': 12, 'CHART': 15, 'DIAGRAM': 1}

LINEAGE
--------------------------------------------------------------------------------
Parent/Edition/Work lineage failures: 0

TEXT STATE BY CONTENT TYPE
--------------------------------------------------------------------------------
('CHART', 'NO_USABLE_TEXT'): 15
('DIAGRAM', 'NO_USABLE_TEXT'): 1
('FOOTNOTE', 'TEXT_HASH_PASS'): 12
Text-bearing records with hash failures: 0
Records without usable text: 16

LOCATOR COVERAGE BY TYPE
--------------------------------------------------------------------------------
('CHART', 'package_part'): 15
('CHART', 'relationship_id'): 15
('DIAGRAM', 'package_part'): 1
('FOOTNOTE', 'note_number'): 12

NON-TEXT RECORD STRUCTURE
-----------------------------------------------------------------------------

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>طبقه‌بندی Supplemental Content به Document متنی و Structured Asset</h3>

<p>
بررسی قبلی نشان داد Supplemental Content کتاب یک ساختار یکنواخت ندارد.
Footnoteها دارای متن و Hash متنی معتبر هستند، در حالی که Chart و Diagram از Payload ساختاری استفاده می‌کنند.
</p>

<p>
در این مرحله هیچ Dataset نهایی ساخته نمی‌شود.
هدف فقط تعیین Representation مناسب برای هر رکورد بر اساس ساختار واقعی آن است.
</p>

<h4>Candidateهای مورد بررسی</h4>

<ul>
<li><strong>Text Document Candidate:</strong> رکوردی که دارای متن غیرخالی و <code>text_sha256</code> معتبر باشد.</li>
<li><strong>Structured Asset Candidate:</strong> رکوردی که متن Document نداشته باشد، اما Payload ساختاری، Hash، Locator و Parent Lineage معتبر داشته باشد.</li>
</ul>

<p>
Chart و Diagram در این مرحله به متن مصنوعی تبدیل نمی‌شوند.
مقادیر Chart، Formulaها یا Diagram Text فقط به‌عنوان Evidence ساختاری بررسی می‌شوند.
</p>

<h4>بررسی‌های مهم</h4>

<ul>
<li>تعداد Text Document Candidateها</li>
<li>تعداد Structured Asset Candidateها</li>
<li>ساختار واقعی Diagram</li>
<li>سلامت Payload Hash از نظر قالب</li>
<li>عدم برخورد <code>content_id</code> Footnoteها با <code>document_id</code>های موجود</li>
<li>محاسبه جمع Candidate اسناد در صورت اضافه شدن فقط Supplementalهای متنی</li>
</ul>

<h4>معیار PASS</h4>

<p>
تمام ۲۸ Supplemental Content باید بدون هم‌پوشانی در یکی از دو گروه قابل طبقه‌بندی باشند،
شناسه Text Candidateها با Common Documentهای فعلی برخورد نداشته باشد
و هیچ رکورد بدون Representation باقی نماند.
</p>

</div>

In [20]:
# Code Cell 20

import re


sha256_pattern = re.compile(
    r"^[0-9a-f]{64}$"
)

text_document_candidates = []
structured_asset_candidates = []
unclassified_supplemental = []

for record in supplemental_content:
    text = record.get("text")
    text_sha256 = record.get("text_sha256")
    content_type = record.get("content_type")

    has_valid_text_document = (
        isinstance(text, str)
        and len(text) > 0
        and isinstance(text_sha256, str)
        and hashlib.sha256(
            text.encode("utf-8")
        ).hexdigest() == text_sha256
    )

    payload_sha256 = record.get(
        "payload_sha256"
    )

    has_valid_payload_hash_format = (
        isinstance(payload_sha256, str)
        and sha256_pattern.fullmatch(
            payload_sha256
        ) is not None
    )

    has_structured_payload = False

    if content_type == "CHART":
        has_structured_payload = (
            isinstance(
                record.get("chart_values"),
                list,
            )
            and isinstance(
                record.get("chart_formulas"),
                list,
            )
        )

    elif content_type == "DIAGRAM":
        has_structured_payload = (
            "diagram_text" in record
        )

    if has_valid_text_document:
        text_document_candidates.append(
            record
        )

    elif (
        content_type in {"CHART", "DIAGRAM"}
        and has_valid_payload_hash_format
        and has_structured_payload
        and isinstance(
            record.get("source_locator"),
            dict,
        )
        and record.get("parent_paragraph_id")
        in paragraph_lookup
    ):
        structured_asset_candidates.append(
            record
        )

    else:
        unclassified_supplemental.append(
            record
        )


current_common_ids = {
    document["document_id"]
    for document in common_documents_v1_candidate
}

supplemental_text_ids = {
    record["content_id"]
    for record in text_document_candidates
}

text_id_collisions = sorted(
    current_common_ids
    & supplemental_text_ids
)


chart_candidates = [
    record
    for record in structured_asset_candidates
    if record.get("content_type") == "CHART"
]

diagram_candidates = [
    record
    for record in structured_asset_candidates
    if record.get("content_type") == "DIAGRAM"
]


chart_value_lengths = [
    len(record.get("chart_values", []))
    for record in chart_candidates
]

chart_formula_lengths = [
    len(record.get("chart_formulas", []))
    for record in chart_candidates
]


print("SUPPLEMENTAL REPRESENTATION CLASSIFICATION")
print("=" * 80)

print("\nCLASSIFICATION")
print("-" * 80)
print(
    "Total supplemental records:",
    len(supplemental_content),
)
print(
    "Text Document Candidates:",
    len(text_document_candidates),
)
print(
    "Structured Asset Candidates:",
    len(structured_asset_candidates),
)
print(
    "Unclassified records:",
    len(unclassified_supplemental),
)

print("\nTEXT DOCUMENT CANDIDATES")
print("-" * 80)
print(
    "Type distribution:",
    dict(
        Counter(
            record.get("content_type")
            for record
            in text_document_candidates
        )
    ),
)
print(
    "document_id collisions with current Common Documents:",
    len(text_id_collisions),
)

for record in text_document_candidates[:5]:
    print(
        " -",
        record["content_id"],
        "→ parent:",
        record["parent_paragraph_id"],
    )


print("\nSTRUCTURED ASSET CANDIDATES")
print("-" * 80)
print(
    "Type distribution:",
    dict(
        Counter(
            record.get("content_type")
            for record
            in structured_asset_candidates
        )
    ),
)

print(
    "Chart value counts:",
    {
        "min": min(chart_value_lengths)
        if chart_value_lengths else None,
        "max": max(chart_value_lengths)
        if chart_value_lengths else None,
    },
)

print(
    "Chart formula counts:",
    {
        "min": min(chart_formula_lengths)
        if chart_formula_lengths else None,
        "max": max(chart_formula_lengths)
        if chart_formula_lengths else None,
    },
)


print("\nDIAGRAM STRUCTURE")
print("-" * 80)

for record in diagram_candidates:
    diagram_text = record.get(
        "diagram_text"
    )

    print(
        "content_id:",
        record.get("content_id"),
    )
    print(
        "diagram_text type:",
        type(diagram_text).__name__,
    )

    if isinstance(diagram_text, str):
        print(
            "diagram_text length:",
            len(diagram_text),
        )
        print(
            "diagram_text preview:",
            diagram_text[:300],
        )

    elif isinstance(diagram_text, list):
        print(
            "diagram_text item count:",
            len(diagram_text),
        )
        print(
            "diagram_text preview:",
            diagram_text[:10],
        )

    else:
        print(
            "diagram_text value:",
            diagram_text,
        )

    print(
        "source_locator:",
        record.get("source_locator"),
    )
    print(
        "payload_sha256:",
        record.get("payload_sha256"),
    )


candidate_document_population = (
    len(common_documents_v1_candidate)
    + len(text_document_candidates)
)

print("\nPOPULATION IMPACT — DISCOVERY ONLY")
print("-" * 80)
print(
    "Current Common Document Candidate:",
    len(common_documents_v1_candidate),
)
print(
    "Additional text Supplemental Candidates:",
    len(text_document_candidates),
)
print(
    "Candidate population if text supplements are included:",
    candidate_document_population,
)
print(
    "Structured assets remain separate:",
    len(structured_asset_candidates),
)


classification_gate = (
    len(text_document_candidates)
    + len(structured_asset_candidates)
    == len(supplemental_content)
    and len(unclassified_supplemental) == 0
    and len(text_id_collisions) == 0
)

print("\nSUPPLEMENTAL REPRESENTATION GATE")
print("=" * 80)
print(
    "Complete classification:",
    "PASS"
    if (
        len(text_document_candidates)
        + len(structured_asset_candidates)
        == len(supplemental_content)
    )
    else "FAIL",
)
print(
    "Unclassified records:",
    "PASS"
    if len(unclassified_supplemental) == 0
    else "FAIL",
)
print(
    "Text document ID collisions:",
    "PASS"
    if len(text_id_collisions) == 0
    else "FAIL",
)
print(
    "SUPPLEMENTAL REPRESENTATION:",
    "PASS" if classification_gate else "FAIL",
)

if unclassified_supplemental:
    print("\nUNCLASSIFIED RECORDS")
    for record in unclassified_supplemental:
        print(
            json.dumps(
                record,
                ensure_ascii=False,
                indent=2,
            )
        )

assert (
    len(text_document_candidates)
    + len(structured_asset_candidates)
    == len(supplemental_content)
)

assert len(unclassified_supplemental) == 0

assert len(text_id_collisions) == 0

SUPPLEMENTAL REPRESENTATION CLASSIFICATION

CLASSIFICATION
--------------------------------------------------------------------------------
Total supplemental records: 28
Text Document Candidates: 12
Structured Asset Candidates: 16
Unclassified records: 0

TEXT DOCUMENT CANDIDATES
--------------------------------------------------------------------------------
Type distribution: {'FOOTNOTE': 12}
document_id collisions with current Common Documents: 0
 - book:financial-literacy-for-all:fa-source:fn0001 → parent: book:financial-literacy-for-all:fa-source:p000019
 - book:financial-literacy-for-all:fa-source:fn0002 → parent: book:financial-literacy-for-all:fa-source:p000084
 - book:trainer-training-financial-literacy:fa-source:fn0001 → parent: book:trainer-training-financial-literacy:fa-source:p000231
 - book:trainer-training-financial-literacy:fa-source:fn0002 → parent: book:trainer-training-financial-literacy:fa-source:p000232
 - book:trainer-training-financial-literacy:fa-source:fn0003 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>افزودن Footnoteهای متنی به Common Document Contract</h3>

<p>
طبقه‌بندی Supplemental Content نشان داد ۱۲ Footnote دارای متن واقعی و Hash معتبر هستند و بنابراین باید در Unified Text Corpus به‌عنوان Common Document مستقل حضور داشته باشند.
</p>

<p>
در مقابل، ۱۵ Chart و ۱ Diagram فاقد representation متنی پذیرفته‌شده هستند و در این Notebook به متن مصنوعی تبدیل نمی‌شوند.
این ۱۶ رکورد به‌عنوان Structured Asset در Artifact اصلی Book و گزارش Lineage حفظ می‌شوند، اما وارد <code>documents_unified_v1.jsonl</code> متنی نخواهند شد.
</p>

<p>
برای جلوگیری از ابهام میان Paragraph و Footnote، Book Common Documents یک Metadata صریح به نام <code>book_unit_type</code> خواهند داشت.
مقدار آن برای Paragraph برابر <code>paragraph</code> و برای Footnote برابر <code>supplemental_text</code> است.
</p>

<h4>این مرحله</h4>

<ul>
<li>۱۲ Footnote به‌صورت Common Document فقط در Memory ساخته می‌شوند.</li>
<li>Paragraphهای Book با <code>book_unit_type</code> مشخص می‌شوند.</li>
<li>Schema در Memory برای هر دو نوع Book unit گسترش داده می‌شود.</li>
<li>کل Corpus متنی جدید دوباره با JSON Schema رسمی Validation می‌شود.</li>
<li>هیچ فایل روی Disk هنوز بازنویسی نمی‌شود.</li>
</ul>

<h4>معیار PASS</h4>

<p>
تمام Common Documentهای Website، Paragraphهای Book، Footnoteهای متنی و Transcript باید Schema را پاس کنند،
شناسه تکراری صفر باشد،
Hash متن سالم بماند،
Footnoteها به Paragraph والد صحیح متصل باشند
و هیچ Chart یا Diagram به‌اشتباه وارد Corpus متنی نشود.
</p>

</div>

In [21]:
# Code Cell 21

book_paragraph_documents_v2 = []

for document in book_common_documents_v1:
    updated = deepcopy(document)

    updated["source_metadata"][
        "book_unit_type"
    ] = "paragraph"

    book_paragraph_documents_v2.append(
        updated
    )


def build_book_footnote_common_v1(record):
    edition = edition_lookup[
        record["edition_id"]
    ]

    source_file = edition["source_file"]

    return {
        "schema_version": "common_document_v1",
        "document_id": record["content_id"],
        "source_type": "book",
        "source_id": record["edition_id"],
        "text": record["text"],
        "text_sha256": record["text_sha256"],
        "raw_content_hash": source_file[
            "file_sha256"
        ],
        "title": edition.get("title"),
        "language": edition.get("language"),
        "content_origin": None,
        "attribution": {
            "author": edition.get("author"),
            "author_id": None,
            "knowledge_owner": edition.get(
                "knowledge_owner"
            ),
            "speaker": None,
        },
        "rights": {
            "status": edition.get(
                "rights_status"
            ),
            "basis": None,
            "rights_holder": edition.get(
                "rights_holder"
            ),
            "rights_evidence_id": edition.get(
                "rights_evidence_id"
            ),
            "allowed_uses": edition.get(
                "allowed_uses"
            ),
        },
        "locator": {
            "work_id": record.get("work_id"),
            "edition_id": record.get(
                "edition_id"
            ),
            "content_id": record.get(
                "content_id"
            ),
            "parent_paragraph_id": record.get(
                "parent_paragraph_id"
            ),
            "source_locator": deepcopy(
                record.get("source_locator")
            ),
        },
        "lineage": {
            "parent_id": record.get(
                "parent_paragraph_id"
            ),
            "raw_source_path": source_file.get(
                "relative_path"
            ),
            "raw_source_hash": source_file.get(
                "file_sha256"
            ),
        },
        "source_metadata": {
            "book_unit_type": "supplemental_text",
            "content_type": record.get(
                "content_type"
            ),
            "content_id": record.get(
                "content_id"
            ),
            "work_id": record.get("work_id"),
            "edition_id": record.get(
                "edition_id"
            ),
            "parent_paragraph_id": record.get(
                "parent_paragraph_id"
            ),
            "source_locator": deepcopy(
                record.get("source_locator")
            ),
        },
    }


book_footnote_documents_v1 = [
    build_book_footnote_common_v1(record)
    for record in text_document_candidates
]


common_documents_v2_candidate = (
    website_common_documents_v1
    + book_paragraph_documents_v2
    + book_footnote_documents_v1
    + transcript_common_documents_v1
)


document_schema_v2_candidate = deepcopy(
    document_schema_v1
)


book_condition = None

for condition in document_schema_v2_candidate[
    "allOf"
]:
    source_const = (
        condition
        .get("if", {})
        .get("properties", {})
        .get("source_type", {})
        .get("const")
    )

    if source_const == "book":
        book_condition = condition
        break


assert book_condition is not None, (
    "Book conditional schema could not be located."
)


book_condition["then"]["properties"][
    "locator"
] = {
    "type": "object",
}

book_condition["then"]["properties"][
    "source_metadata"
] = {
    "type": "object",
    "required": [
        "book_unit_type",
    ],
    "properties": {
        "book_unit_type": {
            "enum": [
                "paragraph",
                "supplemental_text",
            ],
        },
    },
}


book_condition["then"]["allOf"] = [
    {
        "if": {
            "properties": {
                "source_metadata": {
                    "properties": {
                        "book_unit_type": {
                            "const": "paragraph",
                        }
                    },
                    "required": [
                        "book_unit_type",
                    ],
                }
            },
            "required": [
                "source_metadata",
            ],
        },
        "then": {
            "properties": {
                "locator": {
                    "type": "object",
                    "required": [
                        "work_id",
                        "edition_id",
                        "paragraph_id",
                        "paragraph_number",
                        "heading_path",
                    ],
                    "properties": {
                        "work_id": {
                            "type": "string",
                            "minLength": 1,
                        },
                        "edition_id": {
                            "type": "string",
                            "minLength": 1,
                        },
                        "paragraph_id": {
                            "type": "string",
                            "minLength": 1,
                        },
                        "paragraph_number": {
                            "type": "integer",
                        },
                        "heading_path": {
                            "type": "array",
                        },
                    },
                    "additionalProperties": False,
                },
                "source_metadata": {
                    "type": "object",
                    "required": [
                        "book_unit_type",
                        "work_id",
                        "edition_id",
                        "heading_level",
                        "heading_path",
                        "is_heading",
                        "style",
                        "in_table",
                        "paragraph_number",
                    ],
                },
            },
        },
    },
    {
        "if": {
            "properties": {
                "source_metadata": {
                    "properties": {
                        "book_unit_type": {
                            "const": "supplemental_text",
                        }
                    },
                    "required": [
                        "book_unit_type",
                    ],
                }
            },
            "required": [
                "source_metadata",
            ],
        },
        "then": {
            "properties": {
                "locator": {
                    "type": "object",
                    "required": [
                        "work_id",
                        "edition_id",
                        "content_id",
                        "parent_paragraph_id",
                        "source_locator",
                    ],
                    "properties": {
                        "work_id": {
                            "type": "string",
                            "minLength": 1,
                        },
                        "edition_id": {
                            "type": "string",
                            "minLength": 1,
                        },
                        "content_id": {
                            "type": "string",
                            "minLength": 1,
                        },
                        "parent_paragraph_id": {
                            "type": "string",
                            "minLength": 1,
                        },
                        "source_locator": {
                            "type": "object",
                            "required": [
                                "note_number",
                            ],
                        },
                    },
                    "additionalProperties": False,
                },
                "source_metadata": {
                    "type": "object",
                    "required": [
                        "book_unit_type",
                        "content_type",
                        "content_id",
                        "work_id",
                        "edition_id",
                        "parent_paragraph_id",
                        "source_locator",
                    ],
                    "properties": {
                        "book_unit_type": {
                            "const": "supplemental_text",
                        },
                        "content_type": {
                            "const": "FOOTNOTE",
                        },
                    },
                },
            },
        },
    },
]


Draft202012Validator.check_schema(
    document_schema_v2_candidate
)

document_validator_v2 = Draft202012Validator(
    document_schema_v2_candidate
)


schema_failures_v2 = []
text_hash_failures_v2 = []
lineage_failures_v2 = []

for document in common_documents_v2_candidate:
    errors = list(
        document_validator_v2.iter_errors(
            document
        )
    )

    if errors:
        schema_failures_v2.append(
            {
                "document_id": document[
                    "document_id"
                ],
                "source_type": document[
                    "source_type"
                ],
                "errors": [
                    error.message
                    for error in errors[:5]
                ],
            }
        )

    calculated_text_sha256 = (
        hashlib.sha256(
            document["text"].encode("utf-8")
        ).hexdigest()
    )

    if (
        calculated_text_sha256
        != document["text_sha256"]
    ):
        text_hash_failures_v2.append(
            document["document_id"]
        )

    if document["source_type"] == "book":
        book_unit_type = (
            document["source_metadata"]
            .get("book_unit_type")
        )

        if book_unit_type == "supplemental_text":
            parent_id = document[
                "lineage"
            ]["parent_id"]

            parent = paragraph_lookup.get(
                parent_id
            )

            if (
                parent is None
                or parent["edition_id"]
                != document["source_id"]
            ):
                lineage_failures_v2.append(
                    document["document_id"]
                )


document_ids_v2 = [
    document["document_id"]
    for document in common_documents_v2_candidate
]

duplicate_document_ids_v2 = (
    len(document_ids_v2)
    - len(set(document_ids_v2))
)


book_unit_distribution = Counter(
    document["source_metadata"].get(
        "book_unit_type"
    )
    for document in common_documents_v2_candidate
    if document["source_type"] == "book"
)


included_structured_asset_ids = {
    document["document_id"]
    for document in common_documents_v2_candidate
} & {
    record["content_id"]
    for record in structured_asset_candidates
}


expected_unified_population = (
    len(common_documents_v1_candidate)
    + len(text_document_candidates)
)


print("BOOK FOOTNOTE COMMON DOCUMENT INTEGRATION")
print("=" * 80)

print("\nPOPULATION")
print("-" * 80)
print(
    "Website documents:",
    len(website_common_documents_v1),
)
print(
    "Book paragraph documents:",
    len(book_paragraph_documents_v2),
)
print(
    "Book footnote documents:",
    len(book_footnote_documents_v1),
)
print(
    "Transcript documents:",
    len(transcript_common_documents_v1),
)
print(
    "Unified text candidate:",
    len(common_documents_v2_candidate),
)
print(
    "Expected unified text population:",
    expected_unified_population,
)

print("\nBOOK UNIT DISTRIBUTION")
print("-" * 80)
print(
    dict(book_unit_distribution)
)

print("\nVALIDATION")
print("-" * 80)
print(
    "Duplicate document_id:",
    duplicate_document_ids_v2,
)
print(
    "Schema failures:",
    len(schema_failures_v2),
)
print(
    "Text hash failures:",
    len(text_hash_failures_v2),
)
print(
    "Supplemental parent lineage failures:",
    len(lineage_failures_v2),
)
print(
    "Structured assets accidentally included:",
    len(included_structured_asset_ids),
)

if schema_failures_v2:
    print("\nFirst schema failures:")
    for failure in schema_failures_v2[:5]:
        print(
            json.dumps(
                failure,
                ensure_ascii=False,
                indent=2,
            )
        )


book_footnote_integration_gate = (
    len(common_documents_v2_candidate)
    == expected_unified_population
    and len(book_footnote_documents_v1)
    == len(text_document_candidates)
    and duplicate_document_ids_v2 == 0
    and len(schema_failures_v2) == 0
    and len(text_hash_failures_v2) == 0
    and len(lineage_failures_v2) == 0
    and len(included_structured_asset_ids) == 0
)

print("\nBOOK FOOTNOTE INTEGRATION GATE")
print("=" * 80)
print(
    "Population:",
    "PASS"
    if len(common_documents_v2_candidate)
    == expected_unified_population
    else "FAIL",
)
print(
    "Footnote coverage:",
    "PASS"
    if len(book_footnote_documents_v1)
    == len(text_document_candidates)
    else "FAIL",
)
print(
    "Global ID uniqueness:",
    "PASS"
    if duplicate_document_ids_v2 == 0
    else "FAIL",
)
print(
    "Formal JSON Schema:",
    "PASS"
    if len(schema_failures_v2) == 0
    else "FAIL",
)
print(
    "Text hash integrity:",
    "PASS"
    if len(text_hash_failures_v2) == 0
    else "FAIL",
)
print(
    "Supplemental lineage:",
    "PASS"
    if len(lineage_failures_v2) == 0
    else "FAIL",
)
print(
    "Structured assets excluded from text corpus:",
    "PASS"
    if len(included_structured_asset_ids) == 0
    else "FAIL",
)
print(
    "BOOK FOOTNOTE INTEGRATION:",
    "PASS"
    if book_footnote_integration_gate
    else "FAIL",
)

assert book_footnote_integration_gate, (
    "Book Footnote Common Document integration failed."
)

BOOK FOOTNOTE COMMON DOCUMENT INTEGRATION

POPULATION
--------------------------------------------------------------------------------
Website documents: 529
Book paragraph documents: 2689
Book footnote documents: 12
Transcript documents: 16226
Unified text candidate: 19456
Expected unified text population: 19456

BOOK UNIT DISTRIBUTION
--------------------------------------------------------------------------------
{'paragraph': 2689, 'supplemental_text': 12}

VALIDATION
--------------------------------------------------------------------------------
Duplicate document_id: 0
Schema failures: 0
Text hash failures: 0
Supplemental parent lineage failures: 0
Structured assets accidentally included: 0

BOOK FOOTNOTE INTEGRATION GATE
Population: PASS
Footnote coverage: PASS
Global ID uniqueness: PASS
Formal JSON Schema: PASS
Text hash integrity: PASS
Supplemental lineage: PASS
Structured assets excluded from text corpus: PASS
BOOK FOOTNOTE INTEGRATION: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>به‌روزرسانی Schema رسمی با پشتیبانی از Footnoteهای متنی</h3>

<p>
پس از بررسی Supplemental Content مشخص شد ۱۲ Footnote باید به Unified Text Corpus اضافه شوند و Book Common Document اکنون دو نوع واحد متنی دارد:
<code>paragraph</code> و <code>supplemental_text</code>.
</p>

<p>
فایل Schema فعلی روی Disk مربوط به Contract پیش از این تصمیم است.
در این مرحله همان فایل به‌صورت کنترل‌شده با Schema کامل‌شده Replace می‌شود.
</p>

<p>
پس از نوشتن، Schema دوباره از Disk خوانده می‌شود و Validation مستقل روی تمام ۱۹۴۵۶ Common Document انجام خواهد شد.
</p>

<h4>محدوده تغییر</h4>

<ul>
<li>فقط <code>schemas/phase2/document.schema.json</code> به‌روزرسانی می‌شود.</li>
<li>هیچ Unified Dataset یا Report هنوز نوشته نمی‌شود.</li>
<li>هیچ Git Commit، Push یا عملیات Repository انجام نمی‌شود.</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>Schema ذخیره‌شده از نظر Draft 2020-12 معتبر باشد.</li>
<li>تمام ۱۹۴۵۶ سند آن را پاس کنند.</li>
<li>Paragraph و Supplemental Text هر دو به‌درستی Validation شوند.</li>
<li>SHA-256 جدید Schema ثبت شود.</li>
</ul>

</div>

In [22]:
# Code Cell 22

updated_schema_output_path = (
    project_root
    / "schemas"
    / "phase2"
    / "document.schema.json"
)

assert updated_schema_output_path.is_file(), (
    "Existing persisted schema is missing."
)

previous_schema_sha256 = sha256_file(
    updated_schema_output_path
)

updated_schema_text = json.dumps(
    document_schema_v2_candidate,
    ensure_ascii=False,
    indent=2,
    sort_keys=True,
) + "\n"

temporary_updated_schema_path = (
    updated_schema_output_path
    .with_suffix(".schema.json.tmp")
)

temporary_updated_schema_path.write_text(
    updated_schema_text,
    encoding="utf-8",
)


with temporary_updated_schema_path.open(
    "r",
    encoding="utf-8",
) as handle:
    temporary_updated_schema = json.load(
        handle
    )

Draft202012Validator.check_schema(
    temporary_updated_schema
)

temporary_updated_validator = (
    Draft202012Validator(
        temporary_updated_schema
    )
)

temporary_schema_failures = []

for document in common_documents_v2_candidate:
    errors = list(
        temporary_updated_validator
        .iter_errors(document)
    )

    if errors:
        temporary_schema_failures.append(
            {
                "document_id": document[
                    "document_id"
                ],
                "errors": [
                    error.message
                    for error in errors[:5]
                ],
            }
        )


assert len(temporary_schema_failures) == 0, (
    "Updated temporary schema failed "
    "full-corpus validation."
)


temporary_updated_schema_path.replace(
    updated_schema_output_path
)


with updated_schema_output_path.open(
    "r",
    encoding="utf-8",
) as handle:
    persisted_schema_v2 = json.load(handle)

Draft202012Validator.check_schema(
    persisted_schema_v2
)

persisted_validator_v2 = (
    Draft202012Validator(
        persisted_schema_v2
    )
)

persisted_schema_failures_v2 = []

for document in common_documents_v2_candidate:
    errors = list(
        persisted_validator_v2.iter_errors(
            document
        )
    )

    if errors:
        persisted_schema_failures_v2.append(
            {
                "document_id": document[
                    "document_id"
                ],
                "source_type": document[
                    "source_type"
                ],
                "errors": [
                    error.message
                    for error in errors[:5]
                ],
            }
        )


updated_schema_sha256 = sha256_file(
    updated_schema_output_path
)

schema_hash_changed = (
    updated_schema_sha256
    != previous_schema_sha256
)


book_paragraph_schema_failures = sum(
    failure["document_id"]
    in {
        document["document_id"]
        for document
        in book_paragraph_documents_v2
    }
    for failure in persisted_schema_failures_v2
)

book_footnote_schema_failures = sum(
    failure["document_id"]
    in {
        document["document_id"]
        for document
        in book_footnote_documents_v1
    }
    for failure in persisted_schema_failures_v2
)


print("UPDATED PERSISTED COMMON DOCUMENT SCHEMA")
print("=" * 80)

print(
    "Path:",
    updated_schema_output_path.relative_to(
        project_root
    ),
)

print("\nHASH STATE")
print("-" * 80)
print(
    "Previous SHA-256:",
    previous_schema_sha256,
)
print(
    "Updated SHA-256:",
    updated_schema_sha256,
)
print(
    "Schema content changed:",
    schema_hash_changed,
)

print("\nVALIDATION")
print("-" * 80)
print(
    "Schema reload:",
    "PASS",
)
print(
    "Draft 2020-12 validation:",
    "PASS",
)
print(
    "Documents validated:",
    len(common_documents_v2_candidate),
)
print(
    "Total schema failures:",
    len(persisted_schema_failures_v2),
)
print(
    "Book paragraph failures:",
    book_paragraph_schema_failures,
)
print(
    "Book footnote failures:",
    book_footnote_schema_failures,
)

if persisted_schema_failures_v2:
    print("\nFirst persisted schema failures:")
    for failure in persisted_schema_failures_v2[:5]:
        print(
            json.dumps(
                failure,
                ensure_ascii=False,
                indent=2,
            )
        )


updated_schema_gate = (
    schema_hash_changed
    and len(common_documents_v2_candidate)
    == 19456
    and len(persisted_schema_failures_v2) == 0
    and book_paragraph_schema_failures == 0
    and book_footnote_schema_failures == 0
)

print("\nUPDATED PERSISTED SCHEMA GATE")
print("=" * 80)

print(
    "Schema replacement:",
    "PASS"
    if schema_hash_changed
    else "FAIL",
)
print(
    "Full-corpus validation:",
    "PASS"
    if len(persisted_schema_failures_v2) == 0
    else "FAIL",
)
print(
    "Book paragraph support:",
    "PASS"
    if book_paragraph_schema_failures == 0
    else "FAIL",
)
print(
    "Book supplemental-text support:",
    "PASS"
    if book_footnote_schema_failures == 0
    else "FAIL",
)
print(
    "UPDATED DOCUMENT SCHEMA:",
    "PASS"
    if updated_schema_gate
    else "FAIL",
)

assert schema_hash_changed, (
    "Persisted schema did not change after "
    "Footnote support was added."
)

assert len(common_documents_v2_candidate) == 19456

assert len(persisted_schema_failures_v2) == 0

assert book_paragraph_schema_failures == 0

assert book_footnote_schema_failures == 0

UPDATED PERSISTED COMMON DOCUMENT SCHEMA
Path: schemas/phase2/document.schema.json

HASH STATE
--------------------------------------------------------------------------------
Previous SHA-256: a42f57aa53e3ee8c955232664ef49e58bc7abe65f816a51db6836ce62dba3d11
Updated SHA-256: 174551afb77055a9012c7e29fa2044dd0dd67b78a54135a27947be1eb1f655de
Schema content changed: True

VALIDATION
--------------------------------------------------------------------------------
Schema reload: PASS
Draft 2020-12 validation: PASS
Documents validated: 19456
Total schema failures: 0
Book paragraph failures: 0
Book footnote failures: 0

UPDATED PERSISTED SCHEMA GATE
Schema replacement: PASS
Full-corpus validation: PASS
Book paragraph support: PASS
Book supplemental-text support: PASS
UPDATED DOCUMENT SCHEMA: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ممیزی آمادگی Attribution وب‌سایت و جلوگیری از ارتقای بدون مدرک</h3>

<p>
نقشه اجرایی فاز ۲ برای Website فقط وجود نام نویسنده یا مقدارهای قدیمی
<code>primary_author</code> و <code>supervised</code> را برای تعیین نقش نهایی کافی نمی‌داند.
تبدیل این مقادیر به
<code>direct_author</code>
یا
<code>editorial_supervision</code>
نیازمند Byline یا مدرک انتساب قابل ردیابی است.
</p>

<p>
بازاعتبارسنجی قبلی نشان داد در Rawهای ۵۲۹ مقاله فیلد Attribution اضافی مشاهده نشده است.
بنابراین این مرحله بررسی می‌کند آیا فیلدهای الزامی Attribution در هیچ‌یک از Artifactهای پذیرفته‌شده فعلی وجود دارند یا خیر.
</p>

<h4>فیلدهای مورد انتظار Website Attribution</h4>

<ul>
<li><code>credited_author_name</code></li>
<li><code>reference_expert_name</code></li>
<li><code>editorial_supervisor_name</code></li>
<li><code>content_authority_type</code></li>
<li><code>byline_raw</code></li>
<li><code>editorial_credit_raw</code></li>
<li><code>authorship_assertion_source</code></li>
<li><code>attribution_status</code></li>
</ul>

<p>
اگر Evidence لازم در Artifactهای فعلی وجود نداشته باشد، اطلاعات قدیمی
<code>author</code>،
<code>author_id</code>،
<code>knowledge_owner</code>
و
<code>content_origin</code>
حذف نمی‌شوند، اما به نقش تأییدشده تبدیل نیز نخواهند شد.
در آن حالت رکورد با
<code>content_authority_type=unknown_pending_review</code>
و
<code>attribution_status=pending_review</code>
نمایندگی می‌شود.
</p>

<h4>معیار PASS</h4>

<p>
تمام ۵۲۹ مقاله باید بدون جعل نام یا نقش قابل نمایندگی باشند،
Legacy attribution باید حفظ شود
و تعداد Attributionهای تأییدشده بدون Evidence برابر صفر باشد.
نبود Evidence یک محدودیت Dataset است، نه مجوز برای حدس‌زدن.
</p>

</div>

In [23]:
# Code Cell 23

required_website_attribution_fields = (
    "credited_author_name",
    "reference_expert_name",
    "editorial_supervisor_name",
    "content_authority_type",
    "byline_raw",
    "editorial_credit_raw",
    "authorship_assertion_source",
    "attribution_status",
)


def recursive_key_counts(value):
    counts = Counter()

    if isinstance(value, dict):
        for key, child in value.items():
            counts[key] += 1
            counts.update(
                recursive_key_counts(child)
            )

    elif isinstance(value, list):
        for child in value:
            counts.update(
                recursive_key_counts(child)
            )

    return counts


processed_key_counts = recursive_key_counts(
    website_data
)

manifest_key_counts = recursive_key_counts(
    website_manifest_records
)

acceptance_key_counts = recursive_key_counts(
    loaded_acceptance_reports["website"]
)


raw_required_field_counts = Counter()

for article_id in sorted(processed_ids):
    raw_source_path = (
        project_root
        / "data"
        / "phase2"
        / "raw"
        / "website"
        / article_id
        / "source.json"
    )

    with raw_source_path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        raw_record = json.load(handle)

    raw_counts = recursive_key_counts(
        raw_record
    )

    for field in required_website_attribution_fields:
        if raw_counts.get(field, 0) > 0:
            raw_required_field_counts[field] += 1


legacy_field_coverage = {
    field: sum(
        record.get(field) is not None
        and record.get(field) != ""
        for record in website_data
    )
    for field in (
        "author",
        "author_id",
        "knowledge_owner",
        "content_origin",
    )
}


website_attribution_candidates = []

for record in website_data:
    website_attribution_candidates.append(
        {
            "document_id": record["document_id"],

            "credited_author_name": None,
            "reference_expert_name": None,
            "editorial_supervisor_name": None,

            "content_authority_type": (
                "unknown_pending_review"
            ),

            "byline_raw": None,
            "editorial_credit_raw": None,
            "authorship_assertion_source": None,

            "attribution_status": (
                "pending_review"
            ),

            "legacy_attribution": {
                "author": record.get("author"),
                "author_id": record.get(
                    "author_id"
                ),
                "knowledge_owner": record.get(
                    "knowledge_owner"
                ),
                "content_origin": record.get(
                    "content_origin"
                ),
            },
        }
    )


verified_without_evidence = [
    record
    for record
    in website_attribution_candidates
    if (
        record["attribution_status"]
        == "verified"
        or record["content_authority_type"]
        in {
            "direct_author",
            "editorial_supervision",
        }
    )
    and not record.get(
        "authorship_assertion_source"
    )
]


pending_review_count = sum(
    record["attribution_status"]
    == "pending_review"
    for record in website_attribution_candidates
)

unknown_authority_count = sum(
    record["content_authority_type"]
    == "unknown_pending_review"
    for record in website_attribution_candidates
)

legacy_claims_preserved = sum(
    isinstance(
        record.get("legacy_attribution"),
        dict,
    )
    and nonempty_text(
        record["legacy_attribution"].get(
            "author"
        )
    )
    and nonempty_text(
        record["legacy_attribution"].get(
            "content_origin"
        )
    )
    for record in website_attribution_candidates
)


print("WEBSITE ATTRIBUTION READINESS AUDIT")
print("=" * 80)

print("\nREQUIRED ATTRIBUTION FIELD DISCOVERY")
print("-" * 80)

for field in required_website_attribution_fields:
    print(
        field,
        {
            "processed": processed_key_counts.get(
                field,
                0,
            ),
            "manifest": manifest_key_counts.get(
                field,
                0,
            ),
            "acceptance_report": (
                acceptance_key_counts.get(
                    field,
                    0,
                )
            ),
            "raw_records": (
                raw_required_field_counts.get(
                    field,
                    0,
                )
            ),
        },
    )


print("\nLEGACY ACCEPTED FIELD COVERAGE")
print("-" * 80)

for field, count in legacy_field_coverage.items():
    print(
        f"{field}: {count}/{len(website_data)}"
    )


print("\nSAFE ATTRIBUTION REPRESENTATION")
print("-" * 80)
print(
    "Candidate records:",
    len(website_attribution_candidates),
)
print(
    "pending_review:",
    pending_review_count,
)
print(
    "unknown_pending_review:",
    unknown_authority_count,
)
print(
    "Legacy attribution claims preserved:",
    legacy_claims_preserved,
)
print(
    "Verified/promoted without evidence:",
    len(verified_without_evidence),
)


exact_evidence_field_presence = {
    field: (
        processed_key_counts.get(field, 0)
        + manifest_key_counts.get(field, 0)
        + acceptance_key_counts.get(field, 0)
        + raw_required_field_counts.get(field, 0)
    )
    for field
    in required_website_attribution_fields
}


print("\nEVIDENCE READINESS")
print("-" * 80)

for field, count in exact_evidence_field_presence.items():
    print(
        f"{field}:",
        "AVAILABLE"
        if count > 0
        else "NOT AVAILABLE",
    )


website_safe_attribution_gate = (
    len(website_attribution_candidates)
    == len(website_data)
    and pending_review_count
    == len(website_data)
    and unknown_authority_count
    == len(website_data)
    and legacy_claims_preserved
    == len(website_data)
    and len(verified_without_evidence) == 0
)


print("\nWEBSITE ATTRIBUTION SAFETY GATE")
print("=" * 80)

print(
    "All Website records represented:",
    "PASS"
    if len(website_attribution_candidates)
    == len(website_data)
    else "FAIL",
)

print(
    "Legacy attribution preserved:",
    "PASS"
    if legacy_claims_preserved
    == len(website_data)
    else "FAIL",
)

print(
    "Unsafe verified attribution:",
    "PASS"
    if len(verified_without_evidence) == 0
    else "FAIL",
)

print(
    "Pending-review fallback:",
    "PASS"
    if (
        pending_review_count
        == len(website_data)
        and unknown_authority_count
        == len(website_data)
    )
    else "FAIL",
)

print(
    "WEBSITE ATTRIBUTION SAFETY:",
    "PASS"
    if website_safe_attribution_gate
    else "FAIL",
)


assert len(website_attribution_candidates) == len(
    website_data
)

assert legacy_claims_preserved == len(
    website_data
)

assert len(verified_without_evidence) == 0

assert pending_review_count == len(
    website_data
)

assert unknown_authority_count == len(
    website_data
)

WEBSITE ATTRIBUTION READINESS AUDIT

REQUIRED ATTRIBUTION FIELD DISCOVERY
--------------------------------------------------------------------------------
credited_author_name {'processed': 0, 'manifest': 0, 'acceptance_report': 0, 'raw_records': 0}
reference_expert_name {'processed': 0, 'manifest': 0, 'acceptance_report': 0, 'raw_records': 0}
editorial_supervisor_name {'processed': 0, 'manifest': 0, 'acceptance_report': 0, 'raw_records': 0}
content_authority_type {'processed': 0, 'manifest': 0, 'acceptance_report': 0, 'raw_records': 0}
byline_raw {'processed': 0, 'manifest': 0, 'acceptance_report': 0, 'raw_records': 0}
editorial_credit_raw {'processed': 0, 'manifest': 0, 'acceptance_report': 0, 'raw_records': 0}
authorship_assertion_source {'processed': 0, 'manifest': 0, 'acceptance_report': 0, 'raw_records': 0}
attribution_status {'processed': 0, 'manifest': 0, 'acceptance_report': 0, 'raw_records': 0}

LEGACY ACCEPTED FIELD COVERAGE
--------------------------------------------------

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>اعتبارسنجی سیاست رسمی Content Origin وب‌سایت</h3>

<p>
طبق سیاست داده‌ای پروژه، مقدار <code>content_origin</code> در Pipeline وب‌سایت از قبل با معنای مشخص ثبت شده است:
</p>

<ul>
<li><code>primary_author</code>: مقاله‌ای که مؤلف آن خود دکتر کمیل رودی است و در Corpus نقش منبع دست‌اول دارد.</li>
<li><code>supervised</code>: مقاله‌ای با مؤلف دیگر که زیر نظر یا سردبیری دکتر رودی تولید شده و به‌عنوان منبع دست‌دوم معتبر در Corpus نگهداری می‌شود.</li>
</ul>

<p>
بنابراین <code>content_origin</code> یک Classification عمدی پروژه است و نباید صرفاً به دلیل نبود <code>byline_raw</code> حذف یا به <code>unknown_pending_review</code> تبدیل شود.
</p>

<p>
در عین حال نام مؤلف واقعی هر مقاله در <code>author</code> حفظ می‌شود و هیچ Byline یا Evidence خامی که در Dataset وجود ندارد ساخته نخواهد شد.
</p>

<h4>بررسی‌های این مرحله</h4>

<ul>
<li>وجود فقط دو گروه رسمی <code>primary_author</code> و <code>supervised</code></li>
<li>وجود Author، Author ID و Knowledge Owner برای تمام رکوردها</li>
<li>بررسی Authorها و Author IDهای هر گروه</li>
<li>کنترل عدم هم‌پوشانی Author ID میان دو گروه</li>
<li>کنترل یکپارچگی Knowledge Owner در کل Corpus سایت</li>
<li>تطبیق تعداد دو گروه با Acceptance Report قبلی</li>
</ul>

<h4>معیار PASS</h4>

<p>
Classification موجود باید در تمام ۵۲۹ رکورد کامل و بدون ابهام ساختاری باشد،
Author IDهای دو گروه با یکدیگر برخورد نداشته باشند،
Knowledge Owner یکپارچه باشد
و توزیع Content Origin با Acceptance Report پذیرفته‌شده منطبق بماند.
</p>

</div>

In [24]:
# Code Cell 24

website_origin_groups = defaultdict(
    list
)

for record in website_data:
    website_origin_groups[
        record.get("content_origin")
    ].append(record)


observed_origin_values = set(
    website_origin_groups
)

expected_origin_values = {
    "primary_author",
    "supervised",
}


origin_author_names = {}

origin_author_ids = {}

origin_knowledge_owners = {}

for origin, records in website_origin_groups.items():
    origin_author_names[origin] = sorted(
        {
            record.get("author")
            for record in records
            if nonempty_text(
                record.get("author")
            )
        }
    )

    origin_author_ids[origin] = sorted(
        {
            record.get("author_id")
            for record in records
            if nonempty_text(
                record.get("author_id")
            )
        }
    )

    origin_knowledge_owners[origin] = sorted(
        {
            record.get("knowledge_owner")
            for record in records
            if nonempty_text(
                record.get("knowledge_owner")
            )
        }
    )


missing_author = missing_count(
    website_data,
    lambda record: record.get("author"),
)

missing_author_id = missing_count(
    website_data,
    lambda record: record.get("author_id"),
)

missing_knowledge_owner = missing_count(
    website_data,
    lambda record: record.get(
        "knowledge_owner"
    ),
)

missing_content_origin = missing_count(
    website_data,
    lambda record: record.get(
        "content_origin"
    ),
)


primary_author_ids = set(
    origin_author_ids.get(
        "primary_author",
        [],
    )
)

supervised_author_ids = set(
    origin_author_ids.get(
        "supervised",
        [],
    )
)

cross_origin_author_id_overlap = sorted(
    primary_author_ids
    & supervised_author_ids
)


all_knowledge_owners = sorted(
    {
        record.get("knowledge_owner")
        for record in website_data
        if nonempty_text(
            record.get("knowledge_owner")
        )
    }
)


current_origin_distribution = Counter(
    record["content_origin"]
    for record in website_data
)

accepted_origin_distribution = {
    "primary_author": (
        loaded_acceptance_reports[
            "website"
        ]
        .get("attribution", {})
        .get("primary_author")
    ),
    "supervised": (
        loaded_acceptance_reports[
            "website"
        ]
        .get("attribution", {})
        .get("supervised")
    ),
}


origin_distribution_matches = (
    dict(current_origin_distribution)
    == accepted_origin_distribution
)


print("WEBSITE CONTENT ORIGIN POLICY VALIDATION")
print("=" * 80)

print("\nORIGIN DISTRIBUTION")
print("-" * 80)
print(
    "Observed origin values:",
    sorted(observed_origin_values),
)
print(
    "Current distribution:",
    dict(current_origin_distribution),
)
print(
    "Acceptance distribution:",
    accepted_origin_distribution,
)
print(
    "Distribution comparison:",
    "PASS"
    if origin_distribution_matches
    else "FAIL",
)


print("\nPRIMARY AUTHOR GROUP")
print("-" * 80)
print(
    "Records:",
    len(
        website_origin_groups.get(
            "primary_author",
            [],
        )
    ),
)
print(
    "Unique author names:",
    origin_author_names.get(
        "primary_author",
        [],
    ),
)
print(
    "Unique author IDs:",
    origin_author_ids.get(
        "primary_author",
        [],
    ),
)
print(
    "Knowledge owners:",
    origin_knowledge_owners.get(
        "primary_author",
        [],
    ),
)


print("\nSUPERVISED GROUP")
print("-" * 80)
print(
    "Records:",
    len(
        website_origin_groups.get(
            "supervised",
            [],
        )
    ),
)
print(
    "Unique author names:",
    origin_author_names.get(
        "supervised",
        [],
    ),
)
print(
    "Unique author IDs:",
    origin_author_ids.get(
        "supervised",
        [],
    ),
)
print(
    "Knowledge owners:",
    origin_knowledge_owners.get(
        "supervised",
        [],
    ),
)


print("\nSTRUCTURAL CONSISTENCY")
print("-" * 80)
print(
    "Missing author:",
    missing_author,
)
print(
    "Missing author_id:",
    missing_author_id,
)
print(
    "Missing knowledge_owner:",
    missing_knowledge_owner,
)
print(
    "Missing content_origin:",
    missing_content_origin,
)
print(
    "Cross-origin author_id overlap:",
    cross_origin_author_id_overlap,
)
print(
    "Unique knowledge_owner values:",
    all_knowledge_owners,
)


origin_policy_gate = (
    observed_origin_values
    == expected_origin_values
    and missing_author == 0
    and missing_author_id == 0
    and missing_knowledge_owner == 0
    and missing_content_origin == 0
    and len(cross_origin_author_id_overlap) == 0
    and len(all_knowledge_owners) == 1
    and origin_distribution_matches
)


print("\nCONTENT ORIGIN POLICY GATE")
print("=" * 80)

print(
    "Expected origin classes:",
    "PASS"
    if observed_origin_values
    == expected_origin_values
    else "FAIL",
)

print(
    "Attribution field completeness:",
    "PASS"
    if (
        missing_author == 0
        and missing_author_id == 0
        and missing_knowledge_owner == 0
        and missing_content_origin == 0
    )
    else "FAIL",
)

print(
    "Author identity separation:",
    "PASS"
    if len(
        cross_origin_author_id_overlap
    ) == 0
    else "FAIL",
)

print(
    "Knowledge owner consistency:",
    "PASS"
    if len(all_knowledge_owners) == 1
    else "FAIL",
)

print(
    "Acceptance distribution:",
    "PASS"
    if origin_distribution_matches
    else "FAIL",
)

print(
    "WEBSITE CONTENT ORIGIN POLICY:",
    "PASS"
    if origin_policy_gate
    else "FAIL",
)


assert observed_origin_values == expected_origin_values

assert missing_author == 0
assert missing_author_id == 0
assert missing_knowledge_owner == 0
assert missing_content_origin == 0

assert len(cross_origin_author_id_overlap) == 0

assert len(all_knowledge_owners) == 1

assert origin_distribution_matches

WEBSITE CONTENT ORIGIN POLICY VALIDATION

ORIGIN DISTRIBUTION
--------------------------------------------------------------------------------
Observed origin values: ['primary_author', 'supervised']
Current distribution: {'supervised': 453, 'primary_author': 76}
Acceptance distribution: {'primary_author': 76, 'supervised': 453}
Distribution comparison: PASS

PRIMARY AUTHOR GROUP
--------------------------------------------------------------------------------
Records: 76
Unique author names: ['دکتر کمیل رودی']
Unique author IDs: ['4']
Knowledge owners: ['Dr. Komeil Roudi']

SUPERVISED GROUP
--------------------------------------------------------------------------------
Records: 453
Unique author names: ['الهام میرمحمدی', 'حمیده معماریان', 'دکتر ماجد ناجی', 'دکتر نیره\u200cالسادات علوی', 'دکتر کوثر اخوان', 'سجاد یزدانی', 'سودابه فرخنده', 'شیما فدوی', 'صاحب\u200cنظران', 'عاطفه جلادتی', 'فاطمه قاسمی\u200cپور', 'مائده محبوب', 'محمدرضا سیمیاری', 'مریم محمدخواه', 'مهسا صادقی', 'هادی نجف', '

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>تکمیل Attribution Contract بر اساس سیاست رسمی Website</h3>

<p>
اعتبارسنجی کامل Website نشان داد مقدار <code>content_origin</code> یک Classification عمدی و پایدار در Pipeline است.
بنابراین Attribution نهایی Website می‌تواند بدون حدس‌زدن از این سیاست استفاده کند.
</p>

<h4>Mapping رسمی Website</h4>

<ul>
<li>
برای <code>primary_author</code>:
نام موجود در <code>author</code> به‌عنوان <code>credited_author_name</code> حفظ می‌شود و
<code>content_authority_type</code> برابر <code>direct_author</code> خواهد بود.
</li>
<li>
برای <code>supervised</code>:
نام موجود در <code>author</code> مؤلف واقعی مقاله باقی می‌ماند،
<code>knowledge_owner</code> به‌عنوان <code>editorial_supervisor_name</code> ثبت می‌شود
و <code>content_authority_type</code> برابر <code>editorial_supervision</code> خواهد بود.
</li>
</ul>

<p>
فیلدهای <code>byline_raw</code> و <code>editorial_credit_raw</code> همچنان <code>null</code> باقی می‌مانند،
زیرا چنین Evidence خامی در Artifactهای فعلی وجود ندارد.
منبع این Assertion نیز صریحاً به سیاست پذیرفته‌شده <code>content_origin</code> نسبت داده می‌شود و نه به Raw Byline.
</p>

<p>
برای Book و Transcript همین ساختار Attribution حفظ می‌شود، اما فیلدهای Website-specific بدون Evidence برابر <code>null</code> خواهند ماند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام ۵۲۹ Website record باید Mapping معتبر Attribution داشته باشند.</li>
<li>۷۶ Primary record باید <code>direct_author</code> شوند.</li>
<li>۴۵۳ Supervised record باید <code>editorial_supervision</code> شوند.</li>
<li>هیچ Byline یا Editorial Credit خامی ساخته نشود.</li>
<li>تمام ۱۹۴۵۶ Common Document باید Schema تکمیل‌شده را پاس کنند.</li>
</ul>

</div>

In [25]:
# Code Cell 25

def website_attribution_v2(record):
    origin = record["content_origin"]

    if origin == "primary_author":
        return {
            "author": record.get("author"),
            "author_id": record.get("author_id"),
            "knowledge_owner": record.get(
                "knowledge_owner"
            ),
            "speaker": None,

            "credited_author_name": record.get(
                "author"
            ),
            "reference_expert_name": None,
            "editorial_supervisor_name": None,

            "content_authority_type": (
                "direct_author"
            ),

            "byline_raw": None,
            "editorial_credit_raw": None,

            "authorship_assertion_source": (
                "accepted_pipeline_content_origin"
            ),

            "attribution_status": (
                "policy_validated"
            ),
        }

    if origin == "supervised":
        return {
            "author": record.get("author"),
            "author_id": record.get("author_id"),
            "knowledge_owner": record.get(
                "knowledge_owner"
            ),
            "speaker": None,

            "credited_author_name": record.get(
                "author"
            ),
            "reference_expert_name": None,

            "editorial_supervisor_name": (
                record.get("knowledge_owner")
            ),

            "content_authority_type": (
                "editorial_supervision"
            ),

            "byline_raw": None,
            "editorial_credit_raw": None,

            "authorship_assertion_source": (
                "accepted_pipeline_content_origin"
            ),

            "attribution_status": (
                "policy_validated"
            ),
        }

    raise ValueError(
        f"Unsupported Website content_origin: {origin}"
    )


def neutral_extended_attribution(
    existing_attribution
):
    return {
        "author": existing_attribution.get(
            "author"
        ),
        "author_id": existing_attribution.get(
            "author_id"
        ),
        "knowledge_owner": (
            existing_attribution.get(
                "knowledge_owner"
            )
        ),
        "speaker": existing_attribution.get(
            "speaker"
        ),

        "credited_author_name": None,
        "reference_expert_name": None,
        "editorial_supervisor_name": None,

        "content_authority_type": None,

        "byline_raw": None,
        "editorial_credit_raw": None,
        "authorship_assertion_source": None,

        "attribution_status": None,
    }


website_common_documents_v3 = []

for source_record, document in zip(
    website_data,
    website_common_documents_v1,
):
    updated = deepcopy(document)

    updated["attribution"] = (
        website_attribution_v2(
            source_record
        )
    )

    # Raw Website path was independently
    # resolved and hash-validated in Cell 18.
    updated["lineage"]["raw_source_path"] = (
        website_raw_resolution[
            document["document_id"]
        ]
    )

    website_common_documents_v3.append(
        updated
    )


book_common_documents_v3 = []

for document in (
    book_paragraph_documents_v2
    + book_footnote_documents_v1
):
    updated = deepcopy(document)

    updated["attribution"] = (
        neutral_extended_attribution(
            document["attribution"]
        )
    )

    book_common_documents_v3.append(
        updated
    )


transcript_common_documents_v3 = []

for document in transcript_common_documents_v1:
    updated = deepcopy(document)

    updated["attribution"] = (
        neutral_extended_attribution(
            document["attribution"]
        )
    )

    transcript_common_documents_v3.append(
        updated
    )


common_documents_v3_candidate = (
    website_common_documents_v3
    + book_common_documents_v3
    + transcript_common_documents_v3
)


document_schema_v3_candidate = deepcopy(
    document_schema_v2_candidate
)


attribution_schema = (
    document_schema_v3_candidate[
        "properties"
    ]["attribution"]
)

attribution_schema["required"] = [
    "author",
    "author_id",
    "knowledge_owner",
    "speaker",
    "credited_author_name",
    "reference_expert_name",
    "editorial_supervisor_name",
    "content_authority_type",
    "byline_raw",
    "editorial_credit_raw",
    "authorship_assertion_source",
    "attribution_status",
]

attribution_schema["properties"].update(
    {
        "credited_author_name": {
            "type": ["string", "null"],
        },
        "reference_expert_name": {
            "type": ["string", "null"],
        },
        "editorial_supervisor_name": {
            "type": ["string", "null"],
        },
        "content_authority_type": {
            "enum": [
                "direct_author",
                "editorial_supervision",
                None,
            ],
        },
        "byline_raw": {
            "type": ["string", "null"],
        },
        "editorial_credit_raw": {
            "type": ["string", "null"],
        },
        "authorship_assertion_source": {
            "type": ["string", "null"],
        },
        "attribution_status": {
            "enum": [
                "policy_validated",
                None,
            ],
        },
    }
)


website_condition_v3 = None

for condition in document_schema_v3_candidate[
    "allOf"
]:
    source_const = (
        condition
        .get("if", {})
        .get("properties", {})
        .get("source_type", {})
        .get("const")
    )

    if source_const == "website":
        website_condition_v3 = condition
        break


assert website_condition_v3 is not None


website_condition_v3["then"].setdefault(
    "allOf",
    [],
)

website_condition_v3["then"]["allOf"] = [
    {
        "if": {
            "properties": {
                "content_origin": {
                    "const": "primary_author",
                }
            },
            "required": [
                "content_origin",
            ],
        },
        "then": {
            "properties": {
                "attribution": {
                    "properties": {
                        "content_authority_type": {
                            "const": "direct_author",
                        },
                        "attribution_status": {
                            "const": "policy_validated",
                        },
                        "authorship_assertion_source": {
                            "const": (
                                "accepted_pipeline_content_origin"
                            ),
                        },
                        "credited_author_name": {
                            "type": "string",
                            "minLength": 1,
                        },
                        "editorial_supervisor_name": {
                            "type": "null",
                        },
                        "byline_raw": {
                            "type": "null",
                        },
                        "editorial_credit_raw": {
                            "type": "null",
                        },
                    },
                },
            },
        },
    },
    {
        "if": {
            "properties": {
                "content_origin": {
                    "const": "supervised",
                }
            },
            "required": [
                "content_origin",
            ],
        },
        "then": {
            "properties": {
                "attribution": {
                    "properties": {
                        "content_authority_type": {
                            "const": (
                                "editorial_supervision"
                            ),
                        },
                        "attribution_status": {
                            "const": "policy_validated",
                        },
                        "authorship_assertion_source": {
                            "const": (
                                "accepted_pipeline_content_origin"
                            ),
                        },
                        "credited_author_name": {
                            "type": "string",
                            "minLength": 1,
                        },
                        "editorial_supervisor_name": {
                            "type": "string",
                            "minLength": 1,
                        },
                        "byline_raw": {
                            "type": "null",
                        },
                        "editorial_credit_raw": {
                            "type": "null",
                        },
                    },
                },
            },
        },
    },
]


Draft202012Validator.check_schema(
    document_schema_v3_candidate
)

validator_v3 = Draft202012Validator(
    document_schema_v3_candidate
)


schema_failures_v3 = []

for document in common_documents_v3_candidate:
    errors = list(
        validator_v3.iter_errors(document)
    )

    if errors:
        schema_failures_v3.append(
            {
                "document_id": document[
                    "document_id"
                ],
                "source_type": document[
                    "source_type"
                ],
                "errors": [
                    error.message
                    for error in errors[:5]
                ],
            }
        )


website_authority_distribution = Counter(
    document["attribution"][
        "content_authority_type"
    ]
    for document
    in website_common_documents_v3
)

website_attribution_status_distribution = (
    Counter(
        document["attribution"][
            "attribution_status"
        ]
        for document
        in website_common_documents_v3
    )
)

website_assertion_source_distribution = (
    Counter(
        document["attribution"][
            "authorship_assertion_source"
        ]
        for document
        in website_common_documents_v3
    )
)

fabricated_raw_evidence = [
    document["document_id"]
    for document
    in website_common_documents_v3
    if (
        document["attribution"][
            "byline_raw"
        ] is not None
        or document["attribution"][
            "editorial_credit_raw"
        ] is not None
    )
]


website_raw_path_missing = sum(
    not nonempty_text(
        document["lineage"].get(
            "raw_source_path"
        )
    )
    for document
    in website_common_documents_v3
)


document_ids_v3 = [
    document["document_id"]
    for document
    in common_documents_v3_candidate
]

duplicate_document_ids_v3 = (
    len(document_ids_v3)
    - len(set(document_ids_v3))
)


print("EXTENDED ATTRIBUTION CONTRACT VALIDATION")
print("=" * 80)

print("\nPOPULATION")
print("-" * 80)
print(
    "Unified documents:",
    len(common_documents_v3_candidate),
)
print(
    "Duplicate document_id:",
    duplicate_document_ids_v3,
)

print("\nWEBSITE AUTHORITY CLASSIFICATION")
print("-" * 80)
print(
    dict(website_authority_distribution)
)
print(
    "Attribution status:",
    dict(
        website_attribution_status_distribution
    ),
)
print(
    "Assertion source:",
    dict(
        website_assertion_source_distribution
    ),
)

print("\nEVIDENCE SAFETY")
print("-" * 80)
print(
    "Fabricated raw byline/editorial evidence:",
    len(fabricated_raw_evidence),
)
print(
    "Website records missing resolved raw path:",
    website_raw_path_missing,
)

print("\nFORMAL VALIDATION")
print("-" * 80)
print(
    "Schema failures:",
    len(schema_failures_v3),
)

if schema_failures_v3:
    print("\nFirst schema failures:")
    for failure in schema_failures_v3[:5]:
        print(
            json.dumps(
                failure,
                ensure_ascii=False,
                indent=2,
            )
        )


expected_website_authority = {
    "direct_author": 76,
    "editorial_supervision": 453,
}

extended_attribution_gate = (
    len(common_documents_v3_candidate)
    == 19456
    and duplicate_document_ids_v3 == 0
    and dict(
        website_authority_distribution
    )
    == expected_website_authority
    and website_attribution_status_distribution[
        "policy_validated"
    ] == len(website_data)
    and website_assertion_source_distribution[
        "accepted_pipeline_content_origin"
    ] == len(website_data)
    and len(fabricated_raw_evidence) == 0
    and website_raw_path_missing == 0
    and len(schema_failures_v3) == 0
)


print("\nEXTENDED ATTRIBUTION GATE")
print("=" * 80)

print(
    "Unified population:",
    "PASS"
    if len(common_documents_v3_candidate)
    == 19456
    else "FAIL",
)

print(
    "Website authority distribution:",
    "PASS"
    if dict(
        website_authority_distribution
    )
    == expected_website_authority
    else "FAIL",
)

print(
    "Raw evidence safety:",
    "PASS"
    if len(fabricated_raw_evidence) == 0
    else "FAIL",
)

print(
    "Website raw lineage path:",
    "PASS"
    if website_raw_path_missing == 0
    else "FAIL",
)

print(
    "Formal schema validation:",
    "PASS"
    if len(schema_failures_v3) == 0
    else "FAIL",
)

print(
    "EXTENDED ATTRIBUTION CONTRACT:",
    "PASS"
    if extended_attribution_gate
    else "FAIL",
)


assert extended_attribution_gate, (
    "Extended attribution contract validation failed."
)

EXTENDED ATTRIBUTION CONTRACT VALIDATION

POPULATION
--------------------------------------------------------------------------------
Unified documents: 19456
Duplicate document_id: 0

WEBSITE AUTHORITY CLASSIFICATION
--------------------------------------------------------------------------------
{'editorial_supervision': 453, 'direct_author': 76}
Attribution status: {'policy_validated': 529}
Assertion source: {'accepted_pipeline_content_origin': 529}

EVIDENCE SAFETY
--------------------------------------------------------------------------------
Fabricated raw byline/editorial evidence: 0
Website records missing resolved raw path: 0

FORMAL VALIDATION
--------------------------------------------------------------------------------
Schema failures: 0

EXTENDED ATTRIBUTION GATE
Unified population: PASS
Website authority distribution: PASS
Raw evidence safety: PASS
Website raw lineage path: PASS
Formal schema validation: PASS
EXTENDED ATTRIBUTION CONTRACT: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ثبت نسخه تکمیل‌شده Schema با Attribution رسمی Website</h3>

<p>
Common Document Contract اکنون علاوه بر Book Supplemental Text، سیاست رسمی Attribution وب‌سایت را نیز پوشش می‌دهد.
بنابراین Schema ذخیره‌شده روی Disk باید با این Contract جدید همگام شود.
</p>

<p>
در این مرحله فقط فایل <code>schemas/phase2/document.schema.json</code> Replace می‌شود.
پس از نوشتن، فایل دوباره از Disk خوانده شده و روی تمام ۱۹۴۵۶ Common Document اعتبارسنجی مستقل انجام می‌شود.
</p>

<h4>کنترل‌های الزامی</h4>

<ul>
<li>اعتبار JSON Schema Draft 2020-12</li>
<li>Validation تمام ۱۹۴۵۶ Common Document</li>
<li>پشتیبانی از ۱۲ Book Supplemental Text</li>
<li>پشتیبانی از Attribution کامل Website</li>
<li>توزیع ۷۶ <code>direct_author</code> و ۴۵۳ <code>editorial_supervision</code></li>
<li>عدم ثبت Byline یا Editorial Credit ساختگی</li>
<li>ثبت SHA-256 جدید Schema</li>
</ul>

<p>
هیچ Unified Dataset یا Report در این مرحله نوشته نمی‌شود و هیچ Git operation انجام نخواهد شد.
</p>

</div>

In [26]:
# Code Cell 26

final_schema_output_path = (
    project_root
    / "schemas"
    / "phase2"
    / "document.schema.json"
)

assert final_schema_output_path.is_file(), (
    "Persisted document schema is missing."
)

schema_v2_disk_sha256 = sha256_file(
    final_schema_output_path
)


schema_v3_text = json.dumps(
    document_schema_v3_candidate,
    ensure_ascii=False,
    indent=2,
    sort_keys=True,
) + "\n"


temporary_schema_v3_path = (
    final_schema_output_path
    .with_suffix(".schema.json.tmp")
)

temporary_schema_v3_path.write_text(
    schema_v3_text,
    encoding="utf-8",
)


with temporary_schema_v3_path.open(
    "r",
    encoding="utf-8",
) as handle:
    temporary_schema_v3 = json.load(handle)


Draft202012Validator.check_schema(
    temporary_schema_v3
)

temporary_validator_v3 = (
    Draft202012Validator(
        temporary_schema_v3
    )
)


temporary_failures_v3 = []

for document in common_documents_v3_candidate:
    errors = list(
        temporary_validator_v3.iter_errors(
            document
        )
    )

    if errors:
        temporary_failures_v3.append(
            {
                "document_id": document[
                    "document_id"
                ],
                "source_type": document[
                    "source_type"
                ],
                "errors": [
                    error.message
                    for error in errors[:5]
                ],
            }
        )


assert len(temporary_failures_v3) == 0, (
    "Temporary Schema v3 failed full-corpus validation."
)


temporary_schema_v3_path.replace(
    final_schema_output_path
)


with final_schema_output_path.open(
    "r",
    encoding="utf-8",
) as handle:
    persisted_schema_v3 = json.load(handle)


Draft202012Validator.check_schema(
    persisted_schema_v3
)

persisted_validator_v3 = (
    Draft202012Validator(
        persisted_schema_v3
    )
)


persisted_failures_v3 = []

for document in common_documents_v3_candidate:
    errors = list(
        persisted_validator_v3.iter_errors(
            document
        )
    )

    if errors:
        persisted_failures_v3.append(
            {
                "document_id": document[
                    "document_id"
                ],
                "source_type": document[
                    "source_type"
                ],
                "errors": [
                    error.message
                    for error in errors[:5]
                ],
            }
        )


schema_v3_disk_sha256 = sha256_file(
    final_schema_output_path
)

schema_changed_v3 = (
    schema_v3_disk_sha256
    != schema_v2_disk_sha256
)


website_persisted_failures = [
    failure
    for failure in persisted_failures_v3
    if failure["source_type"] == "website"
]

book_persisted_failures = [
    failure
    for failure in persisted_failures_v3
    if failure["source_type"] == "book"
]

transcript_persisted_failures = [
    failure
    for failure in persisted_failures_v3
    if failure["source_type"] == "transcript"
]


persisted_website_authority = Counter(
    document["attribution"][
        "content_authority_type"
    ]
    for document in common_documents_v3_candidate
    if document["source_type"] == "website"
)


persisted_fake_raw_evidence = [
    document["document_id"]
    for document in common_documents_v3_candidate
    if document["source_type"] == "website"
    and (
        document["attribution"][
            "byline_raw"
        ] is not None
        or document["attribution"][
            "editorial_credit_raw"
        ] is not None
    )
]


print("PERSISTED EXTENDED DOCUMENT SCHEMA")
print("=" * 80)

print(
    "Path:",
    final_schema_output_path.relative_to(
        project_root
    ),
)

print("\nHASH STATE")
print("-" * 80)
print(
    "Previous SHA-256:",
    schema_v2_disk_sha256,
)
print(
    "Updated SHA-256:",
    schema_v3_disk_sha256,
)
print(
    "Schema content changed:",
    schema_changed_v3,
)


print("\nFULL-CORPUS VALIDATION")
print("-" * 80)
print(
    "Documents validated:",
    len(common_documents_v3_candidate),
)
print(
    "Total failures:",
    len(persisted_failures_v3),
)
print(
    "Website failures:",
    len(website_persisted_failures),
)
print(
    "Book failures:",
    len(book_persisted_failures),
)
print(
    "Transcript failures:",
    len(transcript_persisted_failures),
)


print("\nWEBSITE ATTRIBUTION")
print("-" * 80)
print(
    "Authority distribution:",
    dict(persisted_website_authority),
)
print(
    "Fabricated raw evidence:",
    len(persisted_fake_raw_evidence),
)


if persisted_failures_v3:
    print("\nFirst schema failures:")
    for failure in persisted_failures_v3[:5]:
        print(
            json.dumps(
                failure,
                ensure_ascii=False,
                indent=2,
            )
        )


persisted_schema_v3_gate = (
    schema_changed_v3
    and len(common_documents_v3_candidate)
    == 19456
    and len(persisted_failures_v3) == 0
    and dict(persisted_website_authority)
    == {
        "direct_author": 76,
        "editorial_supervision": 453,
    }
    and len(persisted_fake_raw_evidence) == 0
)


print("\nPERSISTED EXTENDED SCHEMA GATE")
print("=" * 80)

print(
    "Schema replacement:",
    "PASS"
    if schema_changed_v3
    else "FAIL",
)

print(
    "Full-corpus validation:",
    "PASS"
    if len(persisted_failures_v3) == 0
    else "FAIL",
)

print(
    "Website attribution support:",
    "PASS"
    if dict(persisted_website_authority)
    == {
        "direct_author": 76,
        "editorial_supervision": 453,
    }
    else "FAIL",
)

print(
    "Evidence safety:",
    "PASS"
    if len(persisted_fake_raw_evidence) == 0
    else "FAIL",
)

print(
    "PERSISTED EXTENDED DOCUMENT SCHEMA:",
    "PASS"
    if persisted_schema_v3_gate
    else "FAIL",
)


assert schema_changed_v3

assert len(common_documents_v3_candidate) == 19456

assert len(persisted_failures_v3) == 0

assert dict(persisted_website_authority) == {
    "direct_author": 76,
    "editorial_supervision": 453,
}

assert len(persisted_fake_raw_evidence) == 0

PERSISTED EXTENDED DOCUMENT SCHEMA
Path: schemas/phase2/document.schema.json

HASH STATE
--------------------------------------------------------------------------------
Previous SHA-256: 174551afb77055a9012c7e29fa2044dd0dd67b78a54135a27947be1eb1f655de
Updated SHA-256: 3f577c71e8faa3b50c231bd4163cb5425158d0390e5fe5fcf39e00f530baa755
Schema content changed: True

FULL-CORPUS VALIDATION
--------------------------------------------------------------------------------
Documents validated: 19456
Total failures: 0
Website failures: 0
Book failures: 0
Transcript failures: 0

WEBSITE ATTRIBUTION
--------------------------------------------------------------------------------
Authority distribution: {'editorial_supervision': 453, 'direct_author': 76}
Fabricated raw evidence: 0

PERSISTED EXTENDED SCHEMA GATE
Schema replacement: PASS
Full-corpus validation: PASS
Website attribution support: PASS
Evidence safety: PASS
PERSISTED EXTENDED DOCUMENT SCHEMA: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ساخت و بازاعتبارسنجی مستقل Unified Document Dataset</h3>

<p>
Schema رسمی Common Document اکنون روی Disk ثبت و روی تمام ۱۹۴۵۶ Candidate معتبر شده است.
در این مرحله Artifact اصلی Unified Dataset ساخته می‌شود.
</p>

<h4>خروجی</h4>

<p><code>data/phase2/processed/documents_unified_v1.jsonl</code></p>

<p>
Dataset شامل فقط واحدهای متنی پذیرفته‌شده است:
Website Article،
Book Paragraph،
Book Footnote
و Transcript Utterance.
</p>

<p>
۱۵ Chart و ۱ Diagram کتاب به متن مصنوعی تبدیل نمی‌شوند و در این JSONL وارد نخواهند شد.
وجود و Lineage آن‌ها در مراحل گزارش‌دهی NB07 جداگانه ثبت خواهد شد.
</p>

<h4>روش Write</h4>

<ul>
<li>ابتدا Dataset به فایل موقت نوشته می‌شود.</li>
<li>فایل موقت دوباره از Disk به‌صورت JSONL Parse می‌شود.</li>
<li>تمام رکوردها با Schema ذخیره‌شده روی Disk Validation می‌شوند.</li>
<li>یکتایی <code>document_id</code> و Hash متن کنترل می‌شود.</li>
<li>Lineage Raw Path و Raw Hash برای تمام رکوردها دوباره بررسی می‌شود.</li>
<li>فقط پس از PASS کامل، فایل موقت جایگزین Artifact نهایی می‌شود.</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>تعداد رکوردها دقیقاً برابر Candidate معتبر فعلی باشد.</li>
<li>Schema Error برابر صفر باشد.</li>
<li>Duplicate <code>document_id</code> برابر صفر باشد.</li>
<li>Text Hash mismatch برابر صفر باشد.</li>
<li>Broken Raw Lineage برابر صفر باشد.</li>
<li>Structured Assetهای غیرمتنی به Dataset وارد نشده باشند.</li>
<li>SHA-256 Artifact نهایی ثبت شود.</li>
</ul>

</div>

In [27]:
# Code Cell 27

unified_output_path = (
    project_root
    / "data"
    / "phase2"
    / "processed"
    / "documents_unified_v1.jsonl"
)

unified_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

temporary_unified_path = (
    unified_output_path
    .with_suffix(".jsonl.tmp")
)


with temporary_unified_path.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as handle:
    for document in common_documents_v3_candidate:
        handle.write(
            json.dumps(
                document,
                ensure_ascii=False,
                sort_keys=True,
                separators=(",", ":"),
            )
        )
        handle.write("\n")


persisted_schema_for_dataset = json.loads(
    final_schema_output_path.read_text(
        encoding="utf-8"
    )
)

Draft202012Validator.check_schema(
    persisted_schema_for_dataset
)

dataset_validator = Draft202012Validator(
    persisted_schema_for_dataset
)


reloaded_unified_documents = []

with temporary_unified_path.open(
    "r",
    encoding="utf-8",
) as handle:
    for line_number, line in enumerate(
        handle,
        start=1,
    ):
        stripped = line.strip()

        if not stripped:
            raise ValueError(
                f"Blank JSONL line at {line_number}"
            )

        reloaded_unified_documents.append(
            json.loads(stripped)
        )


dataset_schema_failures = []
dataset_text_hash_failures = []
dataset_lineage_failures = []

for document in reloaded_unified_documents:
    schema_errors = list(
        dataset_validator.iter_errors(
            document
        )
    )

    if schema_errors:
        dataset_schema_failures.append(
            {
                "document_id": document.get(
                    "document_id"
                ),
                "errors": [
                    error.message
                    for error
                    in schema_errors[:5]
                ],
            }
        )

    actual_text_hash = hashlib.sha256(
        document["text"].encode("utf-8")
    ).hexdigest()

    if (
        actual_text_hash
        != document["text_sha256"]
    ):
        dataset_text_hash_failures.append(
            document["document_id"]
        )

    raw_source_path = (
        document["lineage"]
        .get("raw_source_path")
    )

    raw_source_hash = (
        document["lineage"]
        .get("raw_source_hash")
    )

    if (
        not nonempty_text(raw_source_path)
        or not nonempty_text(raw_source_hash)
    ):
        dataset_lineage_failures.append(
            {
                "document_id": document[
                    "document_id"
                ],
                "reason": (
                    "missing_raw_path_or_hash"
                ),
            }
        )
        continue

    resolved_raw_path = (
        project_root
        / raw_source_path
    )

    if not resolved_raw_path.is_file():
        dataset_lineage_failures.append(
            {
                "document_id": document[
                    "document_id"
                ],
                "reason": "raw_file_missing",
                "path": raw_source_path,
            }
        )
        continue

    actual_raw_hash = sha256_file(
        resolved_raw_path
    )

    if actual_raw_hash != raw_source_hash:
        dataset_lineage_failures.append(
            {
                "document_id": document[
                    "document_id"
                ],
                "reason": "raw_hash_mismatch",
                "path": raw_source_path,
                "expected": raw_source_hash,
                "actual": actual_raw_hash,
            }
        )


reloaded_document_ids = [
    document["document_id"]
    for document in reloaded_unified_documents
]

dataset_duplicate_ids = (
    len(reloaded_document_ids)
    - len(set(reloaded_document_ids))
)


dataset_source_distribution = Counter(
    document["source_type"]
    for document in reloaded_unified_documents
)

dataset_book_unit_distribution = Counter(
    document["source_metadata"].get(
        "book_unit_type"
    )
    for document in reloaded_unified_documents
    if document["source_type"] == "book"
)

dataset_website_authority_distribution = Counter(
    document["attribution"].get(
        "content_authority_type"
    )
    for document in reloaded_unified_documents
    if document["source_type"] == "website"
)


structured_asset_ids = {
    record["content_id"]
    for record in structured_asset_candidates
}

dataset_structured_asset_collisions = sorted(
    set(reloaded_document_ids)
    & structured_asset_ids
)


expected_dataset_population = len(
    common_documents_v3_candidate
)

dataset_gate = (
    len(reloaded_unified_documents)
    == expected_dataset_population
    and dataset_duplicate_ids == 0
    and len(dataset_schema_failures) == 0
    and len(dataset_text_hash_failures) == 0
    and len(dataset_lineage_failures) == 0
    and len(dataset_structured_asset_collisions) == 0
)


print("TEMPORARY UNIFIED DOCUMENT DATASET VALIDATION")
print("=" * 80)

print("\nPOPULATION")
print("-" * 80)
print(
    "Expected records:",
    expected_dataset_population,
)
print(
    "Reloaded records:",
    len(reloaded_unified_documents),
)
print(
    "Source distribution:",
    dict(dataset_source_distribution),
)
print(
    "Book unit distribution:",
    dict(dataset_book_unit_distribution),
)

print("\nWEBSITE ATTRIBUTION")
print("-" * 80)
print(
    "Authority distribution:",
    dict(
        dataset_website_authority_distribution
    ),
)

print("\nINTEGRITY")
print("-" * 80)
print(
    "Duplicate document_id:",
    dataset_duplicate_ids,
)
print(
    "Schema failures:",
    len(dataset_schema_failures),
)
print(
    "Text hash failures:",
    len(dataset_text_hash_failures),
)
print(
    "Broken raw lineage:",
    len(dataset_lineage_failures),
)
print(
    "Structured asset collisions:",
    len(dataset_structured_asset_collisions),
)


if dataset_schema_failures:
    print("\nFirst schema failures:")
    for failure in dataset_schema_failures[:5]:
        print(
            json.dumps(
                failure,
                ensure_ascii=False,
                indent=2,
            )
        )

if dataset_lineage_failures:
    print("\nFirst lineage failures:")
    for failure in dataset_lineage_failures[:5]:
        print(
            json.dumps(
                failure,
                ensure_ascii=False,
                indent=2,
            )
        )


print("\nTEMPORARY DATASET GATE")
print("=" * 80)
print(
    "Population:",
    "PASS"
    if len(reloaded_unified_documents)
    == expected_dataset_population
    else "FAIL",
)
print(
    "Global document_id uniqueness:",
    "PASS"
    if dataset_duplicate_ids == 0
    else "FAIL",
)
print(
    "Persisted-schema validation:",
    "PASS"
    if len(dataset_schema_failures) == 0
    else "FAIL",
)
print(
    "Text hash integrity:",
    "PASS"
    if len(dataset_text_hash_failures) == 0
    else "FAIL",
)
print(
    "Raw lineage integrity:",
    "PASS"
    if len(dataset_lineage_failures) == 0
    else "FAIL",
)
print(
    "Structured assets excluded:",
    "PASS"
    if len(dataset_structured_asset_collisions) == 0
    else "FAIL",
)
print(
    "TEMPORARY UNIFIED DATASET:",
    "PASS" if dataset_gate else "FAIL",
)


assert dataset_gate, (
    "Temporary Unified Document Dataset validation failed."
)


temporary_unified_path.replace(
    unified_output_path
)


final_unified_sha256 = sha256_file(
    unified_output_path
)

final_unified_size = (
    unified_output_path.stat().st_size
)


final_reloaded_count = 0

with unified_output_path.open(
    "r",
    encoding="utf-8",
) as handle:
    for line in handle:
        if line.strip():
            json.loads(line)
            final_reloaded_count += 1


print("\nPERSISTED UNIFIED DOCUMENT DATASET")
print("=" * 80)
print(
    "Path:",
    unified_output_path.relative_to(
        project_root
    ),
)
print(
    "Records:",
    final_reloaded_count,
)
print(
    "Size bytes:",
    final_unified_size,
)
print(
    "SHA-256:",
    final_unified_sha256,
)
print(
    "Persisted JSONL reload:",
    "PASS"
    if final_reloaded_count
    == expected_dataset_population
    else "FAIL",
)


assert unified_output_path.is_file()

assert (
    final_reloaded_count
    == expected_dataset_population
)

TEMPORARY UNIFIED DOCUMENT DATASET VALIDATION

POPULATION
--------------------------------------------------------------------------------
Expected records: 19456
Reloaded records: 19456
Source distribution: {'website': 529, 'book': 2701, 'transcript': 16226}
Book unit distribution: {'paragraph': 2689, 'supplemental_text': 12}

WEBSITE ATTRIBUTION
--------------------------------------------------------------------------------
Authority distribution: {'editorial_supervision': 453, 'direct_author': 76}

INTEGRITY
--------------------------------------------------------------------------------
Duplicate document_id: 0
Schema failures: 0
Text hash failures: 0
Broken raw lineage: 0
Structured asset collisions: 0

TEMPORARY DATASET GATE
Population: PASS
Global document_id uniqueness: PASS
Persisted-schema validation: PASS
Text hash integrity: PASS
Raw lineage integrity: PASS
Structured assets excluded: PASS
TEMPORARY UNIFIED DATASET: PASS

PERSISTED UNIFIED DOCUMENT DATASET
Path: data/phase

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ساخت گزارش رسمی Unified Schema، Lineage و Attribution Coverage</h3>

<p>
Unified Document Dataset با ۱۹۴۵۶ رکورد با موفقیت ساخته و از روی Disk بازاعتبارسنجی شد.
اکنون وضعیت Schema، Lineage، Attribution، Rights و Structured Assetهای خارج از Corpus متنی در یک Report نسخه‌دار ثبت می‌شود.
</p>

<h4>موارد ثبت‌شده در Report</h4>

<ul>
<li>مسیر، SHA-256 و تعداد رکورد Unified Dataset</li>
<li>مسیر و SHA-256 Schema رسمی</li>
<li>توزیع Website، Book و Transcript</li>
<li>توزیع Paragraph و Supplemental Text کتاب</li>
<li>پوشش کامل <code>document_id → source_id → raw_content_hash</code></li>
<li>وضعیت Raw path و Broken Lineage</li>
<li>Attribution رسمی Website شامل Direct Author و Editorial Supervision</li>
<li>پوشش Speaker Attribution در Transcript بدون حدس Speaker ناشناخته</li>
<li>وضعیت Rights هر Source بدون ساخت Evidence مفقود</li>
<li>ثبت ۱۵ Chart و ۱ Diagram به‌عنوان Structured Asset خارج از Text Corpus</li>
<li>تأکید بر اینکه Persian normalization، Deduplication و Chunking در NB07 اجرا نشده‌اند</li>
</ul>

<h4>معیار PASS</h4>

<p>
Report باید از داده واقعی Artifact ساخته شود،
Broken Lineage و Invalid Attribution برابر صفر باشند،
جمع رکوردها با Unified Dataset منطبق باشد
و پس از Write دوباره از Disk Parse و Hash شود.
</p>

</div>

In [28]:
# Code Cell 28

report_output_path = (
    project_root
    / "reports"
    / "phase2"
    / "unified_schema_lineage_report_v1.json"
)

report_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


unified_documents_for_report = (
    reloaded_unified_documents
)


source_distribution_report = Counter(
    document["source_type"]
    for document in unified_documents_for_report
)


book_unit_distribution_report = Counter(
    document["source_metadata"].get(
        "book_unit_type"
    )
    for document in unified_documents_for_report
    if document["source_type"] == "book"
)


website_authority_distribution_report = Counter(
    document["attribution"].get(
        "content_authority_type"
    )
    for document in unified_documents_for_report
    if document["source_type"] == "website"
)


website_origin_distribution_report = Counter(
    document.get("content_origin")
    for document in unified_documents_for_report
    if document["source_type"] == "website"
)


rights_status_distribution_report = {}

for source_type in (
    "website",
    "book",
    "transcript",
):
    rights_status_distribution_report[
        source_type
    ] = dict(
        Counter(
            document["rights"].get("status")
            for document
            in unified_documents_for_report
            if document["source_type"]
            == source_type
        )
    )


transcript_documents_for_report = [
    document
    for document in unified_documents_for_report
    if document["source_type"] == "transcript"
]


transcript_speaker_canonical_present = sum(
    nonempty_text(
        document["attribution"].get(
            "speaker"
        )
    )
    for document in transcript_documents_for_report
)


transcript_speaker_canonical_missing = (
    len(transcript_documents_for_report)
    - transcript_speaker_canonical_present
)


transcript_effective_label_present = sum(
    nonempty_text(
        document["source_metadata"]
        .get("speaker", {})
        .get("effective", {})
        .get("label")
    )
    for document in transcript_documents_for_report
)


transcript_human_overlay_count = sum(
    document["source_metadata"]
    .get("speaker", {})
    .get("human_attribution_overlay")
    is not None
    for document in transcript_documents_for_report
)


transcript_source_timestamp_annotation_records = sum(
    bool(
        document["source_metadata"]
        .get("timestamps", {})
        .get("source_annotations")
    )
    for document in transcript_documents_for_report
)


lineage_identity_failures = []

for document in unified_documents_for_report:
    if not all(
        nonempty_text(document.get(field))
        for field in (
            "document_id",
            "source_id",
            "raw_content_hash",
        )
    ):
        lineage_identity_failures.append(
            document["document_id"]
        )
        continue

    lineage = document.get(
        "lineage",
        {},
    )

    if (
        lineage.get("raw_source_hash")
        != document.get("raw_content_hash")
    ):
        lineage_identity_failures.append(
            document["document_id"]
        )


website_invalid_attribution = []

for document in unified_documents_for_report:
    if document["source_type"] != "website":
        continue

    attribution = document["attribution"]
    origin = document["content_origin"]

    if origin == "primary_author":
        valid = (
            attribution.get(
                "content_authority_type"
            ) == "direct_author"
            and nonempty_text(
                attribution.get(
                    "credited_author_name"
                )
            )
            and attribution.get(
                "editorial_supervisor_name"
            ) is None
        )

    elif origin == "supervised":
        valid = (
            attribution.get(
                "content_authority_type"
            ) == "editorial_supervision"
            and nonempty_text(
                attribution.get(
                    "credited_author_name"
                )
            )
            and nonempty_text(
                attribution.get(
                    "editorial_supervisor_name"
                )
            )
        )

    else:
        valid = False

    if not valid:
        website_invalid_attribution.append(
            document["document_id"]
        )


structured_asset_distribution_report = dict(
    Counter(
        record["content_type"]
        for record in structured_asset_candidates
    )
)


structured_asset_records_report = [
    {
        "content_id": record["content_id"],
        "content_type": record["content_type"],
        "work_id": record["work_id"],
        "edition_id": record["edition_id"],
        "parent_paragraph_id": record[
            "parent_paragraph_id"
        ],
        "source_locator": deepcopy(
            record["source_locator"]
        ),
        "payload_sha256": record[
            "payload_sha256"
        ],
        "text_corpus_status": (
            "PRESERVED_AS_STRUCTURED_ASSET_NOT_TEXT_DOCUMENT"
        ),
    }
    for record in structured_asset_candidates
]


report_payload = {
    "report_type": (
        "phase2_unified_schema_lineage_report"
    ),
    "report_version": "v1",

    "schema": {
        "path": (
            final_schema_output_path
            .relative_to(project_root)
            .as_posix()
        ),
        "sha256": sha256_file(
            final_schema_output_path
        ),
        "draft": "2020-12",
        "validation_status": "PASS",
    },

    "unified_dataset": {
        "path": (
            unified_output_path
            .relative_to(project_root)
            .as_posix()
        ),
        "sha256": final_unified_sha256,
        "record_count": len(
            unified_documents_for_report
        ),
        "source_distribution": dict(
            source_distribution_report
        ),
        "book_unit_distribution": dict(
            book_unit_distribution_report
        ),
    },

    "lineage": {
        "required_chain": (
            "document_id -> source_id -> raw_content_hash"
        ),
        "records_checked": len(
            unified_documents_for_report
        ),
        "identity_chain_failures": len(
            lineage_identity_failures
        ),
        "broken_raw_lineage": len(
            dataset_lineage_failures
        ),
        "status": (
            "PASS"
            if (
                len(lineage_identity_failures)
                == 0
                and len(
                    dataset_lineage_failures
                ) == 0
            )
            else "FAIL"
        ),
    },

    "website_attribution": {
        "content_origin_distribution": dict(
            website_origin_distribution_report
        ),
        "content_authority_distribution": dict(
            website_authority_distribution_report
        ),
        "attribution_status": {
            "policy_validated": sum(
                document["attribution"].get(
                    "attribution_status"
                )
                == "policy_validated"
                for document
                in unified_documents_for_report
                if document["source_type"]
                == "website"
            )
        },
        "raw_byline_available": False,
        "raw_editorial_credit_available": False,
        "assertion_source": (
            "accepted_pipeline_content_origin"
        ),
        "invalid_attribution_count": len(
            website_invalid_attribution
        ),
        "status": (
            "PASS"
            if len(
                website_invalid_attribution
            ) == 0
            else "FAIL"
        ),
    },

    "transcript_attribution": {
        "record_count": len(
            transcript_documents_for_report
        ),
        "effective_canonical_speaker_present": (
            transcript_speaker_canonical_present
        ),
        "effective_canonical_speaker_missing": (
            transcript_speaker_canonical_missing
        ),
        "effective_label_present": (
            transcript_effective_label_present
        ),
        "human_attribution_overlay_records": (
            transcript_human_overlay_count
        ),
        "source_timestamp_annotation_records": (
            transcript_source_timestamp_annotation_records
        ),
        "unknown_speaker_inference": (
            "FORBIDDEN_AND_NOT_PERFORMED"
        ),
    },

    "rights": {
        "status_distribution_by_source": (
            rights_status_distribution_report
        ),
        "missing_rights_evidence_policy": (
            "PRESERVE_NULL_OR_EVIDENCE_REQUIRED_DO_NOT_INVENT"
        ),
    },

    "book_structured_assets": {
        "record_count": len(
            structured_asset_candidates
        ),
        "distribution": (
            structured_asset_distribution_report
        ),
        "text_corpus_inclusion_count": len(
            dataset_structured_asset_collisions
        ),
        "records": (
            structured_asset_records_report
        ),
    },

    "processing_boundaries": {
        "persian_normalization": (
            "NOT_PERFORMED_IN_NB07"
        ),
        "deduplication": (
            "NOT_PERFORMED_IN_NB07"
        ),
        "chunking": (
            "NOT_PERFORMED_IN_NB07"
        ),
        "embedding": (
            "NOT_PERFORMED_IN_NB07"
        ),
        "indexing": (
            "NOT_PERFORMED_IN_NB07"
        ),
    },

    "quality_gates": {
        "duplicate_document_id": (
            dataset_duplicate_ids
        ),
        "schema_failures": len(
            dataset_schema_failures
        ),
        "text_hash_failures": len(
            dataset_text_hash_failures
        ),
        "broken_raw_lineage": len(
            dataset_lineage_failures
        ),
        "lineage_identity_failures": len(
            lineage_identity_failures
        ),
        "invalid_website_attribution": len(
            website_invalid_attribution
        ),
        "structured_asset_text_collisions": (
            len(
                dataset_structured_asset_collisions
            )
        ),
    },
}


report_gate = (
    len(unified_documents_for_report) == 19456
    and len(lineage_identity_failures) == 0
    and len(dataset_lineage_failures) == 0
    and len(website_invalid_attribution) == 0
    and dataset_duplicate_ids == 0
    and len(dataset_schema_failures) == 0
    and len(dataset_text_hash_failures) == 0
    and len(
        dataset_structured_asset_collisions
    ) == 0
)


print("UNIFIED SCHEMA / LINEAGE / ATTRIBUTION REPORT")
print("=" * 80)

print("\nDATASET")
print("-" * 80)
print(
    "Records:",
    report_payload[
        "unified_dataset"
    ]["record_count"],
)
print(
    "Source distribution:",
    report_payload[
        "unified_dataset"
    ]["source_distribution"],
)
print(
    "Book units:",
    report_payload[
        "unified_dataset"
    ]["book_unit_distribution"],
)

print("\nLINEAGE")
print("-" * 80)
print(
    "Identity-chain failures:",
    len(lineage_identity_failures),
)
print(
    "Broken raw lineage:",
    len(dataset_lineage_failures),
)

print("\nWEBSITE ATTRIBUTION")
print("-" * 80)
print(
    "Authority distribution:",
    dict(
        website_authority_distribution_report
    ),
)
print(
    "Invalid attribution:",
    len(website_invalid_attribution),
)

print("\nTRANSCRIPT ATTRIBUTION")
print("-" * 80)
print(
    "Canonical speaker present:",
    transcript_speaker_canonical_present,
)
print(
    "Canonical speaker missing:",
    transcript_speaker_canonical_missing,
)
print(
    "Effective label present:",
    transcript_effective_label_present,
)
print(
    "Human overlays:",
    transcript_human_overlay_count,
)
print(
    "Timestamp annotation records:",
    transcript_source_timestamp_annotation_records,
)

print("\nSTRUCTURED BOOK ASSETS")
print("-" * 80)
print(
    "Preserved outside text corpus:",
    len(structured_asset_candidates),
)
print(
    "Distribution:",
    structured_asset_distribution_report,
)
print(
    "Accidentally included:",
    len(dataset_structured_asset_collisions),
)

print("\nREPORT GATE")
print("=" * 80)
print(
    "REPORT CANDIDATE:",
    "PASS" if report_gate else "FAIL",
)

assert report_gate, (
    "Unified schema/lineage report candidate failed."
)


temporary_report_path = (
    report_output_path
    .with_suffix(".json.tmp")
)

temporary_report_path.write_text(
    json.dumps(
        report_payload,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


with temporary_report_path.open(
    "r",
    encoding="utf-8",
) as handle:
    reloaded_report = json.load(handle)


assert (
    reloaded_report["unified_dataset"][
        "record_count"
    ]
    == 19456
)

assert (
    reloaded_report["lineage"][
        "status"
    ]
    == "PASS"
)

assert (
    reloaded_report[
        "website_attribution"
    ]["status"]
    == "PASS"
)


temporary_report_path.replace(
    report_output_path
)

report_sha256 = sha256_file(
    report_output_path
)


print("\nPERSISTED REPORT")
print("=" * 80)
print(
    "Path:",
    report_output_path.relative_to(
        project_root
    ),
)
print(
    "SHA-256:",
    report_sha256,
)
print(
    "JSON reload:",
    "PASS",
)
print(
    "Lineage status:",
    reloaded_report[
        "lineage"
    ]["status"],
)
print(
    "Website attribution status:",
    reloaded_report[
        "website_attribution"
    ]["status"],
)

UNIFIED SCHEMA / LINEAGE / ATTRIBUTION REPORT

DATASET
--------------------------------------------------------------------------------
Records: 19456
Source distribution: {'website': 529, 'book': 2701, 'transcript': 16226}
Book units: {'paragraph': 2689, 'supplemental_text': 12}

LINEAGE
--------------------------------------------------------------------------------
Identity-chain failures: 0
Broken raw lineage: 0

WEBSITE ATTRIBUTION
--------------------------------------------------------------------------------
Authority distribution: {'editorial_supervision': 453, 'direct_author': 76}
Invalid attribution: 0

TRANSCRIPT ATTRIBUTION
--------------------------------------------------------------------------------
Canonical speaker present: 11210
Canonical speaker missing: 5016
Effective label present: 13872
Human overlays: 2
Timestamp annotation records: 16

STRUCTURED BOOK ASSETS
--------------------------------------------------------------------------------
Preserved outside text

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ساخت Acceptance Artifact نهایی Notebook 07</h3>

<p>
Schema، Unified Dataset و Lineage/Attribution Report همگی ساخته و مستقل بازاعتبارسنجی شده‌اند.
اکنون وضعیت نهایی Notebook 07 در یک Acceptance Artifact نسخه‌دار ثبت می‌شود.
</p>

<h4>خروجی</h4>

<p><code>reports/phase2/unified_schema_lineage_acceptance_v1.json</code></p>

<h4>Acceptance شامل</h4>

<ul>
<li>مسیر و SHA-256 تمام Artifactهای رسمی NB07</li>
<li>تعداد نهایی Unified Documents</li>
<li>توزیع Sourceها و Book unitها</li>
<li>وضعیت Schema Validation</li>
<li>وضعیت Document ID uniqueness</li>
<li>وضعیت Text Hash و Raw Lineage</li>
<li>وضعیت Website Attribution</li>
<li>وضعیت Transcript Attribution preservation</li>
<li>ثبت Structured Assetهای غیرمتنی کتاب</li>
<li>مرزهای پردازشی NB07 و عدم اجرای Normalization، Deduplication و Chunking</li>
</ul>

<h4>معیار PASS</h4>

<p>
تمام Artifactها باید روی Disk موجود باشند،
Hashهای آن‌ها دوباره محاسبه شوند،
تمام Gateهای بحرانی صفر Failure داشته باشند
و Acceptance Status برابر <code>PASS</code> شود.
</p>

</div>

In [29]:
# Code Cell 29

acceptance_output_path = (
    project_root
    / "reports"
    / "phase2"
    / "unified_schema_lineage_acceptance_v1.json"
)


official_nb07_artifacts = {
    "document_schema": final_schema_output_path,
    "unified_documents": unified_output_path,
    "lineage_report": report_output_path,
}


artifact_state = {}

for name, path in official_nb07_artifacts.items():
    artifact_state[name] = {
        "path": path.relative_to(
            project_root
        ).as_posix(),
        "exists": path.is_file(),
        "sha256": (
            sha256_file(path)
            if path.is_file()
            else None
        ),
        "size_bytes": (
            path.stat().st_size
            if path.is_file()
            else None
        ),
    }


all_artifacts_exist = all(
    state["exists"]
    for state in artifact_state.values()
)


# Independent reload of persisted Schema.
with final_schema_output_path.open(
    "r",
    encoding="utf-8",
) as handle:
    acceptance_schema = json.load(handle)

Draft202012Validator.check_schema(
    acceptance_schema
)

acceptance_validator = Draft202012Validator(
    acceptance_schema
)


# Independent reload of persisted Unified JSONL.
acceptance_documents = []

with unified_output_path.open(
    "r",
    encoding="utf-8",
) as handle:
    for line_number, line in enumerate(
        handle,
        start=1,
    ):
        stripped = line.strip()

        if not stripped:
            raise ValueError(
                f"Blank Unified JSONL line: "
                f"{line_number}"
            )

        acceptance_documents.append(
            json.loads(stripped)
        )


acceptance_schema_failures = []
acceptance_text_hash_failures = []
acceptance_lineage_failures = []

for document in acceptance_documents:
    errors = list(
        acceptance_validator.iter_errors(
            document
        )
    )

    if errors:
        acceptance_schema_failures.append(
            document["document_id"]
        )

    computed_text_hash = hashlib.sha256(
        document["text"].encode("utf-8")
    ).hexdigest()

    if (
        computed_text_hash
        != document["text_sha256"]
    ):
        acceptance_text_hash_failures.append(
            document["document_id"]
        )

    raw_path_value = (
        document["lineage"]
        .get("raw_source_path")
    )

    raw_hash_value = (
        document["lineage"]
        .get("raw_source_hash")
    )

    if (
        not nonempty_text(raw_path_value)
        or not nonempty_text(raw_hash_value)
    ):
        acceptance_lineage_failures.append(
            document["document_id"]
        )
        continue

    raw_path = project_root / raw_path_value

    if (
        not raw_path.is_file()
        or sha256_file(raw_path)
        != raw_hash_value
        or document["raw_content_hash"]
        != raw_hash_value
    ):
        acceptance_lineage_failures.append(
            document["document_id"]
        )


acceptance_document_ids = [
    document["document_id"]
    for document in acceptance_documents
]

acceptance_duplicate_ids = (
    len(acceptance_document_ids)
    - len(set(acceptance_document_ids))
)


acceptance_source_distribution = Counter(
    document["source_type"]
    for document in acceptance_documents
)

acceptance_book_units = Counter(
    document["source_metadata"].get(
        "book_unit_type"
    )
    for document in acceptance_documents
    if document["source_type"] == "book"
)

acceptance_website_authority = Counter(
    document["attribution"].get(
        "content_authority_type"
    )
    for document in acceptance_documents
    if document["source_type"] == "website"
)


acceptance_invalid_website_attribution = []

for document in acceptance_documents:
    if document["source_type"] != "website":
        continue

    attribution = document["attribution"]
    origin = document["content_origin"]

    if origin == "primary_author":
        valid = (
            attribution.get(
                "content_authority_type"
            ) == "direct_author"
            and nonempty_text(
                attribution.get(
                    "credited_author_name"
                )
            )
            and attribution.get(
                "editorial_supervisor_name"
            ) is None
        )

    elif origin == "supervised":
        valid = (
            attribution.get(
                "content_authority_type"
            ) == "editorial_supervision"
            and nonempty_text(
                attribution.get(
                    "credited_author_name"
                )
            )
            and nonempty_text(
                attribution.get(
                    "editorial_supervisor_name"
                )
            )
        )

    else:
        valid = False

    if not valid:
        acceptance_invalid_website_attribution.append(
            document["document_id"]
        )


acceptance_transcript_documents = [
    document
    for document in acceptance_documents
    if document["source_type"] == "transcript"
]

acceptance_transcript_canonical_missing = sum(
    not nonempty_text(
        document["attribution"].get(
            "speaker"
        )
    )
    for document
    in acceptance_transcript_documents
)

acceptance_transcript_human_overlays = sum(
    document["source_metadata"]
    .get("speaker", {})
    .get("human_attribution_overlay")
    is not None
    for document
    in acceptance_transcript_documents
)

acceptance_timestamp_annotation_records = sum(
    bool(
        document["source_metadata"]
        .get("timestamps", {})
        .get("source_annotations")
    )
    for document
    in acceptance_transcript_documents
)


structured_assets_in_text_dataset = (
    set(acceptance_document_ids)
    & {
        record["content_id"]
        for record in structured_asset_candidates
    }
)


critical_gate = (
    all_artifacts_exist
    and len(acceptance_documents) == 19456
    and acceptance_duplicate_ids == 0
    and len(acceptance_schema_failures) == 0
    and len(acceptance_text_hash_failures) == 0
    and len(acceptance_lineage_failures) == 0
    and len(
        acceptance_invalid_website_attribution
    ) == 0
    and dict(
        acceptance_website_authority
    ) == {
        "direct_author": 76,
        "editorial_supervision": 453,
    }
    and len(
        structured_assets_in_text_dataset
    ) == 0
)


acceptance_payload = {
    "acceptance_type": (
        "phase2_notebook_07_unified_schema_and_lineage"
    ),
    "acceptance_version": "v1",

    "notebook": (
        "notebooks/phase2/"
        "07_unified_schema_and_lineage.ipynb"
    ),

    "artifacts": artifact_state,

    "population": {
        "record_count": len(
            acceptance_documents
        ),
        "source_distribution": dict(
            acceptance_source_distribution
        ),
        "book_unit_distribution": dict(
            acceptance_book_units
        ),
        "structured_book_assets_outside_text_corpus": {
            "record_count": len(
                structured_asset_candidates
            ),
            "distribution": dict(
                Counter(
                    record["content_type"]
                    for record
                    in structured_asset_candidates
                )
            ),
        },
    },

    "website_attribution": {
        "authority_distribution": dict(
            acceptance_website_authority
        ),
        "invalid_attribution_count": len(
            acceptance_invalid_website_attribution
        ),
        "policy_source": (
            "accepted_pipeline_content_origin"
        ),
        "raw_byline_invented": False,
        "raw_editorial_credit_invented": False,
    },

    "transcript_attribution": {
        "record_count": len(
            acceptance_transcript_documents
        ),
        "canonical_speaker_missing": (
            acceptance_transcript_canonical_missing
        ),
        "human_attribution_overlay_records": (
            acceptance_transcript_human_overlays
        ),
        "source_timestamp_annotation_records": (
            acceptance_timestamp_annotation_records
        ),
        "unknown_speaker_inference": (
            "FORBIDDEN_AND_NOT_PERFORMED"
        ),
    },

    "quality_gates": {
        "duplicate_document_id": (
            acceptance_duplicate_ids
        ),
        "schema_failures": len(
            acceptance_schema_failures
        ),
        "text_hash_failures": len(
            acceptance_text_hash_failures
        ),
        "broken_lineage": len(
            acceptance_lineage_failures
        ),
        "invalid_website_attribution": len(
            acceptance_invalid_website_attribution
        ),
        "structured_asset_text_collisions": len(
            structured_assets_in_text_dataset
        ),
    },

    "processing_boundaries": {
        "persian_normalization": (
            "NOT_PERFORMED_IN_NB07"
        ),
        "deduplication": (
            "NOT_PERFORMED_IN_NB07"
        ),
        "chunking": (
            "NOT_PERFORMED_IN_NB07"
        ),
        "embedding": (
            "NOT_PERFORMED_IN_NB07"
        ),
        "indexing": (
            "NOT_PERFORMED_IN_NB07"
        ),
    },

    "acceptance_status": (
        "PASS"
        if critical_gate
        else "FAIL"
    ),
}


print("NB07 FINAL ACCEPTANCE CANDIDATE")
print("=" * 80)

print("\nARTIFACTS")
print("-" * 80)

for name, state in artifact_state.items():
    print(
        name,
        "→",
        state["path"],
    )
    print(
        "   exists:",
        state["exists"],
    )
    print(
        "   sha256:",
        state["sha256"],
    )


print("\nPOPULATION")
print("-" * 80)
print(
    "Documents:",
    len(acceptance_documents),
)
print(
    "Source distribution:",
    dict(
        acceptance_source_distribution
    ),
)
print(
    "Book units:",
    dict(acceptance_book_units),
)


print("\nQUALITY GATES")
print("-" * 80)
print(
    "Duplicate document_id:",
    acceptance_duplicate_ids,
)
print(
    "Schema failures:",
    len(acceptance_schema_failures),
)
print(
    "Text hash failures:",
    len(acceptance_text_hash_failures),
)
print(
    "Broken lineage:",
    len(acceptance_lineage_failures),
)
print(
    "Invalid Website attribution:",
    len(
        acceptance_invalid_website_attribution
    ),
)
print(
    "Structured asset collisions:",
    len(
        structured_assets_in_text_dataset
    ),
)


print("\nTRANSCRIPT PRESERVATION")
print("-" * 80)
print(
    "Canonical speaker missing:",
    acceptance_transcript_canonical_missing,
)
print(
    "Human overlays:",
    acceptance_transcript_human_overlays,
)
print(
    "Timestamp annotation records:",
    acceptance_timestamp_annotation_records,
)
print(
    "Unknown speaker inference:",
    "NOT PERFORMED",
)


print("\nFINAL ACCEPTANCE GATE")
print("=" * 80)
print(
    "NB07 ACCEPTANCE:",
    "PASS" if critical_gate else "FAIL",
)

assert critical_gate, (
    "Notebook 07 final acceptance candidate failed."
)


temporary_acceptance_path = (
    acceptance_output_path
    .with_suffix(".json.tmp")
)

temporary_acceptance_path.write_text(
    json.dumps(
        acceptance_payload,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


with temporary_acceptance_path.open(
    "r",
    encoding="utf-8",
) as handle:
    reloaded_acceptance = json.load(
        handle
    )


assert (
    reloaded_acceptance[
        "acceptance_status"
    ]
    == "PASS"
)


temporary_acceptance_path.replace(
    acceptance_output_path
)

acceptance_sha256 = sha256_file(
    acceptance_output_path
)


print("\nPERSISTED NB07 ACCEPTANCE")
print("=" * 80)
print(
    "Path:",
    acceptance_output_path.relative_to(
        project_root
    ),
)
print(
    "SHA-256:",
    acceptance_sha256,
)
print(
    "Acceptance status:",
    reloaded_acceptance[
        "acceptance_status"
    ],
)

NB07 FINAL ACCEPTANCE CANDIDATE

ARTIFACTS
--------------------------------------------------------------------------------
document_schema → schemas/phase2/document.schema.json
   exists: True
   sha256: 3f577c71e8faa3b50c231bd4163cb5425158d0390e5fe5fcf39e00f530baa755
unified_documents → data/phase2/processed/documents_unified_v1.jsonl
   exists: True
   sha256: 4ad6c3bcb1580d49f692e1d8a0406ee4c08bbf580cbb473e37baa107e89f5f04
lineage_report → reports/phase2/unified_schema_lineage_report_v1.json
   exists: True
   sha256: 8e29e4817a1b0be22e3d877f37e0fa7b7b5af5dfe8cb1db4581a9c9be2487cad

POPULATION
--------------------------------------------------------------------------------
Documents: 19456
Source distribution: {'website': 529, 'book': 2701, 'transcript': 16226}
Book units: {'paragraph': 2689, 'supplemental_text': 12}

QUALITY GATES
--------------------------------------------------------------------------------
Duplicate document_id: 0
Schema failures: 0
Text hash failures: 0
Broke

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ممیزی ساختاری و سلامت Notebook پیش از Restart Kernel → Run All</h3>

<p>
Acceptance محتوایی Notebook 07 با موفقیت PASS شده است.
پیش از اجرای نهایی از Kernel خالی، خود فایل Notebook از روی Disk بررسی می‌شود تا هیچ Cell خطادار، شماره تکراری، شماره جاافتاده، Syntax Error یا Markdown خارج از قالب استاندارد باقی نمانده باشد.
</p>

<p>
پیش از اجرای این Cell، Notebook باید ذخیره شده باشد تا نسخه روی Disk شامل همین Step نیز باشد.
</p>

<h4>کنترل‌های این مرحله</h4>

<ul>
<li>وجود فایل صحیح Notebook 07</li>
<li>وجود دقیق Code Cellهای 01 تا 30 بدون Duplicate یا Missing number</li>
<li>ساختار آغاز Notebook با دو Markdown و سپس Code Cell 01</li>
<li>الگوی یک Markdown + یک Code Cell در Stepهای بعدی</li>
<li>استفاده تمام Markdownها از Wrapper استاندارد RTL</li>
<li>Syntax صحیح تمام Code Cellها</li>
<li>صفر بودن Stored Error / Traceback در Outputهای Notebook</li>
</ul>

<h4>معیار PASS</h4>

<p>
تمام کنترل‌های ساختاری باید PASS شوند و تعداد Stored Error برابر صفر باشد.
پس از آن Notebook آماده Restart Kernel → Run All از ابتدا خواهد بود.
</p>

</div>

In [30]:
# Code Cell 30

import re


notebook_path = (
    project_root
    / "notebooks"
    / "phase2"
    / "07_unified_schema_and_lineage.ipynb"
)

assert notebook_path.is_file(), (
    "Notebook 07 file is missing."
)


with notebook_path.open(
    "r",
    encoding="utf-8",
) as handle:
    notebook_json = json.load(handle)


cells = notebook_json.get(
    "cells",
    [],
)

markdown_cells = [
    cell
    for cell in cells
    if cell.get("cell_type") == "markdown"
]

code_cells = [
    cell
    for cell in cells
    if cell.get("cell_type") == "code"
]


def cell_source(cell):
    source = cell.get("source", "")

    if isinstance(source, list):
        return "".join(source)

    return source


code_number_pattern = re.compile(
    r"^\s*# Code Cell (\d{2})\s*$",
    re.MULTILINE,
)

numbered_code_cells = []
unnumbered_code_cells = []

for index, cell in enumerate(cells):
    if cell.get("cell_type") != "code":
        continue

    source = cell_source(cell)

    match = code_number_pattern.search(
        source
    )

    if match:
        numbered_code_cells.append(
            {
                "cell_index": index,
                "number": int(
                    match.group(1)
                ),
            }
        )
    else:
        unnumbered_code_cells.append(
            index
        )


code_numbers = [
    item["number"]
    for item in numbered_code_cells
]

expected_code_numbers = list(
    range(1, 31)
)

duplicate_code_numbers = sorted(
    number
    for number, count in Counter(
        code_numbers
    ).items()
    if count > 1
)

missing_code_numbers = sorted(
    set(expected_code_numbers)
    - set(code_numbers)
)

unexpected_code_numbers = sorted(
    set(code_numbers)
    - set(expected_code_numbers)
)


rtl_opening = (
    '<div style="direction: rtl; '
    'white-space: normal; line-height: 1.7;">'
)

invalid_markdown_wrappers = []

for index, cell in enumerate(cells):
    if cell.get("cell_type") != "markdown":
        continue

    source = cell_source(cell).strip()

    if (
        not source.startswith(rtl_opening)
        or not source.endswith("</div>")
    ):
        invalid_markdown_wrappers.append(
            index
        )


syntax_failures = []

for index, cell in enumerate(cells):
    if cell.get("cell_type") != "code":
        continue

    source = cell_source(cell)

    try:
        compile(
            source,
            f"notebook_cell_{index}",
            "exec",
        )
    except SyntaxError as exc:
        syntax_failures.append(
            {
                "cell_index": index,
                "message": str(exc),
            }
        )


stored_error_outputs = []

for index, cell in enumerate(cells):
    if cell.get("cell_type") != "code":
        continue

    for output_index, output in enumerate(
        cell.get("outputs", [])
    ):
        if output.get("output_type") == "error":
            stored_error_outputs.append(
                {
                    "cell_index": index,
                    "output_index": output_index,
                    "ename": output.get(
                        "ename"
                    ),
                    "evalue": output.get(
                        "evalue"
                    ),
                }
            )


initial_structure_ok = (
    len(cells) >= 3
    and cells[0].get("cell_type")
    == "markdown"
    and cells[1].get("cell_type")
    == "markdown"
    and cells[2].get("cell_type")
    == "code"
    and numbered_code_cells
    and numbered_code_cells[0][
        "cell_index"
    ] == 2
    and numbered_code_cells[0][
        "number"
    ] == 1
)


pairing_failures = []

# After:
# Markdown 1
# Markdown 2
# Code Cell 01
#
# every remaining step must be:
# Markdown
# Code
for index in range(
    3,
    len(cells),
    2,
):
    markdown_index = index
    code_index = index + 1

    if code_index >= len(cells):
        pairing_failures.append(
            {
                "markdown_index": (
                    markdown_index
                ),
                "code_index": None,
                "reason": (
                    "missing_code_pair"
                ),
            }
        )
        continue

    if (
        cells[markdown_index].get(
            "cell_type"
        ) != "markdown"
        or cells[code_index].get(
            "cell_type"
        ) != "code"
    ):
        pairing_failures.append(
            {
                "markdown_index": (
                    markdown_index
                ),
                "code_index": code_index,
                "types": (
                    cells[
                        markdown_index
                    ].get("cell_type"),
                    cells[
                        code_index
                    ].get("cell_type"),
                ),
            }
        )


expected_markdown_count = 31
expected_code_count = 30
expected_total_cells = 61


notebook_health_gate = (
    len(cells) == expected_total_cells
    and len(markdown_cells)
    == expected_markdown_count
    and len(code_cells)
    == expected_code_count
    and initial_structure_ok
    and code_numbers
    == expected_code_numbers
    and len(
        duplicate_code_numbers
    ) == 0
    and len(
        missing_code_numbers
    ) == 0
    and len(
        unexpected_code_numbers
    ) == 0
    and len(
        unnumbered_code_cells
    ) == 0
    and len(
        invalid_markdown_wrappers
    ) == 0
    and len(pairing_failures) == 0
    and len(syntax_failures) == 0
    and len(stored_error_outputs) == 0
)


print("NB07 NOTEBOOK STRUCTURAL HEALTH AUDIT")
print("=" * 80)

print("\nCELL POPULATION")
print("-" * 80)
print(
    "Total cells:",
    len(cells),
)
print(
    "Markdown cells:",
    len(markdown_cells),
)
print(
    "Code cells:",
    len(code_cells),
)
print(
    "Expected:",
    {
        "total": expected_total_cells,
        "markdown": expected_markdown_count,
        "code": expected_code_count,
    },
)


print("\nCODE CELL NUMBERING")
print("-" * 80)
print(
    "Numbers:",
    code_numbers,
)
print(
    "Duplicate numbers:",
    duplicate_code_numbers,
)
print(
    "Missing numbers:",
    missing_code_numbers,
)
print(
    "Unexpected numbers:",
    unexpected_code_numbers,
)
print(
    "Unnumbered Code cells:",
    unnumbered_code_cells,
)


print("\nNOTEBOOK STRUCTURE")
print("-" * 80)
print(
    "Initial 2 Markdown + Code 01:",
    "PASS"
    if initial_structure_ok
    else "FAIL",
)
print(
    "Markdown/Code pairing failures:",
    len(pairing_failures),
)
print(
    "Invalid RTL Markdown wrappers:",
    len(
        invalid_markdown_wrappers
    ),
)


print("\nCODE HEALTH")
print("-" * 80)
print(
    "Syntax failures:",
    len(syntax_failures),
)
print(
    "Stored error outputs:",
    len(stored_error_outputs),
)


if pairing_failures:
    print("\nPairing failure details:")
    for failure in pairing_failures:
        print(failure)

if invalid_markdown_wrappers:
    print(
        "\nInvalid Markdown cell indices:",
        invalid_markdown_wrappers,
    )

if syntax_failures:
    print("\nSyntax failure details:")
    for failure in syntax_failures:
        print(failure)

if stored_error_outputs:
    print("\nStored error output details:")
    for failure in stored_error_outputs:
        print(failure)


print("\nNOTEBOOK HEALTH GATE")
print("=" * 80)

print(
    "Cell counts:",
    "PASS"
    if (
        len(cells)
        == expected_total_cells
        and len(markdown_cells)
        == expected_markdown_count
        and len(code_cells)
        == expected_code_count
    )
    else "FAIL",
)

print(
    "Code numbering:",
    "PASS"
    if (
        code_numbers
        == expected_code_numbers
        and not duplicate_code_numbers
        and not missing_code_numbers
        and not unexpected_code_numbers
        and not unnumbered_code_cells
    )
    else "FAIL",
)

print(
    "Markdown structure:",
    "PASS"
    if (
        initial_structure_ok
        and not pairing_failures
        and not invalid_markdown_wrappers
    )
    else "FAIL",
)

print(
    "Syntax:",
    "PASS"
    if not syntax_failures
    else "FAIL",
)

print(
    "Stored traceback/error outputs:",
    "PASS"
    if not stored_error_outputs
    else "FAIL",
)

print(
    "NB07 NOTEBOOK HEALTH:",
    "PASS"
    if notebook_health_gate
    else "FAIL",
)


assert notebook_health_gate, (
    "Notebook 07 structural health audit failed."
)

NB07 NOTEBOOK STRUCTURAL HEALTH AUDIT

CELL POPULATION
--------------------------------------------------------------------------------
Total cells: 61
Markdown cells: 31
Code cells: 30
Expected: {'total': 61, 'markdown': 31, 'code': 30}

CODE CELL NUMBERING
--------------------------------------------------------------------------------
Numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
Duplicate numbers: []
Missing numbers: []
Unexpected numbers: []
Unnumbered Code cells: []

NOTEBOOK STRUCTURE
--------------------------------------------------------------------------------
Initial 2 Markdown + Code 01: PASS
Markdown/Code pairing failures: 0
Invalid RTL Markdown wrappers: 0

CODE HEALTH
--------------------------------------------------------------------------------
Syntax failures: 0
Stored error outputs: 0

NOTEBOOK HEALTH GATE
Cell counts: PASS
Code numbering: PASS
Markdown structure: PASS
Syntax: PASS
Stored tr